# DNWE 2.0 — Deterministic Natural Writing Engine

A fully offline, rule-and-statistics **writing engine and humanizer** for research and professional prose. **No LLMs, no generative AI, no neural models, no cloud**, and the code enforces it: section 3 blocks every non-local network connection and makes every neural-network or language-model package unimportable. The engine first studies *how* a text is written (section 9). It then reconstructs vocabulary, syntax and discourse only where those measurements call for it, and applies only edits that (1) a named rule proposes for a measured reason, (2) pass the hard meaning invariants and (3) clear a confidence threshold. Everything else stays exactly as you wrote it. **Same input + same configuration = same output**, checked byte for byte, including across fresh processes (section 18).

**What it uses:** Stanford PCFG parser (a probabilistic grammar, non-neural) · NLTK averaged-perceptron tagger (a linear model) · pySBD sentence splitting · LemmInflect dictionary morphology (its learned out-of-vocabulary model is disabled) · WordNet · PPDB 2.0 (paraphrase candidates) · wordfreq · LanguageTool (local server, rule-based) · proselint · Schwartz–Hearst acronyms · textstat · LexicalRichness · sacrebleu.

**Writing modes:** academic · technical · general · conversational. **Intensities:** *conservative* (fixes and stock phrasing only) · *balanced* (adds plan-driven reconstruction of discourse and syntax) · *extensive* (adds open-ended lexical variation and pushes the style targets further).

**Formats:** LaTeX projects (edited in place, `\input` files included) · DOCX (edited run by run, styles kept) · Markdown · TXT · PDF (text layer → Markdown; scanned PDFs are not supported).

**How to run:** put your papers in `corpus/`, then *Run All*. The first run downloads about 3 GB once (section 2); after that, sections 3 onward never touch the network.

**Outputs (in `out/`):** an edited copy of each document, and the same text as `.txt`, `.md` and `.pdf` · `*.review.html` with the style profile (before → after, against the mode's targets) and a colour diff giving the rule, reason and confidence for every edit · `*.edits.csv` to accept or reject edits · `benchmark.csv` (quality, style markers, preservation, speed, memory) · `stage_timings.csv` · `determinism.csv` · `dev_set.csv` / `test_set.csv` for your labels, with every label ever given kept in `labels_archive.csv`.

> **Confidence is a prior until you label the dev set** (section 19). After labelling, each family's confidence becomes its measured precision, and families below 0.95 switch off automatically.
>
> **The style markers describe writing style, not authorship.** They count variation, stock phrasing and connective density; they are not an AI-detection score, and the engine does not optimise for any detector.

## 1 · Configuration
Change `MODE` (academic · technical · general · conversational) and `INTENSITY` (conservative · balanced · extensive) here.

In [1]:
#@title 1 · Configuration and environment check
import os, sys, platform, shutil, subprocess, json, re
from pathlib import Path

# ---- Paths (everything lives next to this notebook) -----------------------
BASE = Path(os.environ.get("DNWE_BASE", Path.cwd())).resolve()
RES = BASE / "resources"          # downloaded tools and data (one-time)
CORPUS = BASE / "corpus"          # your test documents (DOCX, TeX, MD, TXT, PDF)
OUT = BASE / "out"                # edited documents, reports, dev set
for d in (RES, CORPUS, OUT):
    d.mkdir(parents=True, exist_ok=True)

# Prefer a system Java 17+ over stale user-local JDKs that may appear earlier on PATH.
JAVA_CANDIDATES = [
    Path("/usr/lib/jvm/java-17-openjdk-amd64"),
    Path("/usr/lib/jvm/java-1.17.0-openjdk-amd64"),
    Path("/usr/lib/jvm/default-java"),
]
JAVA_HOME = next((p for p in JAVA_CANDIDATES if p.exists()), None)
if JAVA_HOME is not None:
    os.environ["JAVA_HOME"] = str(JAVA_HOME)
    java_bin = JAVA_HOME / "bin"
    if java_bin.exists():
        os.environ["PATH"] = f"{java_bin}{os.pathsep}{os.environ.get('PATH', '')}"

NLTK_DIR = Path.home() / "nltk_data"
STANFORD_JAR = RES / "stanford-parser-3.9.2.jar"
STANFORD_MODELS_JAR = RES / "stanford-parser-3.9.2-models.jar"
PCFG_MODEL = RES / "sp" / "edu/stanford/nlp/models/lexparser/englishPCFG.ser.gz"
LT_DIR = RES / "languagetool"
LT_VERSION = "6.8"
os.environ["LTP_PATH"] = str(LT_DIR)

# ---- Engine settings ---------------------------------------------------------
MODE = "academic"            # academic | technical | general | conversational
INTENSITY = "balanced"       # conservative | balanced | extensive
assert MODE in ("academic", "technical", "general", "conversational"), MODE
assert INTENSITY in ("conservative", "balanced", "extensive"), INTENSITY
PARSER_THREADS = max(1, (os.cpu_count() or 2) - 1)
JAVA_HEAP = "2g"


def _version(cmd):
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        return (out.stdout + out.stderr).strip()
    except (OSError, subprocess.TimeoutExpired):
        return None


def environment_report():
    import psutil
    java = _version(["java", "-version"])
    java_major = None
    if java:
        m = re.search(r'version "(\d+)', java)
        java_major = int(m.group(1)) if m else None
    rep = {
        "python": platform.python_version(),
        "os": platform.platform(),
        "cpu_cores": os.cpu_count(),
        "ram_gb": round(psutil.virtual_memory().total / 2**30, 1),
        "java_major": java_major,
        "disk_free_gb": round(shutil.disk_usage(BASE).free / 2**30, 1),
    }
    problems = []
    if sys.version_info < (3, 10):
        problems.append("Python 3.10+ is required (3.12 recommended).")
    if java_major is None or java_major < 17:
        problems.append("Java 17+ is required for the parser and LanguageTool: sudo apt install openjdk-17-jre-headless")
    if rep["ram_gb"] < 4:
        problems.append("At least 4 GB RAM is required.")
    return rep, problems


ENV, ENV_PROBLEMS = environment_report()
print(json.dumps(ENV, indent=2))
assert not ENV_PROBLEMS, "\n".join(ENV_PROBLEMS)
print("Environment OK")

{
  "python": "3.13.18",
  "os": "Linux-7.0.0-43-generic-x86_64-with-glibc2.43",
  "cpu_cores": 16,
  "ram_gb": 14.8,
  "java_major": 17,
  "disk_free_gb": 238.2
}
Environment OK


## 2 · One-time install
The only cell that uses the network. Downloads: Python packages, NLTK data (WordNet, tagger, CMUdict), Stanford parser (keeps only the 1.9 MB PCFG model), PPDB 2.0 (2.3 GB), LanguageTool 6.8 (260 MB). Every file is checksum-verified. Set `RUN_INSTALL = False` after the first successful run.

In [2]:
#@title 2 · One-time install of packages and resources (the ONLY cell that uses the network)
# Re-running is safe: anything already present and checksum-verified is skipped.
# Set RUN_INSTALL = False once everything is installed.
RUN_INSTALL = False

import hashlib, urllib.request, zipfile, time

PY_PACKAGES = ["nltk==3.10.4", "pysbd==0.3.4", "lemminflect==0.2.3", "wordfreq==3.1.1",
               "textstat==0.7.13", "lexicalrichness==0.5.1", "language_tool_python==3.4.0",
               "python-docx==1.2.0", "pdfplumber==0.11.10", "sacrebleu==2.6.0", "psutil==7.2.2",
               "proselint==0.16.0", "abbreviations==0.2.5", "fpdf2==2.8.5", "pandas"]

DOWNLOADS = {
    # name: (url, sha256 or sha1 prefix "sha1:...", approx size)
    "stanford-parser-3.9.2.jar": (
        "https://repo1.maven.org/maven2/edu/stanford/nlp/stanford-parser/3.9.2/stanford-parser-3.9.2.jar",
        "sha1:652e57ece7904a7d2ba9723d7971c892d944da6e", "4 MB"),
    "stanford-parser-3.9.2-models.jar": (
        "https://repo1.maven.org/maven2/edu/stanford/nlp/stanford-parser/3.9.2/stanford-parser-3.9.2-models.jar",
        "sha1:432e75a159c5a0c57769898b9aa72f2a7f5511f7", "627 MB (only the 1.9 MB PCFG model is kept)"),
    "ppdb-2.0-l-all.gz": (
        "https://huggingface.co/datasets/HMT2000/ppdb2.0all/resolve/main/ppdb-2.0-l-all.gz",
        "sha256:0fde517b55fcaa08a87534df78236d8803c9e8c00382415f5f15ae51ecf3330e", "2.3 GB"),
}


def _digest(path, algo):
    h = hashlib.new(algo)
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _fetch(name, url, check, tries=6):
    dst = RES / name
    algo, want = check.split(":")
    if dst.exists() and _digest(dst, algo) == want:
        return dst
    for attempt in range(tries):
        try:
            print(f"Downloading {name} ...")
            tmp = dst.with_suffix(dst.suffix + ".part")
            with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as fh:
                shutil.copyfileobj(r, fh, 1 << 20)
            got = _digest(tmp, algo)
            assert got == want, f"checksum mismatch for {name}: {got}"
            tmp.rename(dst)
            return dst
        except Exception as e:  # rate limits (HTTP 429) are common on Maven Central
            wait = 20 * (attempt + 1)
            print(f"  attempt {attempt+1} failed: {e}; retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"Could not download {name}. Fetch it manually from {url} into {RES}")


def install_everything():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PY_PACKAGES], check=True)
    import nltk
    for pkg in ["wordnet", "omw-1.4", "averaged_perceptron_tagger_eng", "cmudict"]:
        assert nltk.download(pkg, download_dir=str(NLTK_DIR), quiet=True), f"NLTK download failed: {pkg}"
    lock = {}
    for name, (url, check, size) in DOWNLOADS.items():
        if name == "stanford-parser-3.9.2-models.jar" and PCFG_MODEL.exists():
            lock[name] = check; continue
        _fetch(name, url, check); lock[name] = check
    if not PCFG_MODEL.exists():
        with zipfile.ZipFile(STANFORD_MODELS_JAR) as z:
            z.extract("edu/stanford/nlp/models/lexparser/englishPCFG.ser.gz", RES / "sp")
        STANFORD_MODELS_JAR.unlink()      # 627 MB jar no longer needed
    import language_tool_python           # downloads LanguageTool into LT_DIR on first use
    with language_tool_python.LanguageTool("en-US", language_tool_download_version=LT_VERSION) as t:
        assert t.check("This are wrong."), "LanguageTool did not start"
    lock["pcfg_model_sha256"] = _digest(PCFG_MODEL, "sha256")
    (RES / "LOCK.json").write_text(json.dumps(lock, indent=2))
    print("Install complete:", ", ".join(sorted(lock)))


if RUN_INSTALL:
    install_everything()

## 3 · Offline and AI-free guards
From here on, any Python connection to a non-local host raises an error. LanguageTool runs as a server on localhost, which is allowed. Importing any neural-network or language-model package (torch, TensorFlow, transformers, spaCy, the OpenAI/Anthropic SDKs, …) also raises an error, and both guards test themselves when the cell runs.

In [3]:
#@title 3 · Offline + AI-free guards: no remote connection and no neural / language-model code can run
import socket, importlib, importlib.abc

# ---- AI-free guard: neural-network and language-model packages cannot be imported ------------
# The engine is classical NLP only: rules, dictionaries, counts, a PCFG grammar and a linear
# perceptron tagger. This finder makes that a hard guarantee instead of a promise.
FORBIDDEN_AI = ("torch", "tensorflow", "keras", "jax", "flax", "transformers", "sentence_transformers",
                "onnxruntime", "spacy", "thinc", "stanza", "flair", "allennlp", "gensim", "fasttext",
                "tokenizers", "huggingface_hub", "vllm", "llama_cpp", "ctransformers", "openai", "anthropic",
                "google.generativeai", "google.genai", "cohere", "mistralai", "ollama", "langchain")


class _AIBlocker(importlib.abc.MetaPathFinder):
    _dnwe_ai_guard = True

    def find_spec(self, name, path=None, target=None):
        if any(name == f or name.startswith(f + ".") for f in FORBIDDEN_AI):
            raise ImportError(f"DNWE AI-free guard: '{name}' is a neural-network / language-model package and is blocked")
        return None


if not any(getattr(f, "_dnwe_ai_guard", False) for f in sys.meta_path):
    sys.meta_path.insert(0, _AIBlocker())
_loaded_ai = sorted(m for m in sys.modules if any(m == f or m.startswith(f + ".") for f in FORBIDDEN_AI))
assert not _loaded_ai, f"AI packages were already imported before the guard: {_loaded_ai} (restart the kernel)"
try:
    importlib.import_module("transformers")
    raise AssertionError("AI-free guard did not block an import")
except ImportError as _e:
    assert "AI-free guard" in str(_e)

# ---- Offline guard -------------------------------------------------------------------------

_LOCAL = {"127.0.0.1", "::1", "localhost", "0.0.0.0"}
if not getattr(socket, "_dnwe_guarded", False):
    _orig_connect = socket.socket.connect
    _orig_create = socket.create_connection

    def _host_of(addr):
        return addr[0] if isinstance(addr, tuple) else addr

    def _guard_connect(self, addr):
        if self.family in (socket.AF_INET, socket.AF_INET6) and _host_of(addr) not in _LOCAL:
            raise ConnectionRefusedError(f"DNWE offline guard blocked a connection to {addr}")
        return _orig_connect(self, addr)

    def _guard_create(addr, *a, **k):
        if _host_of(addr) not in _LOCAL:
            raise ConnectionRefusedError(f"DNWE offline guard blocked a connection to {addr}")
        return _orig_create(addr, *a, **k)

    socket.socket.connect = _guard_connect
    socket.create_connection = _guard_create
    socket._dnwe_guarded = True

for k in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "NO_UPDATE_NOTIFIER"):
    os.environ[k] = "1"

# Self-test: a remote connection must fail, a local one must be allowed.
try:
    socket.create_connection(("pypi.org", 443), timeout=3)
    raise AssertionError("offline guard did not block a remote host")
except ConnectionRefusedError:
    pass
_srv = socket.socket(); _srv.bind(("127.0.0.1", 0)); _srv.listen(1)
_cli = socket.create_connection(_srv.getsockname(), timeout=3); _cli.close(); _srv.close()
print("Offline guard active: remote connections blocked, localhost allowed (LanguageTool server).")
print(f"AI-free guard active: {len(FORBIDDEN_AI)} neural-network / language-model packages cannot be imported.")

Offline guard active: remote connections blocked, localhost allowed (LanguageTool server).
AI-free guard active: 28 neural-network / language-model packages cannot be imported.


## 4 · PPDB lexicon
Compiles the 18-million-rule PPDB file into a 53 MB SQLite index of single-word pairs (about 1–2 minutes, once).

In [4]:
#@title 4 · Compile PPDB 2.0 into a compact SQLite lexicon (runs once, ~2 min; cached afterwards)
# Finding from building this: PPDB 2.0's automatic entailment labels are unreliable for single
# words (it labels "utilize -> use" and "commence -> begin" as *Independent*). So PPDB is used
# only as a ranked CANDIDATE source; equivalence is proven later by WordNet (shared synset).
import gzip, sqlite3, re, time

PPDB_GZ = RES / "ppdb-2.0-l-all.gz"
PPDB_DB = RES / "ppdb_lexical.sqlite"
_PPDB_POS = {"NN", "NNS", "VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "JJ", "JJR", "JJS", "RB", "RBR"}
_WORD = re.compile(r"^[a-z][a-z\-]*[a-z]$")


def compile_ppdb(src=PPDB_GZ, dst=PPDB_DB, min_score=3.0):
    """Keep single-word -> single-word rules with a POS left-hand side, PPDB2.0Score >= min_score,
    and any entailment label except Exclusion."""
    if dst.exists():
        return dst
    assert src.exists(), f"Missing {src}; run section 2 (install) first."
    t0 = time.time()
    tmp = dst.with_suffix(".tmp")
    tmp.unlink(missing_ok=True)
    con = sqlite3.connect(tmp)
    con.execute("CREATE TABLE para(src TEXT, tgt TEXT, score REAL, label TEXT)")
    best, seen = {}, 0
    with gzip.open(src, "rt", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            seen += 1
            parts = line.rstrip("\n").split(" ||| ")
            if len(parts) < 6 or parts[-1] == "Exclusion":
                continue
            if parts[0].strip("[]") not in _PPDB_POS:
                continue
            s, t = parts[1], parts[2]
            if s == t or not _WORD.match(s) or not _WORD.match(t):
                continue
            m = re.search(r"PPDB2\.0Score=([\d.]+)", parts[3])
            if not m:
                continue
            sc = float(m.group(1))
            if sc < min_score:
                continue
            key = (s, t)
            if sc > best.get(key, (0, ""))[0]:
                best[key] = (sc, parts[-1])
    con.executemany("INSERT INTO para VALUES (?,?,?,?)",
                    sorted((s, t, sc, lab) for (s, t), (sc, lab) in best.items()))
    con.execute("CREATE INDEX ix_src ON para(src)")
    con.commit(); con.close()
    tmp.rename(dst)
    print(f"PPDB: scanned {seen:,} rules, kept {len(best):,} single-word pairs in {time.time()-t0:.0f}s")
    return dst

## 5 · Linguistic resources

In [5]:
#@title 5 · Linguistic resources: word lists, WordNet, morphology, frequency, tagging
import nltk, sqlite3, functools
from collections import Counter, defaultdict
from dataclasses import dataclass, field
nltk.data.path.insert(0, str(NLTK_DIR))
from nltk.corpus import wordnet as wn
from nltk.tag.perceptron import PerceptronTagger
from lemminflect import getAllLemmas, getAllInflections
from wordfreq import zipf_frequency
import lemminflect as _lemminflect


def _no_learned_oov(*a, **k):
    raise RuntimeError("DNWE AI-free guard: LemmInflect's learned out-of-vocabulary model is disabled; "
                       "morphology comes from its dictionary only")


for _fn in ("getAllLemmasOOV", "getAllInflectionsOOV", "getLemma", "getInflection"):
    setattr(_lemminflect, _fn, _no_learned_oov)      # these may fall back to the learned OOV model

assert wn.synsets("analyze"), "WordNet not found — run section 2"
_TAGGER = PerceptronTagger()           # averaged perceptron: a linear model, no neural weights
PPDB = sqlite3.connect(compile_ppdb(), check_same_thread=False)

# ---- Closed lists used by analysis, rules and the meaning gate -------------
NEGATIONS = {"not", "no", "never", "none", "nor", "neither", "nothing", "nobody", "nowhere",
             "cannot", "without", "hardly", "scarcely", "barely", "lack", "lacks", "lacked", "absent"}
HEDGES = {"may", "might", "could", "can", "possibly", "perhaps", "likely", "unlikely", "probably",
          "apparently", "suggest", "suggests", "suggested", "suggesting", "appear", "appears",
          "appeared", "seem", "seems", "seemed", "indicate", "indicates", "indicated", "tend",
          "tends", "somewhat", "relatively", "approximately", "roughly", "about", "around",
          "potentially", "presumably", "plausibly", "partly", "partially", "largely", "generally",
          "typically", "usually", "often", "sometimes", "would", "should", "assume", "assumed"}
BOOSTERS = {"clearly", "certainly", "definitely", "undoubtedly", "obviously", "always", "must",
            "prove", "proves", "proved", "proven", "demonstrate", "demonstrates", "demonstrated",
            "show", "shows", "showed", "shown", "establish", "establishes", "established",
            "confirm", "confirms", "confirmed", "strongly", "highly", "significantly",
            "substantially", "completely", "entirely", "fully", "very", "extremely", "really",
            "indeed", "essential", "crucial", "critical", "key", "major"}
QUANTIFIERS = {"all", "every", "each", "any", "some", "most", "many", "few", "several", "only",
               "both", "either", "more", "less", "fewer", "majority", "minority", "none", "half",
               "numerous", "various", "much", "little", "least", "always", "sometimes", "rarely",
               "seldom", "frequently", "first", "second", "third", "last", "single", "multiple"}
CLAIM_WORDS = HEDGES | BOOSTERS | QUANTIFIERS | NEGATIONS
STAT_TERMS = {"significant", "significance", "correlation", "correlated", "regression", "variance",
              "mean", "median", "mode", "error", "bias", "deviation", "estimate", "estimator",
              "model", "parameter", "effect", "power", "sample", "population", "distribution",
              "probability", "likelihood", "random", "normal", "confidence", "interval", "test",
              "control", "treatment", "baseline", "accuracy", "precision", "recall", "score",
              "rate", "ratio", "order", "degree", "function", "set", "field", "network", "agent",
              "state", "process", "term", "value", "data", "method", "approach", "system"}
CONNECTIVE_CLASS = {
    "contrast": {"but", "however", "yet", "whereas", "although", "though", "nevertheless",
                 "nonetheless", "conversely", "instead", "while", "despite", "unlike"},
    "cause": {"because", "since", "therefore", "thus", "hence", "consequently", "so",
              "accordingly", "due"},
    "condition": {"if", "unless", "provided", "otherwise", "whether"},
}
ADDITION_TRANSITIONS = ["in addition", "moreover", "furthermore", "additionally", "also",
                        "besides", "further"]
FUNCTION_TAGS = {"DT", "IN", "CC", "TO", "PRP", "PRP$", "WDT", "WP", "WP$", "EX", "POS", "MD",
                 "RP", "PDT", "UH", ",", ".", ":", "``", "''", "-LRB-", "-RRB-", "(", ")", "$", "#", "SYM"}
AUX_LEMMAS = {"be", "have", "do", "get"}
INFORMAL = {"get", "got", "lot", "lots", "big", "stuff", "thing", "kind", "sort", "really", "pretty", "okay",
            "ok", "huge", "tons", "guy", "nice", "bad", "good", "awesome", "basically", "totally", "plenty",
            "anyway", "maybe", "start", "fix", "check", "go", "put", "keep", "come", "let"}

# ---- Curated phrase tables -----------------------------------------------------------
# (pattern, replacement, condition on the NEXT token's POS or None)
WORDY = [
    ("due to the fact that", "because", None), ("owing to the fact that", "because", None),
    ("in view of the fact that", "because", None), ("on account of the fact that", "because", None),
    ("despite the fact that", "although", None), ("in spite of the fact that", "although", None),
    ("regardless of the fact that", "although", None), ("in the event that", "if", None),
    ("in order to", "to", "VB"), ("so as to", "to", "VB"), ("for the purpose of", "for", "VBG"),
    ("prior to", "before", None), ("subsequent to", "after", None), ("in the course of", "during", None),
    ("at this point in time", "now", None), ("at the present time", "currently", None),
    ("at the present moment", "currently", None), ("in the near future", "soon", None),
    ("until such time as", "until", None), ("in close proximity to", "near", None),
    ("a large number of", "many", "NNS"), ("a great number of", "many", "NNS"),
    ("a small number of", "a few", "NNS"), ("the vast majority of", "most", None),
    ("the majority of", "most", "NNS"), ("is able to", "can", "VB"), ("are able to", "can", "VB"),
    ("has the ability to", "can", "VB"), ("have the ability to", "can", "VB"),
    ("is capable of", "can", "VBG*"), ("are capable of", "can", "VBG*"),
    ("in a manner similar to", "like", None), ("with the exception of", "except", None),
    ("on a daily basis", "daily", None), ("on a regular basis", "regularly", None),
    ("in the absence of", "without", None), ("a sufficient amount of", "enough", None),
    ("an adequate number of", "enough", "NNS"), ("in excess of", "more than", None),
    ("it is important to note that", "", None), ("it should be noted that", "", None),
    ("it is worth noting that", "", None), ("it is interesting to note that", "", None),
    ("it must be noted that", "", None), ("needless to say,", "", None),
    ("as a matter of fact,", "", None), ("for all intents and purposes", "", None),
    ("whether or not", "whether", None),          # ("in terms of the" -> "in the" removed: changes meaning)
    ("the question as to whether", "whether", None), ("as to whether", "whether", None),
    ("there is no doubt that", "certainly", None), ("in light of the fact that", "because", None),
    ("by means of", "by", None), ("in conjunction with", "with", None),
    ("on the grounds that", "because", None), ("for the reason that", "because", None),
]
# Extra preconditions for phrases whose words also have another reading: fn(tokens, i, j) -> bool.
# "a GAN feature prior to guide" uses the NOUN 'prior' + infinitive 'to': not "before".
WORDY_GUARDS = {
    "prior to": lambda T, i, j: not (i > 0 and T[i - 1].tag[:2] in ("DT", "JJ", "NN", "PR", "PO"))
                                and not (j + 1 < len(T) and T[j + 1].tag == "VB"),
}
REDUNDANT = [  # (phrase, keep)
    ("past history", "history"), ("end result", "result"), ("final outcome", "outcome"),
    ("future plans", "plans"), ("basic fundamentals", "fundamentals"), ("each and every", "every"),
    ("first and foremost", "first"), ("true facts", "facts"), ("join together", "join"),
    ("combine together", "combine"), ("combined together", "combined"), ("merge together", "merge"),
    ("merged together", "merged"), ("collaborate together", "collaborate"), ("repeat again", "repeat"),
    ("revert back", "revert"), ("return back", "return"), ("new innovation", "innovation"),
    ("new innovations", "innovations"), ("unexpected surprise", "surprise"),
    ("mutual cooperation", "cooperation"), ("advance planning", "planning"),
    ("absolutely essential", "essential"), ("completely eliminate", "eliminate"),
    ("completely eliminated", "eliminated"), ("totally eliminate", "eliminate"),
    ("period of time", "period"), ("small in size", "small"), ("large in size", "large"),
    ("brief summary", "summary"), ("close proximity", "proximity"), ("still remains", "remains"),
    ("still remain", "remain"), ("added bonus", "bonus"), ("general consensus", "consensus"),
    ("consensus of opinion", "consensus"), ("exact same", "same"), ("both of the", "both"),
    ("various different", "various"), ("plan ahead", "plan"), ("over-exaggerate", "exaggerate"),
]
# light verb + noun (+ optional "of") -> verb lemma; the noun is the head being replaced
LIGHT_VERBS = {"make", "conduct", "perform", "carry", "undertake", "give", "provide", "reach",
               "come", "have", "take", "achieve"}
LIGHT_TABLE = {
    ("make", "decision"): "decide", ("make", "assumption"): "assume", ("make", "comparison"): "compare",
    ("make", "contribution"): "contribute", ("make", "use"): "use", ("make", "estimate"): "estimate",
    ("make", "prediction"): "predict", ("make", "measurement"): "measure",
    ("make", "observation"): "observe", ("make", "recommendation"): "recommend",
    ("make", "adjustment"): "adjust", ("make", "improvement"): "improve",
    ("make", "reference"): "refer", ("make", "distinction"): "distinguish",
    ("give", "consideration"): "consider", ("give", "description"): "describe",
    ("give", "explanation"): "explain", ("provide", "description"): "describe",
    ("provide", "explanation"): "explain", ("provide", "summary"): "summarize",
    ("provide", "estimate"): "estimate", ("reach", "conclusion"): "conclude",
    ("come", "conclusion"): "conclude", ("have", "tendency"): "tend",
    ("perform", "analysis"): "analyze", ("conduct", "analysis"): "analyze",
    ("carry", "analysis"): "analyze", ("undertake", "analysis"): "analyze",
    ("perform", "evaluation"): "evaluate", ("conduct", "evaluation"): "evaluate",
    ("conduct", "investigation"): "investigate", ("carry", "investigation"): "investigate",
    ("conduct", "examination"): "examine", ("perform", "calculation"): "calculate",
    ("perform", "computation"): "compute", ("conduct", "assessment"): "assess",
    ("perform", "assessment"): "assess", ("conduct", "review"): "review",
    ("perform", "comparison"): "compare", ("conduct", "comparison"): "compare",
    ("perform", "estimation"): "estimate", ("perform", "simulation"): "simulate",
    ("conduct", "simulation"): "simulate", ("achieve", "improvement"): "improve",
    ("take", "measurement"): "measure",
}

# Plain-language table: inflated word -> plain word (lemmas). Built from the U.S. federal
# plain-language word list idea, restricted to pairs that are interchangeable in research prose.
PLAIN = {("utilize", "V"): "use", ("utilise", "V"): "use", ("commence", "V"): "begin",
         ("endeavor", "V"): "try", ("endeavour", "V"): "try", ("ascertain", "V"): "determine",
         ("terminate", "V"): "end", ("necessitate", "V"): "require", ("possess", "V"): "have",
         ("assist", "V"): "help", ("purchase", "V"): "buy", ("elucidate", "V"): "explain",
         ("ameliorate", "V"): "improve", ("numerous", "J"): "many", ("cognizant", "J"): "aware",
         ("subsequently", "R"): "later", ("additionally", "R"): "also", ("approximately", "R"): "about",
         ("utilization", "N"): "use", ("utilisation", "N"): "use"}
# Pairs that are terms of art in research/technical prose ("the job is terminated when the
# allocation expires") and are swapped only in general and conversational writing.
PLAIN_MODES = {("terminate", "V"): {"general", "conversational"}}
# After the swap the preposition must change too: "assist in editing" -> "help with editing".
PLAIN_PREP = {("help", "in"): "with"}

# light-verb targets that take a direct object, so "conduct an analysis of X" -> "analyse X" is valid
TRANSITIVE = {"analyze", "investigate", "examine", "evaluate", "assess", "compare", "review", "measure",
              "estimate", "describe", "explain", "summarize", "calculate", "compute", "simulate", "use",
              "observe", "predict", "consider", "improve", "adjust"}

# ---- Morphology & lexical lookups ------------------------------------------------------
_UPOS = {"N": "NOUN", "V": "VERB", "J": "ADJ", "R": "ADV"}
_WNPOS = {"N": wn.NOUN, "V": wn.VERB, "J": wn.ADJ, "R": wn.ADV}


@functools.lru_cache(maxsize=200000)
def lemma(word, tag):
    """Dictionary lemma (LemmInflect dictionary only, no neural OOV path); falls back to word."""
    w = word.lower()
    up = _UPOS.get(tag[:1]) if tag else None
    if up:
        d = getAllLemmas(w, upos=up)
        if d.get(up):
            return d[up][0]
    return w


@functools.lru_cache(maxsize=200000)
def canon(word):
    """Tag-independent lemma used by the meaning gate, so a tagger flip (VBN vs JJ for
    'proposed') is never mistaken for a lost word."""
    w = word.lower()
    for up in ("VERB", "NOUN", "ADJ", "ADV"):
        d = getAllLemmas(w, upos=up)
        if d.get(up):
            return d[up][0]
    return w


@functools.lru_cache(maxsize=200000)
def inflect(lem, tag):
    """Dictionary inflection for a Penn tag, or None if the dictionary has no form."""
    up = _UPOS.get(tag[:1])
    if not up:
        return None
    forms = getAllInflections(lem, upos=up)
    if tag in forms:
        return forms[tag][0]
    if tag in ("VB", "VBP") and up == "VERB":
        return lem
    if tag == "NN":
        return lem
    return None


@functools.lru_cache(maxsize=200000)
def zipf(word):
    return zipf_frequency(word.lower(), "en")


@functools.lru_cache(maxsize=100000)
def wn_synonymous(a_lemma, b_lemma, tag, top=2):
    """True if b shares a synset with one of a's `top` most frequent WordNet senses.
    Restricting to a's dominant senses stops rare-sense swaps (e.g. reviewer -> reader)."""
    p = _WNPOS.get(tag[:1])
    if not p:
        return False
    pos = (wn.ADJ, "s") if p == wn.ADJ else (p,)
    sa = [x for q in pos for x in wn.synsets(a_lemma, pos=q)][:top]
    sb = {x for q in pos for x in wn.synsets(b_lemma, pos=q)}
    return bool(set(sa) & sb)


@functools.lru_cache(maxsize=50000)
def ppdb_candidates(word):
    rows = PPDB.execute("SELECT tgt, score FROM para WHERE src=? ORDER BY score DESC, tgt",
                        (word.lower(),)).fetchall()
    return tuple(rows)


def uk_variant(word):
    """British spelling of an -ize/-yze form if it is attested in wordfreq, else the word."""
    for a, b in (("yze", "yse"), ("yzed", "ysed"), ("yzes", "yses"), ("yzing", "ysing"),
                 ("ize", "ise"), ("ized", "ised"), ("izes", "ises"), ("izing", "ising"),
                 ("ization", "isation"), ("izations", "isations")):
        if word.endswith(a):
            cand = word[: -len(a)] + b
            if zipf(cand) >= 2.0:
                return cand
    return word


# ---- Tokenization ---------------------------------------------------------------------
PH = re.compile(r"⟦\d+⟧")
TOKEN_RE = re.compile(r"⟦\d+⟧|``|''|[A-Za-z]+(?:[-'’][A-Za-z]+)*|\d+(?:[.,]\d+)*|\S")


@dataclass
class Tok:
    text: str
    start: int
    end: int
    tag: str = ""
    lem: str = ""


def tokenize(text, offset=0):
    return [Tok(m.group(), m.start() + offset, m.end() + offset) for m in TOKEN_RE.finditer(text)]


def tag_tokens(toks):
    if not toks:
        return toks
    words = ["Xref" if PH.fullmatch(t.text) else t.text for t in toks]
    for t, (_, tg) in zip(toks, _TAGGER.tag(words)):
        t.tag = "NNP" if PH.fullmatch(t.text) else tg
        t.lem = t.text if t.tag == "NNP" and PH.fullmatch(t.text) else lemma(t.text, t.tag)
    return toks


print(f"Lexicon ready: WordNet {len(list(wn.all_lemma_names()))} lemmas, "
      f"PPDB {PPDB.execute('select count(*) from para').fetchone()[0]:,} pairs, "
      f"{len(WORDY)} wordy phrases, {len(REDUNDANT)} redundancies, {len(LIGHT_TABLE)} light-verb pairs.")

Lexicon ready: WordNet 147306 lemmas, PPDB 817,232 pairs, 56 wordy phrases, 41 redundancies, 45 light-verb pairs.


## 6 · Documents, protection, PDF reading and exports
Every format becomes editable paragraphs in which math, citations, LaTeX commands, Word fields, numbers, URLs, DOIs and direct quotations are replaced by atomic placeholders `⟦n⟧`. Only the paragraph's own span of the source is ever rewritten.

**PDFs** are read from their word positions, never from cropped pages (cropping clips words mid-line):

- tables are kept whole as Markdown tables;
- two-column pages are read left column, then right, with full-width titles in place;
- running headers, footers and page numbers are dropped;
- headings come from font size and weight (small caps and two-line titles included);
- bullets stay list items, ligatures and line-end hyphenation are repaired;
- equations and captions are kept verbatim, and references are kept unedited.

**Exports:** every result can be written as `.txt`, `.md` and a freshly typeset `.pdf` (bundled DejaVu fonts, byte-identical for identical input), plus `.docx` / `.tex` for Word and LaTeX inputs.

In [6]:
#@title 6 · Document model, protection and format readers/writers (LaTeX, Markdown, TXT, DOCX, PDF)
# Design: every format is split into editable BLOCKS (paragraphs). Inside a block, everything that
# must never change (math, citations, commands, numbers, URLs, quotations, Word fields...) is
# replaced by an atomic placeholder ⟦n⟧. Rules only see and edit the masked text; on output the
# placeholders are restored byte-for-byte and only the block's own span of the source is replaced.
import copy, difflib, hashlib, statistics
from docx import Document as DocxDocument
from docx.oxml.ns import qn
from docx.text.run import Run

# ---------- shared protection patterns (applied to every format's editable text) ----------
# Each pattern must END on a character that belongs to the protected item, never on sentence
# punctuation: a placeholder that swallows the full stop in "...on QM9." or "...in 2020." hides the
# sentence boundary from the segmenter and silently merges two sentences.
_COMMON = [
    r"https?://[^\s)\]}>]*[^\s)\]}>.,;:!?'\"]", r"www\.[^\s)\]}>]*[^\s)\]}>.,;:!?'\"]",
    r"\b10\.\d{4,9}/[^\s,;)\]]*[^\s,;)\].:]",
    r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    r"“[^”\n]{1,400}”", r"\"[^\"\n]{1,400}\"",                      # direct quotations
    r"\[(?:\d+(?:\s*[-–,]\s*\d+)*)\]",                              # [1], [2-4]
    r"\((?:[A-Z][A-Za-z'’\-]+(?:\s+(?:et al\.|and|&)\s*(?:[A-Z][A-Za-z'’\-]+)?)*,?\s*(?:19|20)\d{2}[a-z]?(?:[;,]\s*)?)+\)",
    r"[A-Z][A-Za-z'’\-]+(?:\s+et al\.)?\s+\((?:19|20)\d{2}[a-z]?\)",  # Smith et al. (2020)
    r"\bet al\.", r"\b(?:e\.g|i\.e|cf|vs|etc|Fig|Figs|Eq|Eqs|Tab|Sec|Ref|Refs|No|approx|resp)\.",
    r"[-+±]?(?<![\w⟦])\d(?:[\w.,/%‰:\-]*[\w%‰])?(?:\s?%)?",          # numbers, versions, 3D, 95%
    r"\b[A-Za-z]+\d(?:[\w\-./]*\w)?",                               # H2O, COVID-19, v2
    r"[α-ωΑ-Ω∑∏∫∂∇≤≥≈≠±×÷→←↔∞∈∉⊂⊆∪∩√∝∼°′″]+",                      # math symbols
]
COMMON_RE = re.compile("|".join(f"(?:{p})" for p in _COMMON))

SECTION_KEYS = [("abstract", "abstract"), ("introduction", "introduction"), ("reference", "references"),
                ("bibliograph", "references"), ("acknowledg", "acknowledgements"), ("appendix", "appendix"),
                ("supplement", "appendix"), ("result", "results"), ("finding", "results"),
                ("evaluation", "results"), ("discussion", "discussion"), ("limitation", "discussion"),
                ("conclusion", "conclusion"), ("summary", "conclusion"), ("background", "background"),
                ("related work", "related"), ("literature", "related"), ("material", "methods"),
                ("method", "methods"), ("experimental setup", "methods"), ("experiment", "methods"),
                ("data", "methods"), ("model", "methods"),
]
SKIP_SECTIONS = {"references", "acknowledgements", "title"}
CONSERVATIVE_SECTIONS = {"methods", "results"}


def canonical_section(title):
    t = re.sub(r"\\[A-Za-z]+\*?|[{}]|^[\dIVX.\s]+", " ", title or "").lower().strip()
    for key, lab in SECTION_KEYS:
        if key in t:
            return lab
    return "body"


@dataclass
class Block:
    bid: int
    file: str
    start: int
    end: int
    section: str
    masked: str
    spans: dict
    kind: str = "para"
    docx_para: object = None
    docx_owner: list = None


@dataclass
class Doc:
    path: Path
    fmt: str
    sources: dict
    blocks: list
    root: Path = None
    docx: object = None
    files_order: list = field(default_factory=list)


class Masker:
    """Turns a list of (text, protected?) atoms into masked text + placeholder map."""

    def __init__(self):
        self.n = 0

    def ph(self):
        self.n += 1
        return f"⟦{self.n}⟧"

    def mask_atoms(self, atoms):
        out, spans = [], {}
        for text, prot in atoms:
            if prot:
                p = self.ph(); spans[p] = text; out.append(p); continue
            pos = 0
            for m in COMMON_RE.finditer(text):
                if m.start() > pos:
                    out.append(text[pos:m.start()])
                p = self.ph(); spans[p] = m.group(); out.append(p); pos = m.end()
            out.append(text[pos:])
        return "".join(out), spans


def unmask(masked, spans):
    return PH.sub(lambda m: spans[m.group()], masked)


def is_prose_block(bl):
    """Command listings, flattened tables and bullet fragments are not prose, and no rule may edit
    them (on the test corpus LanguageTool 'corrected' the Slurm command `squeue` to `Queue`).
    A block is NOT prose when it has 2+ lines and most lines are both short and unpunctuated.
    Wrapped LaTeX/TXT paragraphs pass: their lines are long even when they end mid-sentence."""
    lines = [ln.strip() for ln in bl.masked.split("\n") if ln.strip()]
    if len(lines) < 2:
        return True
    unpunct = sum(1 for ln in lines if not re.search(r"[.!?:;,]$|⟦\d+⟧$", ln))
    short = sum(1 for ln in lines if len(re.findall(r"[A-Za-z]{2,}", ln)) <= 5)
    return not (unpunct >= 0.5 * len(lines) and short >= 0.5 * len(lines))


def display(text, bl):
    """Human-readable text: Word elements (fields, equations, links) shown as [their text]."""
    if getattr(bl, "elem_text", None):
        text = re.sub(f"[{chr(PUA)}-{chr(PUA + 4095)}]", lambda m: "[" + bl.elem_text.get(m.group(), "?")[:40] + "]", text)
    return text


def _atoms_from_ranges(text, start, end, prot_ranges):
    """Split text[start:end] into atoms using sorted, non-overlapping protected ranges."""
    atoms, pos = [], start
    for a, b in prot_ranges:
        if b <= start or a >= end:
            continue
        a, b = max(a, start), min(b, end)
        if a > pos:
            atoms.append((text[pos:a], False))
        atoms.append((text[a:b], True)); pos = b
    if pos < end:
        atoms.append((text[pos:end], False))
    return atoms


def _chunks(text, start, end, prot_ranges):
    """Split [start,end) on blank lines that are not inside protected ranges."""
    cuts = []
    for m in re.finditer(r"\n[ \t]*\n", text[start:end]):
        a, b = m.start() + start, m.end() + start
        if not any(pa < b and a < pb for pa, pb in prot_ranges):
            cuts.append((a, b))
    pos, out = start, []
    for a, b in cuts:
        out.append((pos, a)); pos = b
    out.append((pos, end))
    return [(a, b) for a, b in out if text[a:b].strip()]


def _prot_ranges(text, start, end, regex):
    return [(m.start(), m.end()) for m in regex.finditer(text, start, end)]


def _editable_words(atoms):
    return sum(len(re.findall(r"[A-Za-z]{2,}", t)) for t, p in atoms if not p)


# ---------------------------------- LaTeX ----------------------------------
_SKIP_ENVS = ("figure|table|equation|align|alignat|flalign|gather|multline|eqnarray|displaymath|math|"
              "tabular|tabularx|tabulary|longtable|verbatim|Verbatim|lstlisting|minted|algorithm|"
              "algorithmic|algorithm2e|tikzpicture|thebibliography|comment|subequations|array|cases|"
              "wrapfigure|sidewaysfigure|sidewaystable|filecontents|quote|quotation|tcolorbox|"
              "lemma|theorem|proof|definition|proposition|corollary|remark|example")
_CMD = r"\\[A-Za-z@]+\*?(?:\s*\[[^\[\]]*\])*(?:\s*\{(?:[^{}]|\{(?:[^{}]|\{(?:[^{}]|\{[^{}]*\})*\})*\})*\})*"
LATEX_PROT = re.compile("|".join([
    rf"\\begin\{{({_SKIP_ENVS})\*?\}}.*?\\end\{{\1\*?\}}",
    r"(?<!\\)%[^\n]*", r"\$\$.*?\$\$", r"\\\[.*?\\\]", r"\\\(.*?\\\)", r"(?<!\\)\$(?:\\.|[^$\\])+?\$",
    r"``(?:(?!\n[ \t]*\n).)*?''", _CMD, r"\\\\(?:\[[^\]]*\])?", r"\\[^A-Za-z\s]", r"~", r"[{}]",
]), re.S)
LATEX_SECTION = re.compile(r"\\(?:chapter|section|subsection|subsubsection|paragraph)\*?\s*(?:\[[^\]]*\])?\s*\{((?:[^{}]|\{[^{}]*\})*)\}")
LATEX_INPUT = re.compile(r"(?<!%)\\(?:input|include|subfile)\s*\{([^}]+)\}")


def latex_files(main):
    """main .tex plus every \\input/\\include reachable from it, in document order."""
    root, order, seen = main.parent, [], set()

    def visit(p):
        if p in seen or not p.exists():
            return
        seen.add(p); order.append(p)
        txt = re.sub(r"(?<!\\)%[^\n]*", "", p.read_text(encoding="utf-8", errors="replace"))
        for m in LATEX_INPUT.finditer(txt):
            q = (root / m.group(1).strip())
            visit(q if q.suffix == ".tex" else q.with_suffix(".tex"))
    visit(main)
    return root, order


def read_latex(path):
    path = Path(path)
    root, files = latex_files(path)
    mk, blocks, sources, section = Masker(), [], {}, "body"
    for f in files:
        key = str(f.relative_to(root))
        text = f.read_text(encoding="utf-8", errors="replace")
        sources[key] = text
        b0 = text.find("\\begin{document}")
        if b0 >= 0:
            start = b0 + len("\\begin{document}")
            e0 = text.find("\\end{document}"); end = e0 if e0 >= 0 else len(text)
        else:
            start, end = 0, len(text)
        prot = _prot_ranges(text, start, end, LATEX_PROT)
        markers = sorted([(m.start(), canonical_section(m.group(1))) for m in LATEX_SECTION.finditer(text, start, end)]
                         + [(m.start(), "abstract") for m in re.finditer(r"\\begin\{abstract\}", text)]
                         + [(m.end(), "body") for m in re.finditer(r"\\end\{abstract\}", text)]
                         + [(m.start(), "appendix") for m in re.finditer(r"\\appendix\b", text)]
                         + [(m.start(), "references") for m in re.finditer(r"\\bibliography\{|\\printbibliography", text)])
        for a, b in _chunks(text, start, end, prot):
            for pos, lab in markers:
                if pos <= a:
                    section = lab
            local = [(pos, lab) for pos, lab in markers if a < pos < b]
            atoms = _atoms_from_ranges(text, a, b, prot)
            if _editable_words(atoms) < 4:
                if local:
                    section = local[-1][1]
                continue
            masked, spans = mk.mask_atoms(atoms)
            sec = section if not local else local[0][1]
            blocks.append(Block(len(blocks), key, a, b, sec, masked, spans))
            if local:
                section = local[-1][1]
    return Doc(path, "latex", sources, blocks, root=root, files_order=[str(f.relative_to(root)) for f in files])


# -------------------------------- Markdown ---------------------------------
MD_PROT = re.compile("|".join([
    r"(?s:\A---\n.*?\n---\n)", r"(?s:^(```|~~~).*?^\1[^\n]*$)", r"(?s:\$\$.*?\$\$)", r"(?<!\\)\$[^$\n]+\$",
    r"`[^`\n]+`", r"!?\[[^\]\n]*\]\([^)\n]*\)", r"\[[^\]\n]+\]\[[^\]\n]*\]", r"\[@[^\]\n]+\]",
    r"(?<![\w@])@[\w:.-]+", r"<https?://[^>]+>", r"</?[A-Za-z][^>\n]*>", r"^#{1,6}[ \t].*$",
    r"^[ \t]*\|.*\|[ \t]*$", r"^[ \t]*(?:[-*+]|\d+[.)])[ \t]+", r"^[ \t]*>[ \t]?", r"\*\*|__",
    r"^[ \t]*\[\^[^\]]+\]:.*$", r"\[[^\]\n]*\]\{[^}\n]*\}", r"\{[#.][^}\n]*\}", r"\{[^}\n]*=[^}\n]*\}",
]), re.M)


def _plain_blocks(text, prot_re, heading_fn):
    mk, blocks, section = Masker(), [], "body"
    prot = _prot_ranges(text, 0, len(text), prot_re) if prot_re else []
    for a, b in _chunks(text, 0, len(text), prot):
        chunk = text[a:b]
        h = heading_fn(chunk)
        if h is not None:
            section = canonical_section(h)
            continue
        atoms = _atoms_from_ranges(text, a, b, prot)
        if _editable_words(atoms) < 4:
            continue
        masked, spans = mk.mask_atoms(atoms)
        blocks.append(Block(len(blocks), "main", a, b, section, masked, spans))
    return blocks


def _md_heading(chunk):
    m = re.match(r"^\s*#{1,6}\s+(.+?)\s*#*\s*$", chunk)
    if m and "\n" not in chunk.strip():
        return m.group(1)
    if re.match(r"^\s*#{1,6}\s+", chunk):      # heading followed by text in same chunk
        return None
    return None


def _md_section_markers(text):
    return [(m.start(), canonical_section(m.group(1))) for m in re.finditer(r"^#{1,6}\s+(.+)$", text, re.M)]


def read_markdown(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8", errors="replace")
    blocks = _plain_blocks(text, MD_PROT, _md_heading)
    markers = _md_section_markers(text)
    for bl in blocks:                       # sections from the nearest heading above each block
        sec = "body"
        for pos, lab in markers:
            if pos <= bl.start:
                sec = lab
        bl.section = sec
    return Doc(path, "markdown", {"main": text}, blocks)


def _txt_heading(chunk):
    s = chunk.strip()
    if "\n" in s or len(s.split()) > 12 or s.endswith((".", ":", ";", ",")):
        return None
    if re.match(r"^(\d+(\.\d+)*\.?|[IVX]+\.)\s+[A-Z]", s) or s.isupper() or canonical_section(s) != "body":
        return s
    return None


def read_txt(path):
    path = Path(path)
    text = path.read_text(encoding="utf-8", errors="replace")
    return Doc(path, "txt", {"main": text}, _plain_blocks(text, None, _txt_heading))


# ---------------------------------- DOCX -----------------------------------
_ZERO_WIDTH = {qn("w:bookmarkStart"), qn("w:bookmarkEnd"), qn("w:proofErr"), qn("w:permStart"),
               qn("w:permEnd"), qn("w:commentRangeStart"), qn("w:commentRangeEnd")}
_PLAIN_RUN = {qn("w:rPr"), qn("w:t"), qn("w:tab"), qn("w:br")}
PUA = 0xE000


def read_docx(path):
    path = Path(path)
    d = DocxDocument(path)
    mk, blocks, section = Masker(), [], "body"
    for p in d.paragraphs:
        style = (p.style.name if p.style is not None else "").lower()
        if style.startswith("heading") or style == "title":
            section = "title" if style == "title" else canonical_section(p.text)
            continue
        if style.startswith(("caption", "toc", "bibliography")):
            continue
        chars, owner, elems, in_field = [], [], [], 0
        for child in p._p.iterchildren():
            tag = child.tag
            if tag == qn("w:pPr") or tag in _ZERO_WIDTH:
                continue
            if tag == qn("w:r"):
                kinds = {c.tag for c in child.iterchildren()}
                fld = [c.get(qn("w:fldCharType")) for c in child.iterchildren() if c.tag == qn("w:fldChar")]
                if "begin" in fld:
                    in_field += 1
                if in_field or not kinds <= _PLAIN_RUN:
                    elems.append(child); chars.append(chr(PUA + len(elems) - 1)); owner.append(None)
                else:
                    r = Run(child, p)
                    for ch in r.text:
                        chars.append(ch); owner.append(r)
                if "end" in fld:
                    in_field = max(0, in_field - 1)
                continue
            elems.append(child); chars.append(chr(PUA + len(elems) - 1)); owner.append(None)
        full = "".join(chars)
        if len(re.findall(r"[A-Za-z]{2,}", full)) < 4:
            continue
        atoms, buf = [], ""
        for ch in full:
            if ord(ch) >= PUA and ord(ch) < PUA + 4096:
                if buf:
                    atoms.append((buf, False)); buf = ""
                atoms.append((ch, True))
            else:
                buf += ch
        if buf:
            atoms.append((buf, False))
        masked, spans = mk.mask_atoms(atoms)
        bl = Block(len(blocks), "main", 0, 0, section, masked, spans)
        bl.docx_para, bl.docx_owner = p, owner
        bl.docx_full = full
        bl.elem_text = {chr(PUA + i): "".join(e.itertext()) or "obj" for i, e in enumerate(elems)}
        blocks.append(bl)
    return Doc(path, "docx", {}, blocks, docx=d)


def apply_docx_block(bl, new_masked):
    """Write a block's new text back into its Word runs. Returns False if the edit would have to
    move or touch a protected Word element (field, hyperlink, equation), which is refused."""
    old, new = bl.docx_full, unmask(new_masked, bl.spans)
    if old == new:
        return True
    pua = lambda s: [c for c in s if PUA <= ord(c) < PUA + 4096]
    if pua(old) != pua(new):
        return False
    sm = difflib.SequenceMatcher(None, old, new, autojunk=False)
    texts = {}
    order = []
    for r in bl.docx_owner:
        if r is not None and id(r) not in texts:
            texts[id(r)] = []; order.append(r)

    def owner_at(i, prefer_prev=True):
        rng = range(i - 1, -1, -1) if prefer_prev else range(i, len(old))
        for k in rng:
            if bl.docx_owner[k] is not None:
                return bl.docx_owner[k]
        return None

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        seg_old = old[i1:i2]
        if tag != "equal" and (pua(seg_old) or pua(new[j1:j2])):
            return False
        if tag == "equal":
            for k in range(i1, i2):
                if bl.docx_owner[k] is not None:
                    texts[id(bl.docx_owner[k])].append(old[k])
            continue
        tgt = bl.docx_owner[i1] if (tag == "replace" and bl.docx_owner[i1] is not None) else \
            (owner_at(i1) or owner_at(i1, prefer_prev=False))
        if tgt is None:
            return False
        texts[id(tgt)].append(new[j1:j2])
    for r in order:
        r.text = "".join(texts[id(r)])
    return True


# ----------------------------------- PDF -----------------------------------
# Layout analysis on WORD boxes (never on cropped pages, which clip words mid-line):
#   1. tables are found first, kept whole as Markdown tables (never edited) and removed from the text;
#   2. each page is tested for two columns: a vertical gutter that no word crosses, with text on both
#      sides reaching it. Lines that do cross it (a title, a full-width figure caption) split the
#      page into bands; inside a band the left column is read fully before the right one;
#   3. running headers/footers and page numbers that repeat across pages are dropped;
#   4. headings come from font size and weight; a title broken over two lines becomes one heading;
#   5. paragraphs come from line spacing, indentation and short last lines; bullets stay list items;
#   6. ligatures are expanded, line-end hyphenation is undone only when the joined word is a word;
#   7. equations, figure/table captions and fragments are kept verbatim in protected code spans;
#      references are kept (the engine never edits that section) instead of being dropped.
_PDF_CHARS = str.maketrans({"ﬀ": "ff", "ﬁ": "fi", "ﬂ": "fl", "ﬃ": "ffi", "ﬄ": "ffl",
                            "ﬅ": "st", "ﬆ": "st", "­": "", "‐": "-", "‑": "-",
                            " ": " ", " ": " ", "​": ""})
_PDF_BULLET = re.compile(r"^(?:[•◦▪▫‣∙●○■□➢➤►✓✔]|[-*–](?=\s))\s*")
_PDF_NUMBERED = re.compile(r"^(?:\(?\d{1,2}[.)]|\(?[a-z][.)])\s+(?=\S)")
# a caption needs punctuation after its number ("Fig. 1.", "Figure 2:", "TABLE II" alone); prose that
# merely starts "Figure 1 summarizes ..." or "Table I lists ..." is ordinary, editable text
_PDF_CAPTION = re.compile(r"^(?:(?:Figure|Fig\.|Table|Algorithm|Listing)\s*[\dIVX]+[a-z]?\s*[.:]|TABLE\s+[IVX\d]+\s*$)", re.I)
_PDF_CID = re.compile(r"\(cid:\d+\)")                          # glyphs with no Unicode mapping
_PDF_SECTION_NUM = re.compile(r"^(\d+(\.\d+)*\.?|[IVX]+\.|[A-Z]\.)\s+[A-Z]")
_PDF_PAGE_NUM = re.compile(r"^(page\s*)?\d{1,4}(\s*(of|/)\s*\d{1,4})?$", re.I)


def _pdf_tables(page):
    """Real tables only: >= 2 rows, >= 2 columns, at least half the cells filled."""
    out = []
    try:
        found = page.find_tables()
    except Exception:
        return out
    for t in found:
        rows = [[re.sub(r"\s+", " ", _PDF_CID.sub("", (c or "").translate(_PDF_CHARS))).strip() for c in r] for r in (t.extract() or [])]
        rows = [r for r in rows if any(r)]
        cells = [c for r in rows for c in r]
        if len(rows) >= 2 and max(len(r) for r in rows) >= 2 and cells and sum(bool(c) for c in cells) >= 0.5 * len(cells):
            out.append({"bbox": t.bbox, "rows": rows})
    return out


def _pdf_gutter(words, W):
    """x of a column gutter, or None for a single-column page. A gutter is an empty vertical STRIP
    at least 8 pt wide (real gutters are 14-25 pt); a 2-pt word space that happens to line up over
    a few lines of a single-column page is not one."""
    if len(words) < 40:
        return None
    best = None
    for gx in range(int(W * .35), int(W * .65)):
        cross = sum(1 for w in words if w["x0"] < gx + 4 and w["x1"] > gx - 4)
        if best is None or cross < best[0]:
            best = (cross, gx)
    cross, gx = best
    left = [w for w in words if w["x1"] <= gx]
    right = [w for w in words if w["x0"] >= gx]
    if cross > max(3, 0.04 * len(words)) or len(left) < 0.25 * len(words) or len(right) < 0.25 * len(words):
        return None
    reach_l = sum(1 for w in left if w["x1"] >= gx - 0.12 * W) / len(left)     # left lines run up to the gutter
    reach_r = sum(1 for w in right if w["x0"] <= gx + 0.12 * W) / len(right)   # right lines start at the gutter
    return gx if reach_l >= 0.08 and reach_r >= 0.08 else None


def _pdf_lines(words, marks=()):
    """Group words (one column/band) into lines: same baseline within half a font size. A small
    drawn mark (dot/square) just left of a line's first word makes that line a bullet item."""
    lines = []
    for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])):
        size = w.get("size", 10)
        for ln in lines[-3:]:
            if abs(ln["top"] - w["top"]) <= 0.5 * size and not any(x["x0"] < w["x1"] - 0.5 and w["x0"] < x["x1"] - 0.5 for x in ln["w"]):
                ln["w"].append(w); ln["x0"] = min(ln["x0"], w["x0"]); ln["x1"] = max(ln["x1"], w["x1"])
                ln["bottom"] = max(ln["bottom"], w["bottom"])
                break
        else:
            lines.append({"w": [w], "top": w["top"], "bottom": w["bottom"], "x0": w["x0"], "x1": w["x1"]})
    out = []
    for ln in lines:
        ws = sorted(ln["w"], key=lambda w: w["x0"])
        text = _PDF_CID.sub("", " ".join(w["text"] for w in ws).translate(_PDF_CHARS)).strip()
        if not text:
            continue
        chars = [len(w["text"]) for w in ws]
        size = sorted(w.get("size", 10) for w in ws)[len(ws) // 2]
        bold_frac = sum(c for c, w in zip(chars, ws) if re.search(r"Bold|Black|Heavy|Semibold|Demi|Medi", w.get("fontname", ""))) / max(1, sum(chars))
        ital_frac = sum(c for c, w in zip(chars, ws) if re.search(r"Ital|Oblique|-It\b|It$", w.get("fontname", ""))) / max(1, sum(chars))
        bold = bold_frac >= 0.6
        drawn_bullet = any(m["x1"] <= ln["x0"] and ln["x0"] - m["x1"] < 3 * size
                           and ln["top"] - 1 <= (m["top"] + m["bottom"]) / 2 <= ln["bottom"] + 1 for m in marks)
        if drawn_bullet and not _PDF_BULLET.match(text):
            text = "• " + text
        out.append({"kind": "line", "text": text, "top": ln["top"], "bottom": ln["bottom"], "x0": ln["x0"],
                    "x1": ln["x1"], "size": size, "bold": bold, "drawn": drawn_bullet,
                    "bold_frac": bold_frac, "ital_frac": ital_frac})
    return sorted(out, key=lambda l: l["top"])


def _pdf_page_units(page):
    """Reading-ordered units (lines and tables) of one page, each tagged with its column box."""
    W = page.width
    tables = _pdf_tables(page)
    inside = lambda w, bb: bb[0] - 1 <= (w["x0"] + w["x1"]) / 2 <= bb[2] + 1 and bb[1] - 1 <= (w["top"] + w["bottom"]) / 2 <= bb[3] + 1
    words = []
    # words are split by spacing only (not by font size, which would break SMALL CAPS headings apart);
    # each word's size and weight then come from its own characters
    for w in page.extract_words(x_tolerance=1.5, y_tolerance=2.5, return_chars=True):
        if any(inside(w, t["bbox"]) for t in tables):
            continue
        ch = w.pop("chars")
        w["size"] = max(c["size"] for c in ch) if ch else 10
        w["fontname"] = Counter(c["fontname"] for c in ch).most_common(1)[0][0] if ch else ""
        words.append(w)
    marks = [m for m in list(page.curves) + list(page.rects)
             if 1 <= m["x1"] - m["x0"] <= 7 and 1 <= m["bottom"] - m["top"] <= 7]
    tunits = [{"kind": "table", "rows": t["rows"], "top": t["bbox"][1], "bottom": t["bbox"][3],
               "x0": t["bbox"][0], "x1": t["bbox"][2]} for t in tables]
    gx = _pdf_gutter(words, W)
    if gx is None:
        units = _pdf_lines(words, marks) + tunits
        for u in units:
            u["col"] = (min(l["x0"] for l in units), max(l["x1"] for l in units))
        return sorted(units, key=lambda u: u["top"])
    # bands: y-ranges of material that crosses the gutter are read full-width, in place
    spans = sorted((w["top"] - 1, w["bottom"] + 1) for w in words if w["x0"] < gx < w["x1"])
    spans += sorted((t["top"], t["bottom"]) for t in tunits if t["x0"] < gx < t["x1"])
    merged = []
    for a, b in sorted(spans):
        if merged and a <= merged[-1][1] + 2:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    cy = lambda u: (u["top"] + u["bottom"]) / 2
    cx = lambda u: (u["x0"] + u["x1"]) / 2
    in_span = lambda y: any(a <= y <= b for a, b in merged)
    full_w = [w for w in words if in_span(cy(w))]
    col_w = [w for w in words if not in_span(cy(w))]
    full = _pdf_lines(full_w, marks) + [t for t in tunits if in_span(cy(t))]
    left = _pdf_lines([w for w in col_w if (w["x0"] + w["x1"]) / 2 < gx], marks) + [t for t in tunits if not in_span(cy(t)) and cx(t) < gx]
    right = _pdf_lines([w for w in col_w if (w["x0"] + w["x1"]) / 2 >= gx], marks) + [t for t in tunits if not in_span(cy(t)) and cx(t) >= gx]
    for u in full:
        u["col"] = (0, W)
    for u in left:
        u["col"] = (min((l["x0"] for l in left), default=0), gx)
    for u in right:
        u["col"] = (gx, max((l["x1"] for l in right), default=W))
    cuts = [b for _, b in merged]
    band = lambda u: sum(1 for c in cuts if c < cy(u))           # how many full-width bands lie above it
    order = []
    for k in range(len(cuts) + 1):
        order += sorted([u for u in full if band(u) == k - 1 and k > 0], key=lambda u: u["top"])
        order += sorted([u for u in left if band(u) == k], key=lambda u: u["top"])
        order += sorted([u for u in right if band(u) == k], key=lambda u: u["top"])
    seen, out = set(), []
    for u in order + full:                                         # anything not yet placed keeps its place at the end
        if id(u) not in seen:
            seen.add(id(u)); out.append(u)
    return out


def _md_table(rows):
    n = max(len(r) for r in rows)
    esc = lambda c: c.replace("|", "\\|")
    rows = [[esc(c) for c in r] + [""] * (n - len(r)) for r in rows]
    return "\n".join(["| " + " | ".join(rows[0]) + " |", "|" + " --- |" * n] + ["| " + " | ".join(r) + " |" for r in rows[1:]])


def _join_line(para, text):
    """Append a wrapped line: undo end-of-line hyphenation only when the joined word is a real word."""
    if not para:
        return text
    m = re.search(r"([A-Za-z]+)-$", para)
    n = re.match(r"([a-z]+)", text)
    if m and n:
        if zipf(m.group(1) + n.group(1)) > 0 and zipf(m.group(1) + "-" + n.group(1)) <= zipf(m.group(1) + n.group(1)):
            return para[:-1] + text                                # "computa-" + "tional" -> "computational"
        return para + text                                         # "state-of-" + "the-art" keeps the hyphen
    return para + " " + text


def pdf_to_markdown(path, dst):
    """Text-layer PDF -> Markdown that the Markdown reader (and so the engine) understands."""
    import pdfplumber
    pages = []
    with pdfplumber.open(path) as pdf:
        for pno, page in enumerate(pdf.pages):
            units = _pdf_page_units(page)
            for u in units:
                u["page"], u["H"] = pno, page.height
            pages.append(units)
    if not any(u["kind"] == "line" for p in pages for u in p):
        raise ValueError("PDF has no text layer (scanned?) — OCR is out of scope.")
    # running headers / footers / page numbers: in the top or bottom 8% and repeated across pages
    npages = len(pages)
    norm = lambda s: re.sub(r"\d+", "#", s.strip().lower())
    edge = lambda u: u["kind"] == "line" and (u["top"] < 0.08 * u["H"] or u["bottom"] > 0.92 * u["H"])
    freq = Counter(norm(u["text"]) for p in pages for u in p if edge(u))
    units = [u for p in pages for u in p
             if not (edge(u) and (_PDF_PAGE_NUM.match(u["text"].strip()) or (npages >= 2 and freq[norm(u["text"])] >= max(2, 0.5 * npages))))]
    lines = [u for u in units if u["kind"] == "line"]
    body = statistics.median([u["size"] for u in lines]) if lines else 10
    big = max((u["size"] for u in lines), default=body)

    def heading_level(u, nxt, prev):
        t, words = u["text"], u["text"].split()
        if t[:1].islower():
            return 0
        if (len(words) > 16 or t.endswith((".", ",", ";")) or not re.search(r"[A-Za-z]{3,}", t) or _PDF_CAPTION.match(t)
                or _PDF_BULLET.match(t)):
            return 0                                                  # a list item is never a heading
        if u["size"] >= body * 1.15 and not table_zone:
            return 1 if (u["size"] >= big - 0.5 and u["page"] == 0) else 2
        whole = lambda frac: frac >= 0.95                              # the WHOLE line, not a bold lead-in phrase
        if _PDF_SECTION_NUM.match(t) and len(words) <= 10 and (whole(u["bold_frac"]) or whole(u["ital_frac"])
                                                                or t.isupper() or u["size"] >= body * 1.02):
            if re.match(r"^[IVX]+\.\s+[A-Z]{2}", t) or re.match(r"^\d+\.?\s", t):
                return 2                                              # "V. RESULTS ...", "3 Methods": sections
            return 3                                                  # "A. Dataset ...", "2.1 ...": subsections
        near = lambda v: v is not None and v["kind"] == "line" and v["page"] == u["page"] and v["col"] == u["col"] \
            and (abs(v["top"] - u["bottom"]) < 1.6 * u["size"] or abs(u["top"] - v["bottom"]) < 1.6 * u["size"])
        in_bold_par = (near(nxt) and nxt["bold"]) or (near(prev) and prev.get("bold") and not prev.get("_h"))
        if whole(u["bold_frac"]) and len(words) <= 8 and not in_bold_par and not table_zone:   # bold table header rows stay in the table   # a bold paragraph (abstract, index terms) is not a heading
            return 3
        return 0

    out, para, kind = [], "", "p"
    in_refs, li_x0, table_zone = False, None, False

    def flush():
        nonlocal para, kind
        if para.strip():
            text = para.strip()
            if kind == "frag" or (kind == "label" and len(text.split()) <= 8):
                out.append("`" + text.replace("`", "'") + "`")
            elif kind == "li":
                out.append(("" if _PDF_NUMBERED.match(text) else "- ") + text)
            else:
                out.append(text)
        para, kind = "", "p"

    prev = None
    for i, u in enumerate(units):
        nxt = units[i + 1] if i + 1 < len(units) else None
        if u["kind"] == "table":
            flush(); out.append(_md_table(u["rows"])); prev = None; continue
        t = u["text"]
        lvl = heading_level(u, nxt, prev)
        if lvl:
            table_zone = False
            if out and prev is not None and prev.get("_h") == lvl and abs(prev["size"] - u["size"]) < 0.6 \
                    and u["top"] - prev["bottom"] < 1.2 * u["size"] and out[-1].startswith("#") \
                    and not _PDF_SECTION_NUM.match(t):                   # a numbered heading always starts anew
                out[-1] += " " + t                                   # a heading broken over two lines
            else:
                flush(); out.append("#" * lvl + " " + t)
            in_refs = canonical_section(t) == "references"
            u["_h"] = lvl; prev = u; continue
        alpha = sum(c.isalpha() for c in t) / max(1, len(t))
        words = t.split()
        if table_zone and prev is not None and (prev["page"], prev["col"]) != (u["page"], u["col"]):
            table_zone = False
        if table_zone and len(words) >= 9 and alpha >= 0.65:
            table_zone = False                                        # prose resumes after a rule-only (booktabs) table
        if alpha < 0.5 or table_zone:
            flush(); para, kind = t, "frag"; flush(); prev = u; continue     # equation, plotted values, table row: one line each
        bullet = _PDF_BULLET.match(t) or (_PDF_NUMBERED.match(t) and not in_refs)
        new_par = prev is None or prev.get("_h")
        if prev is not None and not new_par:
            same_flow = prev["page"] == u["page"] and prev["col"] == u["col"]
            gap = u["top"] - prev["bottom"]
            ended = re.search(r"[.!?:]['\")\]]?$", para or "")
            short_prev = prev["x1"] < prev["col"][1] - 0.12 * (prev["col"][1] - prev["col"][0])
            downward = prev["page"] == u["page"] and u["top"] >= prev["bottom"] - 1
            if same_flow or downward:                                 # same column, or a full-width band -> the columns below it
                new_par = (gap > 0.9 * u["size"] or (ended and short_prev)
                           or (ended and u["x0"] > prev["x0"] + 0.8 * u["size"] and kind != "li")
                           or abs(u["size"] - prev["size"]) > 1.0 or (not same_flow and kind == "frag"))
            else:                                                     # next column (back up the page) or next page
                new_par = bool(ended) and not (t[:1].islower())
        if in_refs and re.match(r"^\[\d+\]", t):
            new_par = True                                            # one reference per paragraph
        caption = new_par and _PDF_CAPTION.match(t)
        if caption:
            flush(); para, kind = t, "frag"                           # caption: protected, wrapped lines join it
            table_zone = bool(re.match(r"^TABLE\s+[IVX\d]+", t))      # IEEE: the table body follows its caption
        elif bullet:
            flush(); kind = "li"; para = _PDF_BULLET.sub("", t)
            # drawn bullet: wrapped lines align with the item's text (x0); glyph bullet: they are indented past it
            li_x0 = u["x0"] - 2 if t.startswith("• ") and u.get("drawn") else u["x0"] + 1
        elif new_par:
            flush(); para = t
            if len(words) <= 4 and not re.search(r"[.!?:;,]$", t):
                kind = "label"                                        # short label; becomes prose if it grows
        elif kind == "li" and li_x0 is not None and u["x0"] < li_x0:
            flush(); para = t                                         # text back at the outer margin ends the item
        else:
            para = _join_line(para, t)
        prev = u
    flush()
    md = "\n\n".join(out) + "\n"
    Path(dst).write_text(md, encoding="utf-8")
    return Path(dst)


# --------------------------------- dispatch --------------------------------
def load_document(path):
    path = Path(path)
    ext = path.suffix.lower()
    if ext == ".tex":
        return read_latex(path)
    if ext in (".md", ".markdown"):
        return read_markdown(path)
    if ext == ".txt":
        return read_txt(path)
    if ext == ".docx":
        return read_docx(path)
    if ext == ".pdf":
        md = pdf_to_markdown(path, OUT / f"{path.stem}.from_pdf.md")
        doc = read_markdown(md); doc.fmt = "pdf-derived-markdown"
        return doc
    raise ValueError(f"Unsupported format: {ext}")


def render_document(doc, new_masked_by_block):
    """Return {file_key: new_text} for text formats, or a modified python-docx Document."""
    if doc.fmt == "docx":
        fresh = read_docx(doc.path)          # always render from the untouched original file
        assert [b.masked for b in fresh.blocks] == [b.masked for b in doc.blocks]
        refused = []
        for bl in fresh.blocks:
            nm = new_masked_by_block.get(bl.bid)
            if nm is not None and nm != bl.masked and not apply_docx_block(bl, nm):
                refused.append(bl.bid)
        return fresh.docx, refused
    out = dict(doc.sources)
    for key in out:
        text = doc.sources[key]
        for bl in sorted((b for b in doc.blocks if b.file == key), key=lambda b: -b.start):
            nm = new_masked_by_block.get(bl.bid)
            if nm is None or nm == bl.masked:
                continue
            text = text[:bl.start] + unmask(nm, bl.spans) + text[bl.end:]
        out[key] = text
    return out, []


def write_document(doc, rendered, tag="dnwe"):
    """Write the edited document next to OUT, mirroring LaTeX projects. Returns the main output path."""
    if doc.fmt == "docx":
        dst = OUT / f"{doc.path.stem}.{tag}.docx"
        rendered.save(dst)
        return dst
    if doc.fmt == "latex":
        dst_root = OUT / f"{doc.path.stem}.{tag}_latex"
        if dst_root.exists():
            shutil.rmtree(dst_root)
        shutil.copytree(doc.root, dst_root)
        for key, txt in rendered.items():
            (dst_root / key).write_text(txt, encoding="utf-8")
        return dst_root / doc.path.relative_to(doc.root)
    ext = ".md" if doc.fmt.endswith("markdown") else doc.path.suffix
    dst = OUT / f"{doc.path.stem}.{tag}{ext}"
    dst.write_text(rendered["main"], encoding="utf-8")
    return dst

# ------------------------- exports: every result as .txt, .md and .pdf -------------------------
# The edited document is turned into a structured view — [(kind, payload)] with kind in
# title | h2 | h3 | p | li | table | pre — and written out as plain text, clean Markdown and a fresh,
# typeset PDF (bundled DejaVu fonts, so Greek letters, ±, ≤ and → render on every machine). A PDF's
# original page layout cannot be edited in place by any tool; the exported PDF is a new, clean document
# with the same headings, paragraphs, lists and tables. Word and LaTeX inputs also keep their own format.
FONTS = RES / "fonts"
PDF_EPOCH = (2026, 1, 1)                                    # fixed PDF timestamps: same input -> same PDF bytes


def _strip_md_inline(s):
    s = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", s)                         # images
    s = re.sub(r"\[([^\]]+)\]\(([^)]+)\)", r"\1", s)                    # links keep their text
    s = re.sub(r"(\*\*|__)(.+?)\1", r"\2", s)
    s = re.sub(r"(?<![\w*])\*(?!\s)(.+?)(?<!\s)\*(?![\w*])", r"\1", s)
    s = re.sub(r"`([^`]+)`", r"\1", s)
    return s.strip()


def _md_elements(md, txt_mode=False):
    """Structured view of Markdown (or of plain text when txt_mode: headings by _txt_heading)."""
    els, buf = [], []
    for chunk in re.split(r"\n[ \t]*\n", md.replace("\r\n", "\n")):
        c = chunk.strip("\n")
        if not c.strip():
            continue
        lines = c.split("\n")
        if txt_mode:
            h = _txt_heading(c)
            els.append(("h2", h.strip()) if h else ("p", re.sub(r"\s*\n\s*", " ", c.strip())))
            continue
        if re.match(r"^(```|~~~)", c.strip()):
            els.append(("pre", re.sub(r"^(```|~~~)[^\n]*\n?|\n?(```|~~~)\s*$", "", c.strip())))
            continue
        if all(re.match(r"^\s*\|.*\|\s*$", ln) for ln in lines) and len(lines) >= 2:
            rows = [[_strip_md_inline(x.replace("\\|", "|")) for x in re.split(r"(?<!\\)\|", ln.strip())[1:-1]]
                    for ln in lines if not re.match(r"^\s*\|[\s:|-]+\|\s*$", ln)]
            els.append(("table", rows))
            continue
        m = re.match(r"^(#{1,6})\s+(.*)$", c.strip())
        if m and len(lines) == 1:
            lvl = len(m.group(1))
            els.append(("title" if lvl == 1 and not any(k == "title" for k, _ in els) else ("h2" if lvl <= 2 else "h3"),
                        _strip_md_inline(m.group(2))))
            continue
        if all(re.match(r"^\s*(?:[-*+•]|\d+[.)])\s+", ln) or not ln.strip() or ln.startswith((" ", "\t")) for ln in lines) \
                and re.match(r"^\s*(?:[-*+•]|\d+[.)])\s+", lines[0]):
            item = None
            for ln in lines:
                mm = re.match(r"^\s*([-*+•]|\d+[.)])\s+(.*)$", ln)
                if mm:
                    if item:
                        els.append(item)
                    num = mm.group(1) if mm.group(1)[0].isdigit() else ""
                    item = ("li", (num + " " if num else "") + _strip_md_inline(mm.group(2)))
                elif item:
                    item = ("li", item[1] + " " + _strip_md_inline(ln))
            if item:
                els.append(item)
            continue
        text = " ".join(ln.strip() for ln in lines)
        verbatim = text.startswith("`") and text.endswith("`") and text.count("`") == 2
        els.append(("pre" if verbatim else "p", _strip_md_inline(text)))
    return els


def _latex_text(s):
    """Readable text from a LaTeX fragment (for the .txt/.md/.pdf exports of a .tex input)."""
    s = re.sub(r"(?<!\\)%[^\n]*", "", s)
    s = re.sub(r"\\(?:cite[a-z]*|citep|citet)\*?(?:\[[^\]]*\])*\{([^}]*)\}", lambda m: "[" + m.group(1).replace(",", ", ") + "]", s)
    s = re.sub(r"\\(?:eq|auto|c|C)?ref\{([^}]*)\}", r"\1", s)
    s = re.sub(r"\\label\{[^}]*\}", "", s)
    for _ in range(3):
        s = re.sub(r"\\(?:textbf|textit|emph|texttt|textsc|textrm|underline|mbox|text|url|footnote)\{([^{}]*)\}", r"\1", s)
    s = re.sub(r"\\href\{[^}]*\}\{([^}]*)\}", r"\1", s)
    s = s.replace("~", " ").replace("---", "—").replace("--", "–").replace("``", "“").replace("''", "”")
    s = re.sub(r"\\([%&#_$])", r"\1", s)
    s = re.sub(r"\\[A-Za-z]+\*?(?:\[[^\]]*\])?", "", s)
    s = re.sub(r"[{}]", "", s)
    return re.sub(r"[ \t]+", " ", s).strip()


def _latex_elements(doc, rendered):
    main = rendered[doc.files_order[0]]
    def inline(m):                                                   # follow \input{...} into the project's other files
        key = m.group(1).strip()
        key = key if key.endswith(".tex") else key + ".tex"
        return rendered.get(key, "")
    body = LATEX_INPUT.sub(inline, main)
    b0 = body.find("\\begin{document}")
    body = body[b0 + len("\\begin{document}"):] if b0 >= 0 else body
    body = body.split("\\end{document}")[0]
    els = []
    t = re.search(r"\\title\{((?:[^{}]|\{[^{}]*\})*)\}", main)
    if t:
        els.append(("title", _latex_text(t.group(1))))
    body = re.sub(r"\\begin\{(figure|table|algorithm|tikzpicture|equation|align|thebibliography)\*?\}.*?\\end\{\1\*?\}",
                  lambda m: "\n\n" + "".join(f"\\captionmark{{{c}}}\n\n" for c in re.findall(r"\\caption\{((?:[^{}]|\{[^{}]*\})*)\}", m.group(0))),
                  body, flags=re.S)
    for chunk in re.split(r"\n[ \t]*\n", body):
        c = chunk.strip()
        if not c:
            continue
        for part in re.split(r"(\\(?:section|subsection|subsubsection|paragraph)\*?\{(?:[^{}]|\{[^{}]*\})*\}|\\begin\{abstract\}|\\end\{abstract\}|\\captionmark\{(?:[^{}]|\{[^{}]*\})*\}|\\item\b)", c):
            if not part or not part.strip():
                continue
            m = re.match(r"\\(section|subsection|subsubsection|paragraph)\*?\{(.*)\}$", part.strip(), re.S)
            if m:
                els.append(("h2" if m.group(1) == "section" else "h3", _latex_text(m.group(2))))
            elif part.strip() == "\\begin{abstract}":
                els.append(("h2", "Abstract"))
            elif part.startswith("\\captionmark"):
                els.append(("pre", _latex_text(part[len("\\captionmark{"):-1])))
            elif part.strip() == "\\item":
                els.append(("li", ""))
            else:
                txt = _latex_text(part)
                if not txt or re.fullmatch(r"\W*", txt):
                    continue
                if els and els[-1] == ("li", ""):
                    els[-1] = ("li", txt)
                else:
                    els.append(("p", txt))
    return [e for e in els if e != ("li", "")]


def _docx_elements(document):
    from docx.table import Table as _T
    from docx.text.paragraph import Paragraph as _P
    els = []
    for el in document.element.body.iterchildren():
        if el.tag == qn("w:tbl"):
            rows = [[re.sub(r"\s+", " ", c.text).strip() for c in r.cells] for r in _T(el, document).rows]
            if rows:
                els.append(("table", rows))
        elif el.tag == qn("w:p"):
            p = _P(el, document)
            text = p.text.strip()
            if not text:
                continue
            style = (p.style.name if p.style is not None else "").lower()
            numbered = p._p.pPr is not None and p._p.pPr.numPr is not None
            if style == "title":
                els.append(("title", text))
            elif style.startswith("heading"):
                els.append(("h2" if style in ("heading 1", "heading1") else "h3", text))
            elif "list" in style or numbered:
                els.append(("li", text))
            else:
                els.append(("p", text))
    return els


def doc_elements(doc, rendered):
    """Structured view of a rendered (edited or original) document."""
    if doc.fmt == "docx":
        return _docx_elements(rendered)
    if doc.fmt == "latex":
        return _latex_elements(doc, rendered)
    return _md_elements(rendered["main"], txt_mode=(doc.fmt == "txt"))


def elements_to_markdown(els):
    out = []
    for kind, x in els:
        if kind == "title":
            out.append("# " + x)
        elif kind == "h2":
            out.append("## " + x)
        elif kind == "h3":
            out.append("### " + x)
        elif kind == "li":
            out.append(x if re.match(r"^\d+[.)]\s", x) else "- " + x)
        elif kind == "table":
            out.append(_md_table(x))
        elif kind == "pre":
            out.append("`" + x.replace("`", "'") + "`" if "\n" not in x else "```\n" + x + "\n```")
        else:
            out.append(x)
    return "\n\n".join(out) + "\n"


def elements_to_text(els):
    out = []
    for kind, x in els:
        if kind == "title":
            out.append(x.upper() if len(x) < 60 else x)
        elif kind in ("h2", "h3"):
            out.append(x)
        elif kind == "li":
            out.append(x if re.match(r"^\d+[.)]\s", x) else "• " + x)
        elif kind == "table":
            widths = [max(len(r[i]) if i < len(r) else 0 for r in x) for i in range(max(len(r) for r in x))]
            out.append("\n".join("  ".join((r[i] if i < len(r) else "").ljust(widths[i]) for i in range(len(widths))).rstrip() for r in x))
        else:
            out.append(x)
    return "\n\n".join(out) + "\n"


def elements_to_pdf(els, title="DNWE output"):
    """A clean, typeset A4 PDF of the document (byte-identical for identical input)."""
    import datetime
    from fpdf import FPDF
    from fpdf.enums import XPos, YPos
    pdf = FPDF(format="A4")
    pdf.set_creation_date(datetime.datetime(*PDF_EPOCH, tzinfo=datetime.timezone.utc))
    pdf.set_title(title[:200]); pdf.set_creator("DNWE 2.0"); pdf.set_producer("DNWE 2.0 (fpdf2)")
    for style, f in (("", "DejaVuSerif.ttf"), ("B", "DejaVuSerif-Bold.ttf"), ("I", "DejaVuSerif-Italic.ttf"),
                     ("BI", "DejaVuSerif-BoldItalic.ttf")):
        pdf.add_font("Serif", style, str(FONTS / f))
    pdf.add_font("Sans", "", str(FONTS / "DejaVuSans.ttf")); pdf.add_font("Sans", "B", str(FONTS / "DejaVuSans-Bold.ttf"))
    pdf.add_font("Mono", "", str(FONTS / "DejaVuSansMono.ttf"))
    pdf.set_margins(22, 20, 22); pdf.set_auto_page_break(True, 20)
    pdf.add_page()
    W = pdf.w - pdf.l_margin - pdf.r_margin
    nl = dict(new_x=XPos.LMARGIN, new_y=YPos.NEXT)
    for kind, x in els:
        x = x if kind == "table" else x.replace("\t", " ")
        if kind == "title":
            pdf.set_font("Sans", "B", 17); pdf.multi_cell(W, 8.5, x, align="C", **nl); pdf.ln(5)
        elif kind == "h2":
            pdf.ln(3); pdf.set_font("Sans", "B", 13); pdf.multi_cell(W, 7, x, **nl); pdf.ln(1.5)
        elif kind == "h3":
            pdf.ln(1.5); pdf.set_font("Sans", "B", 11); pdf.multi_cell(W, 6, x, **nl); pdf.ln(1)
        elif kind == "li":
            pdf.set_font("Serif", "", 10.5)
            num = re.match(r"^(\d+[.)])\s+(.*)$", x)
            mark, body = (num.group(1), num.group(2)) if num else ("•", x)
            y = pdf.get_y(); pdf.set_x(pdf.l_margin + 3); pdf.cell(6, 5.6, mark)
            pdf.set_xy(pdf.l_margin + 9, y); pdf.multi_cell(W - 9, 5.6, body, **nl); pdf.ln(1.2)
        elif kind == "table":
            pdf.set_font("Serif", "", 8.5)
            ncol = max(len(r) for r in x)
            rows = [r + [""] * (ncol - len(r)) for r in x]
            with pdf.table(width=W, text_align="LEFT", line_height=4.6, padding=1.2,
                           first_row_as_headings=True) as tb:
                for r in rows:
                    row = tb.row()
                    for c in r:
                        row.cell(c)
            pdf.ln(3)
        elif kind == "pre":
            pdf.set_font("Mono", "", 8.2); pdf.set_text_color(90, 90, 90)
            pdf.multi_cell(W, 4.4, x, **nl); pdf.set_text_color(0, 0, 0); pdf.ln(1.5)
        else:
            pdf.set_font("Serif", "", 10.5); pdf.multi_cell(W, 5.6, x, align="J", **nl); pdf.ln(2.4)
    return bytes(pdf.output())


EXPORT_MIME = {"txt": "text/plain; charset=utf-8", "md": "text/markdown; charset=utf-8", "pdf": "application/pdf",
               "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
               "tex": "application/x-tex; charset=utf-8"}


def export_all(doc, rendered, title=None):
    """{ext: bytes} for the edited document: its own format (Word, LaTeX, Markdown, text) plus .txt,
    .md and .pdf for every input. A PDF input comes back as .pdf, .md and .txt."""
    import io
    els = doc_elements(doc, rendered)
    title = title or next((x for k, x in els if k == "title"), None) or doc.path.stem
    out = {}
    if doc.fmt == "docx":
        buf = io.BytesIO(); rendered.save(buf); out["docx"] = buf.getvalue()
    elif doc.fmt == "latex":
        out["tex"] = rendered[doc.files_order[0]].encode("utf-8")
    out["md"] = (rendered["main"] if doc.fmt == "markdown" else elements_to_markdown(els)).encode("utf-8")
    out["txt"] = (rendered["main"] if doc.fmt == "txt" else elements_to_text(els)).encode("utf-8")
    out["pdf"] = elements_to_pdf(els, title)
    return out


## 7 · Sentences and parsing
The Stanford PCFG parser runs once per document in a single JVM, only on sentences that a structural rule might change. Parses are cached on disk.

In [7]:
#@title 7 · Sentences, tagging and the non-neural Stanford PCFG parser
import pysbd, warnings, tempfile
from nltk import Tree
warnings.filterwarnings("ignore", category=SyntaxWarning)
_SEG = pysbd.Segmenter(language="en", clean=False, char_span=True)


@dataclass
class Sentence:
    block: object
    idx: int
    start: int          # offsets in block.masked
    end: int
    text: str
    toks: list
    parse: object = None   # (Tree, deps) or None


def split_sentences(bl):
    """pysbd on the masked text; newlines are blanked (same length) so wrapped LaTeX lines
    do not create false boundaries."""
    flat = bl.masked.replace("\n", " ").replace("\t", " ")
    out = []
    for s in _SEG.segment(flat):
        a = s.start
        b = s.start + len(s.sent.rstrip())
        while a < b and bl.masked[a].isspace():
            a += 1
        if b <= a:
            continue
        toks = tag_tokens(tokenize(bl.masked[a:b], a))
        if toks:
            out.append(Sentence(bl, len(out), a, b, bl.masked[a:b], toks))
    return out


# ---------------------------- Stanford PCFG -----------------------------------
PARSE_CACHE = OUT / ".parse_cache.json"
_parse_cache = json.loads(PARSE_CACHE.read_text()) if PARSE_CACHE.exists() else {}
_DEP = re.compile(r"^([\w:]+)\((.+)-(\d+)'*, (.+)-(\d+)'*\)$")


_PTB = {"(": "-LRB-", ")": "-RRB-", "[": "-LSB-", "]": "-RSB-", "{": "-LCB-", "}": "-RCB-"}


def _parser_words(toks):
    """Placeholders become 'Xref'; brackets are pre-escaped so tree strings stay well-formed."""
    return ["Xref" if PH.fullmatch(t.text) else _PTB.get(t.text, t.text) for t in toks]


def _key(words):
    return hashlib.sha1(" ".join(words).encode()).hexdigest()


def parse_batch(sentences, max_len=70):
    """Parse many sentences in ONE JVM call (pre-tokenized, so token i == parser leaf i).
    Results are cached on disk, which also makes reruns deterministic and fast."""
    todo = []
    for s in sentences:
        w = _parser_words(s.toks)
        if len(w) > max_len:
            continue
        k = _key(w)
        if k not in _parse_cache:
            todo.append((k, w))
    todo = list(dict(todo).items())
    if todo:
        with tempfile.TemporaryDirectory() as td:
            inp = Path(td) / "in.txt"
            inp.write_text("\n".join(" ".join(w) for _, w in todo) + "\n", encoding="utf-8")
            cmd = ["java", f"-mx{JAVA_HEAP}", "-cp", str(STANFORD_JAR),
                   "edu.stanford.nlp.parser.lexparser.LexicalizedParser",
                   "-nthreads", str(PARSER_THREADS), "-maxLength", str(max_len),
                   "-sentences", "newline", "-tokenized",
                   "-outputFormat", "oneline,typedDependencies", "-outputFormatOptions", "basicDependencies",
                   str(PCFG_MODEL), str(inp)]
            res = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
            if res.returncode != 0:
                raise RuntimeError(res.stderr[-2000:])
        chunks = [c for c in res.stdout.strip().split("\n\n")]
        assert len(chunks) == len(todo), f"parser returned {len(chunks)} parses for {len(todo)} sentences"
        for (k, w), ch in zip(todo, chunks):
            lines = ch.strip().split("\n")
            tree = lines[0] if lines and lines[0].startswith("(ROOT") else None
            deps = []
            for ln in lines[1:]:
                m = _DEP.match(ln.strip())
                if m:
                    deps.append((m.group(1), int(m.group(3)), int(m.group(5))))
            _parse_cache[k] = {"tree": tree, "deps": deps}
        PARSE_CACHE.write_text(json.dumps(_parse_cache))
    for s in sentences:
        w = _parser_words(s.toks)
        rec = _parse_cache.get(_key(w)) if len(w) <= max_len else None
        if rec and rec["tree"]:
            try:
                t = Tree.fromstring(rec["tree"])
            except ValueError:
                continue
            if len(t.leaves()) == len(s.toks):
                s.parse = (t, [tuple(d) for d in rec["deps"]])


def subtree_span(deps, head):
    """All token indexes (1-based) dominated by `head` in a dependency list."""
    kids = defaultdict(list)
    for rel, h, d in deps:
        kids[h].append(d)
    out, stack = set(), [head]
    while stack:
        x = stack.pop()
        if x in out:
            continue
        out.add(x); stack.extend(kids[x])
    return out


# quick self-test on two sentences
_bl = Block(0, "t", 0, 0, "body", "The samples were analysed by the team. We ran the model ⟦1⟧ again.", {"⟦1⟧": "[1]"})
_ss = split_sentences(_bl)
parse_batch(_ss)
assert len(_ss) == 2 and _ss[0].parse is not None, "parser self-test failed"
assert ("nsubjpass", 4, 2) in _ss[0].parse[1]
print("Parser OK:", _ss[0].parse[0])

Parser OK: (ROOT
  (S
    (NP (DT The) (NNS samples))
    (VP
      (VBD were)
      (VP (VBN analysed) (PP (IN by) (NP (DT the) (NN team)))))
    (. .)))


## 8 · Analysis before editing
Document profile: glossary of technical terms and acronyms (never varied), dominant US/UK spelling, hedge density per section, sentence-length profile, opener repetition, readability.

In [8]:
#@title 8 · Analysis before any edit: document profile, style ledger and diagnostics
import language_tool_python, textstat
from abbreviations import schwartz_hearst
from proselint.tools import LintFile
from proselint.checks import __register__ as _PL_CHECKS
from proselint.registry import CheckRegistry
CheckRegistry().register_many(_PL_CHECKS)

_LT = None


def _harden_languagetool():
    """language_tool_python gives a new LanguageTool server 15 s to start, and on timeout starts ANOTHER one
    on a new port without stopping the first. On a busy machine that cascades into many 0.5-1 GB Java
    processes. Give it 3 minutes, and stop a server whose start failed before anything retries."""
    import language_tool_python.server as _lts
    if getattr(_lts.LanguageTool, "_dnwe_hardened", False):
        return
    _lts.LanguageTool._wait_for_server_ready.__defaults__ = (180,)
    _orig_start = _lts.LanguageTool._start_local_server

    def _start_or_reap(self):
        try:
            _orig_start(self)
        except Exception:
            srv = getattr(self, "_server", None)
            if srv is not None and srv.poll() is None:
                srv.terminate()
                try:
                    srv.wait(10)
                except Exception:
                    srv.kill()
            raise
    _lts.LanguageTool._start_local_server = _start_or_reap
    _lts.LanguageTool._dnwe_hardened = True


def lt():
    """One local LanguageTool server per session (localhost only); stopped when Python exits."""
    global _LT
    if _LT is None:
        _harden_languagetool()
        _LT = language_tool_python.LanguageTool("en-US", language_tool_download_version=LT_VERSION)
    return _LT


def plain_text(doc):
    return "\n\n".join(unmask(b.masked, b.spans) if doc.fmt != "docx" else
                       re.sub(f"[{chr(PUA)}-{chr(PUA + 4095)}]", " ", unmask(b.masked, b.spans))
                       for b in doc.blocks)


def readable_text(masked):
    """Masked text with placeholders replaced by a neutral word (for metrics, LanguageTool, proselint)."""
    return PH.sub("Xref", masked)


def proselint_count(text):
    return len(LintFile("<doc>", text).lint())


@dataclass
class Profile:
    words: int
    sentences: int
    glossary: set          # multiword or rare technical terms, as lemma strings
    glossary_words: set    # every word inside a term + capitalised names: never substituted
    acronyms: dict
    spelling: str
    section_words: Counter
    hedge_density: dict
    opener_runs: list
    sentence_lengths: list
    stats: dict
    vocab: set = field(default_factory=set)
    vocab_counts: dict = field(default_factory=dict)


def build_profile(doc, sents_by_block):
    all_s = [s for ss in sents_by_block.values() for s in ss]
    words = sum(1 for s in all_s for t in s.toks if t.text[:1].isalpha())
    text = plain_text(doc)
    # acronyms (Schwartz-Hearst) and their definitions become glossary terms
    try:
        acr = schwartz_hearst.extract_abbreviation_definition_pairs(doc_text=text)
    except Exception:
        acr = {}
    glossary, gwords = set(), set()      # terms (lemma strings) and the words inside them
    for a, d in acr.items():
        glossary.add(a.lower()); gwords.add(a.lower())
        ws = [lemma(w, "NN") for w in re.findall(r"[A-Za-z]{2,}", d)]
        if ws:
            glossary.add(" ".join(ws)); gwords.update(ws)
    # recurring technical n-grams: nouns/adjectives spread across >=2 blocks, rare in general English
    ng_blocks = defaultdict(set)
    for s in all_s:
        content = [(t.lem, t.tag) for t in s.toks]
        for i in range(len(content)):
            for n in (1, 2, 3):
                g = content[i:i + n]
                if len(g) == n and all(tg[:2] in ("NN", "JJ") for _, tg in g) and g[-1][1].startswith("NN"):
                    ng_blocks[" ".join(l for l, _ in g)].add(s.block.bid)
    for g, bls in ng_blocks.items():
        if len(bls) >= 2:
            ws = g.split()
            if (len(ws) > 1 and any(zipf(w) < 4.5 for w in ws)) or (len(ws) == 1 and zipf(ws[0]) < 3.5):
                glossary.add(g); gwords.update(ws)
    # capitalised words mid-sentence (names, models, datasets) are never substituted
    for s in all_s:
        for t in s.toks[1:]:
            if t.text[:1].isupper() and t.text.isalpha():
                gwords.add(t.text.lower()); gwords.add(t.lem)
    # dominant spelling variant
    ize = len(re.findall(r"\b\w+(?:ize|izes|ized|izing|ization|yze|yzed|yzes)\b", text))
    ise = len([w for w in re.findall(r"\b\w+(?:ise|ises|ised|ising|isation|yse|ysed|yses)\b", text)
               if zipf(w) > 1.5 and w.lower() not in {"otherwise", "precise", "exercise", "rise", "wise", "likewise",
                                                       "promise", "expertise", "noise", "concise", "advise", "raise",
                                                       "surprise", "comprise", "arise", "premise", "devise", "revise"}])
    spelling = "uk" if ise > ize else "us"
    sec_words, hedges = Counter(), Counter()
    for s in all_s:
        n = sum(1 for t in s.toks if t.text[:1].isalpha())
        sec_words[s.block.section] += n
        hedges[s.block.section] += sum(1 for t in s.toks if t.text.lower() in HEDGES)
    hedge_density = {k: round(100 * hedges[k] / max(1, v), 2) for k, v in sec_words.items()}
    runs = []
    for bid, ss in sents_by_block.items():
        op = [s.toks[0].text.lower() for s in ss]
        for i in range(len(op) - 2):
            if op[i] == op[i + 1] == op[i + 2]:
                runs.append((bid, i, op[i]))
    lens = [sum(1 for t in s.toks if t.text[:1].isalnum() or PH.fullmatch(t.text)) for s in all_s]
    rt = readable_text("\n\n".join(b.masked for b in doc.blocks))
    stats = {"fk_grade": textstat.flesch_kincaid_grade(rt), "mean_sentence_len": round(statistics.mean(lens), 1) if lens else 0,
             "sd_sentence_len": round(statistics.pstdev(lens), 1) if lens else 0}
    vocab = {t.text.lower() for s in all_s for t in s.toks if t.text.isalpha()}
    vocab_counts = defaultdict(list)
    for s in all_s:
        for t in s.toks:
            if t.text.isalpha():
                vocab_counts[t.text.lower()].append(s.block.bid)
    return Profile(words, len(all_s), glossary, gwords, acr, spelling, sec_words, hedge_density, runs, lens, stats, vocab, dict(vocab_counts))


def lt_matches(bl):
    """LanguageTool on one block; placeholders are shown to it as 'Xref' and offsets mapped back."""
    seg, mapping, pos = [], [], 0
    for m in re.finditer(r"⟦\d+⟧|[^⟦]+", bl.masked):
        s = m.group()
        r = "Xref" if PH.fullmatch(s) else s
        mapping.append((pos, pos + len(r), m.start(), m.end(), bool(PH.fullmatch(s))))
        seg.append(r); pos += len(r)
    txt = "".join(seg)
    out = []
    for mt in lt().check(txt):
        a, b = mt.offset, mt.offset + mt.error_length
        hit = [mp for mp in mapping if mp[0] < b and a < mp[1]]
        if len(hit) != 1 or hit[0][4]:
            continue                        # spans a placeholder: never touch protected content
        lo, hi, ma, mb, _ = hit[0]
        out.append({"start": ma + (a - lo), "end": ma + (b - lo), "rule": mt.rule_id,
                    "category": getattr(mt, "category", ""), "message": mt.message,
                    "replacements": list(mt.replacements[:3])})
    return out

## 9 · Style profile: analysis before transformation
Before anything is rewritten, the engine measures how the passage is written and compares it with the writing mode's targets. Only markers outside their target unlock the reconstruction families of section 11, and only in the paragraphs or sentences where the marker is out of range.

| Marker | What it counts | What it unlocks (section 11) |
|---|---|---|
| sentence-length variation | coefficient of variation of sentence length inside each paragraph | *merge* two short sentences; *split* contrastive *but* clauses sooner |
| additive openers / 100 sentences | sentence-initial *Moreover, Furthermore, Additionally, In addition…* | *discourse*: drop the most crowded ones until the density reaches the target |
| opener repetition | adjacent sentences that begin with the same word | *opener*: front a sentence-final adverbial clause or phrase |
| stock phrasing / 1k words | *delve into, leverage, showcase, plays a crucial role in, a myriad of…* | *cliche* (always on) |
| hedge stacks / 1k words | *may potentially, could possibly* | *hedge_stack* (always on) |
| dashes / 1k words | parenthetical dash pairs | *dash* |
| contractions / 1k words | *don't, it's* | *register* (follows the mode) |

The targets in `MODE_TARGETS` are editable heuristics, calibrated so that the team's own papers in `corpus/` mostly sit inside them. Re-calibrate them as the corpus grows.

In [9]:
#@title 9 · Style profile: measure HOW the text is written, then plan the reconstruction
# The humanizer never rewrites because a rule *can* fire. It first measures the passage's style
# markers (sentence-length variation, transition stacking, opener repetition, stock phrasing, dash
# use, hedge stacking, register) against the writing mode's targets, and only the markers that are
# out of range unlock the reconstruction families of section 11 — and only where they are out of range.
# Every marker is a plain count or ratio computed here; nothing is learned or predicted.
import math

# ---- Style lexicons (closed, curated lists; extend them for your own field) --------------
# Sentence-initial additive connectives. Adjacent sentences are already read as "and", so
# dropping these never changes meaning; contrast/cause/condition connectives are never dropped.
ADDITIVE_OPENERS = ["in addition", "moreover", "furthermore", "additionally", "also", "besides",
                    "further", "what is more", "on top of that"]
# Summary openers that are redundant inside a section already titled Conclusion/Summary.
SUMMARY_OPENERS = ["in conclusion", "in summary", "to summarize", "to summarise", "to conclude", "overall"]
# Stock vocabulary and phrasing that over-dense machine-drafted prose leans on. MEASURED here;
# only the subset with a meaning-preserving replacement is ever rewritten (section 11).
STOCK_WORDS = {"delve", "leverage", "showcase", "underscore", "foster", "bolster", "streamline",
               "spearhead", "revolutionize", "revolutionise", "harness", "boast", "embark", "pivotal",
               "seamless", "seamlessly", "intricate", "multifaceted", "myriad", "plethora", "paramount",
               "meticulous", "meticulously", "noteworthy", "cutting-edge", "ever-evolving", "ever-changing",
               "tapestry", "testament", "realm", "landscape", "navigate", "unlock", "unleash", "elevate",
               "empower", "holistic", "synergy", "game-changer", "commendable", "transformative"}
STOCK_PHRASES = ["it is important to note", "it is worth noting", "it is essential to note", "it is crucial to note",
                 "it is worth mentioning", "plays a crucial role", "plays a key role", "plays a vital role",
                 "plays a pivotal role", "plays a central role", "a testament to", "in the realm of",
                 "in today's", "when it comes to", "shed light on", "pave the way", "serves as a",
                 "stands as a", "not only", "a wide range of", "of paramount importance", "of utmost importance",
                 "in the ever-evolving", "in the rapidly evolving", "harness the power", "at the end of the day"]
DASHES = re.compile(r"\s(?:—|–|---|--)\s|(?<=[A-Za-z])—(?=[A-Za-z])")
CONTRACTION_RE = re.compile(r"\b[A-Za-z]+(?:n['’]t|['’](?:re|ve|ll|m|d|s))\b")
_HEDGE_PAIR = {"may", "might", "could", "can", "would"}
_HEDGE_ADV = {"potentially", "possibly", "perhaps", "conceivably"}

# ---- Targets per writing mode (editable heuristics, NOT facts about any detector) ----------
# Calibrated on the team's own corpus (8 documents, 2026-11-23): their median paragraph CV is
# 0.31-0.47, so the academic floor sits at 0.30 to leave ordinary paragraphs alone. Re-calibrate
# when the corpus grows: style_markers() on each document prints every value used here.
# cv_min: minimum coefficient of variation of sentence length inside a paragraph (sd / mean).
# additive_per_100s: sentence-initial additive connectives per 100 sentences.
# opener_repeat_max: share of adjacent sentence pairs that begin with the same word.
# dash_per_1k: dashes per 1,000 words.  stock_per_1k: stock words/phrases per 1,000 words.
MODE_TARGETS = {
    "academic":       {"cv_min": 0.30, "additive_per_100s": 6.0, "opener_repeat_max": 0.15, "dash_per_1k": 1.5, "stock_per_1k": 1.0},
    "technical":      {"cv_min": 0.28, "additive_per_100s": 5.0, "opener_repeat_max": 0.20, "dash_per_1k": 1.0, "stock_per_1k": 1.0},
    "general":        {"cv_min": 0.35, "additive_per_100s": 5.0, "opener_repeat_max": 0.15, "dash_per_1k": 2.5, "stock_per_1k": 1.0},
    "conversational": {"cv_min": 0.40, "additive_per_100s": 4.0, "opener_repeat_max": 0.20, "dash_per_1k": 3.0, "stock_per_1k": 1.5},
}
# How far each intensity pushes toward the target (multipliers on the targets above).
INTENSITY_TARGET_SCALE = {"conservative": 1.0, "balanced": 1.0, "extensive": 0.5}
MERGE_MAX_EACH, MERGE_MAX_TOTAL = 16, 30          # merging never creates a sentence above 30 words


def _nwords(s):
    return sum(1 for t in s.toks if t.text[:1].isalnum() or PH.fullmatch(t.text))


def _opener_word(s):
    return s.toks[0].text.lower() if s.toks else ""


def _starts_with(s, phrases):
    """Length (in tokens) of a sentence-initial phrase from `phrases` followed by a comma, else 0."""
    low = [t.text.lower().replace("’", "'") for t in s.toks]
    for ph in sorted(phrases, key=len, reverse=True):
        w = ph.split()
        if low[:len(w)] == w and len(low) > len(w) + 1 and low[len(w)] == ",":
            return len(w)
    return 0


def _cv(xs):
    return (statistics.pstdev(xs) / statistics.mean(xs)) if len(xs) >= 2 and statistics.mean(xs) > 0 else None


def style_markers(sents_by_block, sections=None):
    """Document-level style markers from segmented sentences {bid: [Sentence]}. Pure counting."""
    all_s = [s for bid in sorted(sents_by_block) for s in sents_by_block[bid]]
    words = sum(_nwords(s) for s in all_s) or 1
    n = len(all_s) or 1
    lens = [_nwords(s) for s in all_s]
    para_cv = [c for bid in sorted(sents_by_block)
               if len(sents_by_block[bid]) >= 3 and (c := _cv([_nwords(s) for s in sents_by_block[bid]])) is not None]
    pairs = [(a, b) for bid in sorted(sents_by_block) for a, b in zip(sents_by_block[bid], sents_by_block[bid][1:])]
    rep = sum(1 for a, b in pairs if _opener_word(a) == _opener_word(b))
    low = [t.text.lower().replace("’", "'") for s in all_s for t in s.toks]
    lem = [t.lem for s in all_s for t in s.toks]
    joined = " " + " ".join(low) + " "
    stock_w = sum(1 for w, l in zip(low, lem) if w in STOCK_WORDS or l in STOCK_WORDS)
    stock_p = sum(joined.count(f" {p} ") for p in STOCK_PHRASES)
    hedge_stack = sum(1 for a, b in zip(low, low[1:]) if a in _HEDGE_PAIR and b in _HEDGE_ADV)
    dashes = sum(len(DASHES.findall(s.text)) for s in all_s)
    contractions = sum(len(CONTRACTION_RE.findall(s.text)) for s in all_s)
    additive = sum(1 for s in all_s if _starts_with(s, ADDITIVE_OPENERS))
    triads = sum(len(re.findall(r"\w[\w\- ]{0,30}, \w[\w\- ]{0,30},? and \w", s.text)) for s in all_s)
    return {
        "sentences": len(all_s), "words": words,
        "sent_len_mean": round(statistics.mean(lens), 2) if lens else 0,
        "sent_len_cv": round(_cv(lens), 3) if _cv(lens) is not None else None,
        "para_len_cv_median": round(statistics.median(para_cv), 3) if para_cv else None,
        "additive_per_100s": round(100 * additive / n, 2),
        "opener_repeat_rate": round(rep / max(1, len(pairs)), 3),
        "stock_per_1k": round(1000 * (stock_w + stock_p) / words, 2),
        "hedge_stacks_per_1k": round(1000 * hedge_stack / words, 2),
        "dash_per_1k": round(1000 * dashes / words, 2),
        "contractions_per_1k": round(1000 * contractions / words, 2),
        "triads_per_100s": round(100 * triads / n, 2),
    }


def markers_vs_target(m, mode, intensity="balanced"):
    """Which markers are outside the mode's target range. Returns {marker: (value, target, ok)}."""
    tg, k = MODE_TARGETS[mode], INTENSITY_TARGET_SCALE[intensity]
    out = {}
    if m.get("para_len_cv_median") is not None:
        out["para_len_cv_median"] = (m["para_len_cv_median"], f">= {tg['cv_min']}", m["para_len_cv_median"] >= tg["cv_min"])
    out["additive_per_100s"] = (m["additive_per_100s"], f"<= {tg['additive_per_100s'] * k:g}", m["additive_per_100s"] <= tg["additive_per_100s"] * k)
    out["opener_repeat_rate"] = (m["opener_repeat_rate"], f"<= {tg['opener_repeat_max']}", m["opener_repeat_rate"] <= tg["opener_repeat_max"])
    out["stock_per_1k"] = (m["stock_per_1k"], f"<= {tg['stock_per_1k']}", m["stock_per_1k"] <= tg["stock_per_1k"])
    out["dash_per_1k"] = (m["dash_per_1k"], f"<= {tg['dash_per_1k'] * k:g}", m["dash_per_1k"] <= tg["dash_per_1k"] * k)
    out["hedge_stacks_per_1k"] = (m["hedge_stacks_per_1k"], "= 0", m["hedge_stacks_per_1k"] == 0)
    return out


@dataclass
class HumanPlan:
    """Where reconstruction is warranted. Sets hold (bid, sentence_idx) or bid; all built in sorted order."""
    mode: str
    intensity: str
    thin: set = field(default_factory=set)             # sentences whose additive opener should go
    summary: set = field(default_factory=set)          # redundant "In conclusion," inside a conclusion
    opener: set = field(default_factory=set)           # sentences whose opening should be varied
    low_variation: set = field(default_factory=set)    # blocks whose sentence lengths are too uniform
    merge_pairs: dict = field(default_factory=dict)    # bid -> [(i, i+1)]
    dash: bool = False
    split_min_low_var: int = 26
    notes: list = field(default_factory=list)


def plan_humanizer(sents_by_block, block_plan, markers, mode, intensity):
    """Turn out-of-range markers into a concrete, deterministic plan. `block_plan` maps the blocks
    that may be edited to (effective_intensity, families, split_min)."""
    tg, k = MODE_TARGETS[mode], INTENSITY_TARGET_SCALE[intensity]
    hp = HumanPlan(mode, intensity)
    bids = sorted(block_plan)
    all_s = [(bid, s) for bid in bids for s in sents_by_block[bid]]
    n = max(1, len(all_s))

    # 1 · transitions: drop the most crowded additive openers until the density reaches the target
    cands = [(bid, s) for bid, s in all_s if _starts_with(s, ADDITIVE_OPENERS) and "discourse" in block_plan[bid][1]]
    allowed = math.floor(tg["additive_per_100s"] * k * n / 100)
    if len(cands) > allowed:
        conn_pos = [i for i, (bid, s) in enumerate(all_s) if _starts_with(s, ADDITIVE_OPENERS + SUMMARY_OPENERS)
                    or (s.toks and s.toks[0].text.lower() in set().union(*CONNECTIVE_CLASS.values()))]
        index = {(bid, s.idx): i for i, (bid, s) in enumerate(all_s)}

        def crowding(c):
            i = index[(c[0], c[1].idx)]
            gaps = [abs(i - j) for j in conn_pos if j != i]
            return (min(gaps) if gaps else 99, i)        # closest neighbour first, then document order
        def opens_with(bid, i):
            """First word of sentence i of block bid once the planned drops are applied."""
            ss = sents_by_block[bid]
            if not 0 <= i < len(ss):
                return None
            s = ss[i]
            m = _starts_with(s, ADDITIVE_OPENERS) if (bid, s.idx) in hp.thin else 0
            return s.toks[m + 1 if m else 0].text.lower()
        kept_for_variety = 0
        for bid, s in sorted(cands, key=crowding):
            if len(cands) - len(hp.thin) <= allowed:
                break
            hp.thin.add((bid, s.idx))
            new = opens_with(bid, s.idx)
            if new in (opens_with(bid, s.idx - 1), opens_with(bid, s.idx + 1)):
                hp.thin.discard((bid, s.idx))      # "Moreover, it ... Furthermore, it ..." -> "It ... It ..."
                kept_for_variety += 1
        hp.notes.append(f"{len(cands)} additive openers for {n} sentences (target {tg['additive_per_100s'] * k:g}/100): "
                        f"plan drops {len(hp.thin)}" + (f", keeps {kept_for_variety} that would leave two sentences "
                                                        f"opening with the same word" if kept_for_variety else ""))

    # 2 · summary openers that repeat the section heading
    for bid, s in all_s:
        if s.block.section == "conclusion" and _starts_with(s, SUMMARY_OPENERS) and "discourse" in block_plan[bid][1]:
            hp.summary.add((bid, s.idx))

    # 3 · repeated openers: every sentence that begins like the one before it (if the document is
    #     over the target), otherwise only the third sentence of a run of three
    pairs = [(bid, a, b) for bid in bids for a, b in zip(sents_by_block[bid], sents_by_block[bid][1:])]
    rate = sum(1 for _, a, b in pairs if _opener_word(a) == _opener_word(b)) / max(1, len(pairs))
    over = rate > tg["opener_repeat_max"] or intensity == "extensive"
    for bid in bids:
        if "opener" not in block_plan[bid][1]:
            continue
        ss = sents_by_block[bid]
        for i in range(1, len(ss)):
            same = _opener_word(ss[i]) == _opener_word(ss[i - 1])
            run3 = i >= 2 and same and _opener_word(ss[i - 2]) == _opener_word(ss[i])
            if same and (over or run3) and not _starts_with(ss[i], ADDITIVE_OPENERS):
                hp.opener.add((bid, ss[i].idx))
    if hp.opener:
        hp.notes.append(f"opener repetition {rate:.2f} (target <= {tg['opener_repeat_max']}): {len(hp.opener)} sentences to vary")

    # 4 · sentence-length variation per paragraph: uniform paragraphs get one merge per four sentences
    cv_min = tg["cv_min"] + (0.05 if intensity == "extensive" else 0.0)
    for bid in bids:
        ss = sents_by_block[bid]
        if len(ss) < 3:
            continue
        cv = _cv([_nwords(s) for s in ss])
        if cv is None or cv >= cv_min:
            continue
        hp.low_variation.add(bid)
        if "merge" not in block_plan[bid][1]:
            continue
        budget = max(1, len(ss) // 4)
        # the merged sentence opens like ss[i]; it must not then sit next to a sentence that opens the same way
        elig = [(i, i + 1) for i in range(len(ss) - 1) if merge_eligible(ss[i], ss[i + 1])
                and not (i + 2 < len(ss) and _opener_word(ss[i + 2]) == _opener_word(ss[i]))]
        chosen, used = [], set()
        for i, j in sorted(elig, key=lambda p: (_nwords(ss[p[0]]) + _nwords(ss[p[1]]), p[0])):
            if len(chosen) >= budget or i in used or j in used:
                continue
            chosen.append((i, j)); used |= {i, j}
        if chosen:
            hp.merge_pairs[bid] = sorted(chosen)
    if hp.low_variation:
        hp.notes.append(f"{len(hp.low_variation)} paragraphs with uniform sentence lengths (cv < {cv_min:.2f}); "
                        f"{sum(len(v) for v in hp.merge_pairs.values())} merges planned")

    # 5 · dashes: rewrite parenthetical dash pairs only when the document over-uses dashes
    hp.dash = markers["dash_per_1k"] > tg["dash_per_1k"] * k
    if hp.dash:
        hp.notes.append(f"dashes {markers['dash_per_1k']}/1k words (target <= {tg['dash_per_1k'] * k:g})")
    return hp


# Subjects a second sentence may start with so that "A. It B." -> "A, and it B." reads naturally.
MERGE_SUBJECTS = {"it", "they", "this", "these", "we", "he", "she", "its", "their", "our", "the", "i"}


def merge_eligible(a, b):
    """Two short declarative sentences, each with its own verb, the second opening with a plain
    subject (not a connective), neither already carrying a coordinated clause, list or question."""
    na, nb = _nwords(a), _nwords(b)
    if na > MERGE_MAX_EACH or nb > MERGE_MAX_EACH or na + nb > MERGE_MAX_TOTAL or na < 4 or nb < 4:
        return False
    if a.toks[-1].text != "." or b.toks[-1].text != ".":
        return False
    if b.toks[0].text.lower() not in MERGE_SUBJECTS:
        return False
    for s in (a, b):
        txt = [t.text for t in s.toks]
        if ";" in txt or ":" in txt or "?" in txt or not any(t.tag.startswith("VB") or t.tag == "MD" for t in s.toks):
            return False
    if re.search(r",\s+(and|but|or)\s", a.text) or re.search(r",\s+(and|but|or)\s", b.text):
        return False
    if PH.fullmatch(a.toks[0].text) or PH.fullmatch(b.toks[0].text):
        return False
    if [t.text.lower() for t in a.toks[:2]] == [t.text.lower() for t in b.toks[:2]]:
        return False                                 # "The framework ..., and the framework ..." repeats the subject
    if any(t.text.lower() in NEGATIONS or t.text.lower().endswith(("n't", "n’t")) for s in (a, b) for t in s.toks):
        return False                                 # "X does not Y because Z. They are ..." corrects, it does not add
    return True


## 10 · Rule families
| Family | Example | Default |
|---|---|---|
| wordy | *due to the fact that* → *because* | all intensities |
| redundancy | *past history* → *history* | all |
| light_verb | *made a decision* → *decided* | all |
| transition | stacked *Moreover… Furthermore…* → drop the second | all |
| grammar | LanguageTool, high-precision rules only (a/an, repeated word, its/it's, clear typos that keep their first letter) | all |
| nominalization | *conducted an analysis of X* → *analysed X* | balanced+ |
| plain | *utilize* → *use*, *assist in* → *help with* (curated table; terms of art such as *terminate* only in general/conversational) | balanced+ |
| passive | *X was measured by Y* → *Y measured X* (simple agents only; re-parsed to verify) | balanced+ |
| split | *S, but S* → *S. However, S*; *S, and S* only at 45+ words; *S; S* only in general/conversational; the new sentence must open with its own subject | balanced+ |
| lexical | open-ended PPDB + WordNet (dominant sense both ways) | extensive |
| repetition | third use of the same verb/adjective in a paragraph | extensive |

Methods and Results sections are always edited at *conservative*; References, Acknowledgements and Title are never edited, and neither are non-prose blocks (command listings, flattened tables).

*Hardened 2026-11-23 against the labelled dev/test sets.* Semicolon splits had been judged wrong 5/5. Protected numbers had swallowed the sentence-final full stop and merged sentences. *prior to* had been rewritten where *prior* is a noun. A spelling fix had turned the command `squeue` into `Queue`. Each case now has a regression test in section 17.

In [10]:
#@title 10 · Rule families: each proposes edits with a precondition, an action and a declared footprint
# Every Edit declares exactly which lemmas it may remove (removed_ok) and add (added_ok);
# the meaning gate (section 10) rejects anything outside that footprint.


@dataclass
class Edit:
    start: int
    end: int
    new: str
    family: str
    rule: str
    reason: str
    signal: str
    removed_ok: frozenset = frozenset()
    added_ok: frozenset = frozenset()
    structural: bool = False
    verify: object = None            # optional structural check on the re-parsed sentence
    sentence: object = None
    conf: float = 0.0
    conf_source: str = "prior"


PRIORS = {"wordy": 0.95, "redundancy": 0.93, "light_verb": 0.90, "grammar": 0.88, "transition": 0.86,
          "plain": 0.88, "nominalization": 0.85, "passive": 0.78, "split": 0.78, "lexical": 0.64,
          "repetition": 0.62}


@dataclass
class Ctx:
    profile: object
    mode: str
    intensity: str
    fmt: str
    block_sents: list = None
    lt: list = None
    subs: dict = field(default_factory=dict)      # substitution memory: lemma -> chosen lemma
    rep_seen: Counter = field(default_factory=Counter)
    hplan: object = None                          # humanizer plan from the style profile (section 9)


def _lems(text):
    toks = tag_tokens(tokenize(text))
    return ({t.lem for t in toks if t.text[:1].isalpha()} | {canon(t.text) for t in toks if t.text[:1].isalpha()}
            | {w.lower() for w in re.findall(r"[A-Za-z']+", text)})


def author_spelling(form, profile):
    """Use the British form if the document is British or the author already wrote that form."""
    uk = uk_variant(form)
    if uk == form:
        return form
    stem = re.sub(r"(?:is|ys)(?:e|ed|es|ing|ation|ations)$", "", uk)
    same_word_uk = any(w.startswith(stem) and re.search(r"(?:is|ys)(?:e|ed|es|ing|ation|ations)$", w[len(stem):])
                       for w in profile.vocab)
    return uk if (profile.spelling == "uk" or same_word_uk) else form


def _cap(s):
    return s[:1].upper() + s[1:] if s else s


def _decap(s, tok):
    """Lower-case a sentence-initial word moved mid-sentence, unless it is a name, an acronym or a
    mixed-case term ("LLM-based", "GPT-4", "iPhone", "QM9"): any capital after the first letter keeps it."""
    if (tok.tag in ("NNP", "NNPS") or tok.text.isupper() or PH.fullmatch(tok.text) or tok.text == "I"
            or any(ch.isupper() for ch in tok.text[1:])):
        return s
    return s[:1].lower() + s[1:]


def _phrase_hits(s, phrase):
    ptoks = [t.text.lower() for t in tokenize(phrase)]
    n, T = len(ptoks), s.toks
    for i in range(len(T) - n + 1):
        if all(T[i + k].text.lower() == ptoks[k] for k in range(n)):
            yield i, i + n - 1


def _replace_phrase(s, i, j, repl, family, rule, reason, extra_removed=(), removes_lead=False):
    T, txt = s.toks, s.block.masked
    start, end = T[i].start, T[j].end
    removed = _lems(txt[start:end]) | set(extra_removed)
    added = _lems(repl) if repl else set()
    if repl == "":
        if j + 1 >= len(T):
            return None
        nxt = T[j + 1]
        if nxt.text in {",", ".", ";", ":"}:
            return None
        cap = i == 0 or txt[T[i].start:T[i].start + 1].isupper()   # phrase opened a sentence
        new = _cap(nxt.text) if cap else nxt.text
        end = nxt.end
        if i > 0:                                      # also drop the space before the phrase
            start = T[i - 1].end
            new = (" " if txt[start:T[i].start] else "") + new
        return Edit(start, end, new, family, rule, reason, f"matched '{txt[T[i].start:T[j].end]}'",
                    frozenset(removed), frozenset({nxt.lem, nxt.text.lower()}))
    if i == 0:
        repl = _cap(repl)
    elif txt[start:start + 1].isupper() and T[i].tag not in ("NNP", "NNPS"):
        repl = _cap(repl)
    return Edit(start, end, repl, family, rule, reason, f"matched '{txt[start:end]}'",
                frozenset(removed), frozenset(added))


def fam_wordy(s, ctx):
    out = []
    for pat, repl, cond in WORDY:
        for i, j in _phrase_hits(s, pat):
            T = s.toks
            if pat in WORDY_GUARDS and not WORDY_GUARDS[pat](T, i, j):
                continue
            if cond:
                if j + 1 >= len(T):
                    continue
                want = cond.rstrip("*")
                if T[j + 1].tag != want:
                    continue
                if cond.endswith("*"):                  # "is capable of analysing" -> "can analyse"
                    v = inflect(T[j + 1].lem, "VB")
                    if not v:
                        continue
                    e = _replace_phrase(s, i, j + 1, f"{repl} {v}", "wordy", f"wordy:{pat}", f"'{pat} + -ing' is a wordy form of '{repl}'")
                    if e:
                        out.append(e)
                    continue
            e = _replace_phrase(s, i, j, repl, "wordy", f"wordy:{pat}",
                                f"'{pat}' says the same as '{repl}'" if repl else f"'{pat}' adds no content")
            if e:
                out.append(e)
    return out


def fam_redundancy(s, ctx):
    out = []
    for pat, keep in REDUNDANT:
        for i, j in _phrase_hits(s, pat):
            e = _replace_phrase(s, i, j, keep, "redundancy", f"redundant:{pat}", f"'{pat}' repeats its own meaning")
            if e:
                out.append(e)
    return out


def _derived_verb(noun_lem):
    if not re.search(r"(tion|sion|ment|ance|ence|sis|ure|al)$", noun_lem):
        return None
    best = None
    for lm in wn.lemmas(noun_lem, pos=wn.NOUN):
        for rel in lm.derivationally_related_forms():
            v = rel.name()
            if rel.synset().pos() == "v" and "_" not in v and v[:4] == noun_lem[:4]:
                if best is None or zipf(v) > zipf(best):
                    best = v
    return best


def fam_light_verb(s, ctx):
    out, T = [], s.toks
    for i, t in enumerate(T):
        if not t.tag.startswith("VB") or t.lem not in LIGHT_VERBS:
            continue
        if t.tag == "VBN" and i > 0 and T[i - 1].lem in ("be", "get"):
            continue                                    # passive light verb: leave it
        j = i + 1
        lv = t.lem
        if lv == "carry":
            if j >= len(T) or T[j].text.lower() != "out":
                continue
            j += 1
        if lv == "come":
            if j >= len(T) or T[j].text.lower() != "to":
                continue
            j += 1
        det = None
        if j < len(T) and T[j].text.lower() in ("a", "an", "the"):
            det = T[j].text.lower(); j += 1
        if j >= len(T) or not T[j].tag.startswith("NN") or T[j].tag.startswith("NNP"):
            continue
        noun = T[j]
        has_of = j + 1 < len(T) and T[j + 1].text.lower() == "of"
        verb = LIGHT_TABLE.get((lv, noun.lem))
        family = "light_verb"
        if verb and has_of and verb not in TRANSITIVE:
            continue                                   # "make the decisions of X" != "decide X"
        if verb is None:
            if lv not in ("conduct", "perform", "carry", "undertake", "make") or not has_of:
                continue
            verb = _derived_verb(noun.lem)
            family = "nominalization"
            if not verb:
                continue
        if noun.lem in ctx.profile.glossary:
            continue
        stag = surface_tag(t.lem, t.text, t.tag)
        form = inflect(verb, stag) if stag else None
        if not form:
            continue
        form = author_spelling(form, ctx.profile)
        end_idx = j + 1 if has_of else j
        if not has_of and (lv, noun.lem) not in LIGHT_TABLE:
            continue
        if has_of and lv == "make" and noun.lem == "use" and det:   # "make the use of" is rare; skip
            continue
        start, end = t.start, T[end_idx].end
        new = _cap(form) if i == 0 else form
        removed = {lv, noun.lem, noun.text.lower(), t.text.lower(), "of", "out", "to"} | ({det} if det else set())
        out.append(Edit(start, end, new, family, f"{family}:{lv}+{noun.lem}",
                        f"'{s.block.masked[start:end]}' is a noun phrase hiding the verb '{verb}'",
                        f"light verb '{lv}' + nominal '{noun.lem}'", frozenset(removed), frozenset({verb, lemma(form, 'VB')})))
    return out


_ADD_TR = sorted(ADDITION_TRANSITIONS, key=len, reverse=True)


def _addition_opener(sent):
    low = [t.text.lower() for t in sent.toks]
    for tr in _ADD_TR:
        w = tr.split()
        if low[:len(w)] == w and len(low) > len(w) and low[len(w)] == ",":
            return len(w)
    return 0


def _planned_drop(s, ctx):
    """Tokens at the start of `s` that the humanizer plan (section 9) is already removing."""
    hp = ctx.hplan
    if hp is None or (s.block.bid, s.idx) not in (hp.thin | hp.summary):
        return 0
    return max(_starts_with(s, ADDITIVE_OPENERS), _starts_with(s, SUMMARY_OPENERS)) + 1


def _opener_after_plan(s, ctx):
    """First word of `s` once the plan's own opener drops are applied."""
    n = _planned_drop(s, ctx)
    return s.toks[n].text.lower() if n and n < len(s.toks) else s.toks[0].text.lower()


def fam_transition(s, ctx):
    if s.idx == 0 or not ctx.block_sents or _planned_drop(s, ctx):
        return []                                    # planned sentences are handled by 'discourse'
    n0 = _addition_opener(s)
    if n0 and len(s.toks) > n0 + 1:
        new_first = s.toks[n0 + 1].text.lower()
        nbrs = [ctx.block_sents[j] for j in (s.idx - 1, s.idx + 1) if 0 <= j < len(ctx.block_sents)]
        if any(_opener_after_plan(x, ctx) == new_first for x in nbrs):
            return []                                # dropping it would leave "It ... It ..."
    n = _addition_opener(s)
    if not n or not _addition_opener(ctx.block_sents[s.idx - 1]) or len(s.toks) <= n + 2:
        return []
    T = s.toks
    nxt = T[n + 1]
    removed = {t.text.lower() for t in T[:n]} | {t.lem for t in T[:n]}
    return [Edit(T[0].start, nxt.end, _cap(nxt.text), "transition", "transition:stacked-addition",
                 "two consecutive sentences open with an additive transition; the second one is dropped",
                 f"previous sentence also opens with '{ctx.block_sents[s.idx - 1].toks[0].text}'",
                 frozenset(removed), frozenset({nxt.lem, nxt.text.lower()}))]


# Only LanguageTool rules that proved precise on research prose are auto-applied; every other
# match is reported as advice. (On the test corpus TO_NON_BASE turned "close to calibrated" into
# "close to calibrate" and AGREEMENT_SENT_START broke "flags questions".)
GRAMMAR_AUTO = {"EN_A_VS_AN", "ENGLISH_WORD_REPEAT_RULE", "DT_DT", "IT_IS", "ITS_TO_IT_S", "IT_S_ITS",
                "THE_SUPERLATIVE", "MORFOLOGIK_RULE_EN_US", "MORFOLOGIK_RULE_EN_GB", "EN_COMPOUNDS",
                "THERE_THEIR", "THEIR_IS", "LESS_FEWER", "ADVISE_ADVICE", "AFFECT_EFFECT", "LOOSE_LOSE",
                "THAN_THEN", "WHOS_WHOSE", "YOUR_YOU_RE", "TO_TOO", "A_INFINITIVE", "PERS_PRONOUN_AGREEMENT_SENT_START"}
_GRAMMAR_CATS = {"GRAMMAR", "TYPOS", "CONFUSED_WORDS", "COLLOCATIONS"}
_GRAMMAR_SKIP = {"UPPERCASE_SENTENCE_START", "WHITESPACE_RULE", "COMMA_PARENTHESIS_WHITESPACE", "EN_QUOTES",
                 "DASH_RULE", "DOUBLE_PUNCTUATION", "EN_UNPAIRED_BRACKETS", "PUNCTUATION_PARAGRAPH_END",
                 "SENTENCE_WHITESPACE", "ENGLISH_WORD_REPEAT_BEGINNING_RULE", "PASSIVE_VOICE"}


def _edit_distance(a, b):
    return sum(1 for op in difflib.SequenceMatcher(None, a, b).get_opcodes() if op[0] != "equal") + abs(len(a) - len(b)) // 2


def fam_grammar(s, ctx):
    out = []
    for m in ctx.lt or []:
        if not (s.start <= m["start"] and m["end"] <= s.end) or len(m["replacements"]) != 1:
            continue
        rule, cat = m["rule"], str(m["category"]).upper()
        if rule in _GRAMMAR_SKIP or rule not in GRAMMAR_AUTO:
            continue
        if "MORFOLOGIK" in rule and ctx.fmt.startswith("pdf"):
            continue                                   # PDF text layers contain extraction artefacts
        old = s.block.masked[m["start"]:m["end"]]
        new = m["replacements"][0]
        if "MORFOLOGIK" in rule:                       # spelling: only clear, one-off non-words
            w = old.lower()
            if (not old.isalpha() or not old.islower() or zipf(old) > 0 or w in ctx.profile.glossary_words
                    or " " in new or zipf(new) < 3 or _edit_distance(old, new) > 2
                    or new[:1].lower() != old[:1].lower()      # real typos keep the first letter: 'squeue' is not 'Queue'
                    or w.startswith(new.lower()) or new.lower().startswith(w)      # 'susceptibles' is a term, not a typo
                    or sum(1 for x in ctx.profile.vocab_counts.get(w, ()) ) > 1):
                continue
            cat = "TYPOS"
        nxt = [t for t in s.toks if t.start >= m["end"]][:1]
        if rule == "EN_A_VS_AN":                       # double-check a/an against our own rule
            if not nxt or PH.fullmatch(nxt[0].text) or not nxt[0].text[:1].isalpha() or \
                    _article_for(nxt[0].text) != new.lower():
                continue
        if rule == "ENGLISH_WORD_REPEAT_RULE" and not (old.split()[0].lower() in
                {"the", "a", "an", "of", "to", "in", "and", "is", "that", "for", "with", "are", "be"} or len(new) >= 4):
            continue                                   # "on on off off" is table debris, not a typo
        ol, nl = _lems(old), _lems(new)
        content_old = {t.lem for t in tag_tokens(tokenize(old)) if t.tag[:2] in ("NN", "VB", "JJ", "RB") and t.lem not in AUX_LEMMAS}
        content_new = {t.lem for t in tag_tokens(tokenize(new)) if t.tag[:2] in ("NN", "VB", "JJ", "RB") and t.lem not in AUX_LEMMAS}
        if content_old != content_new and "MORFOLOGIK" not in rule:
            continue                                   # grammar fixes may not swap content words
        out.append(Edit(m["start"], m["end"], new, "grammar", f"lt:{rule}", m["message"],
                        f"LanguageTool {rule}", frozenset(ol), frozenset(nl)))
    return out


_A_AN_EXC = ("uni", "use", "usu", "uti", "ure", "uro", "ubiq", "eu", "ewe", "one", "once", "unanim")


def _article_for(word):
    """a/an by sound, approximated from spelling with the usual exceptions (a unimodal, an hour)."""
    w = word.lower()
    if w.startswith(("hour", "honest", "honor", "honour", "heir")):
        return "an"
    if w[:1] in "aeiou" and not w.startswith(_A_AN_EXC):
        return "an"
    return "a"


_TAG_ORDER = {"VERB": ("VBP", "VB", "VBZ", "VBD", "VBN", "VBG"), "NOUN": ("NN", "NNS"), "ADJ": ("JJ", "JJR", "JJS"),
              "ADV": ("RB", "RBR", "RBS")}


def surface_tag(lem, word, tag):
    """The Penn tag that the SPELLING of `word` supports as a form of `lem`, preferring the tagger's tag.
    The tagger reads 'existing tools showcase a ...' as past tense (VBD); 'showcase' cannot be past, so a
    replacement conjugated to the tagger's tag would change the tense ('showed'). None = no form fits."""
    up = _UPOS.get(tag[:1]) if tag else None
    if not up:
        return tag
    forms = getAllInflections(lem, upos=up)
    w = word.lower()
    fits = lambda tg: w in {f.lower() for f in forms.get(tg, ())} or (tg in ("VB", "VBP", "NN") and w == lem.lower())
    if fits(tag):
        return tag
    return next((tg for tg in _TAG_ORDER[up] if fits(tg)), None)


def _strict_synonym(a_lem, b_lem, tag):
    """b must be in a's dominant (first) WordNet sense AND a in one of b's two dominant senses."""
    return wn_synonymous(a_lem, b_lem, tag, top=1) and wn_synonymous(b_lem, a_lem, tag, top=2)


def _synonym_for(tok, ctx, min_gain, allow_any_zipf=False):
    if tok.lem in ctx.subs:
        cands = [(ctx.subs[tok.lem], 9.9)]
    else:
        cands = list(ppdb_candidates(tok.text)) + list(ppdb_candidates(tok.lem))
    for c, sc in cands:
        cl = lemma(c, tok.tag)
        if ctx.mode not in ("general", "conversational") and cl in INFORMAL:
            continue
        if cl == tok.lem or cl in CLAIM_WORDS or cl in STAT_TERMS or cl in ctx.profile.glossary_words:
            continue
        if not _strict_synonym(tok.lem, cl, tok.tag):
            continue
        stag = surface_tag(tok.lem, tok.text, tok.tag)
        form = inflect(cl, stag) if stag else None
        if not form:
            continue
        form = author_spelling(form, ctx.profile)
        if allow_any_zipf:
            if zipf(form) < 3.0:
                continue
        elif zipf(form) < zipf(tok.text) + min_gain or len(form) > len(tok.text) + 2:
            continue
        return cl, form
    return None


def _substitute(s, i, ctx, family, reason, min_gain, allow_any_zipf=False, fixed=None):
    T = s.toks
    t = T[i]
    got = fixed or _synonym_for(t, ctx, min_gain, allow_any_zipf)
    if not got:
        return None
    cl, form = got
    if t.text[:1].isupper():
        form = _cap(form)
    start, new = t.start, form
    if i > 0 and T[i - 1].text.lower() in ("a", "an"):
        art = _article_for(form)
        if art != T[i - 1].text.lower():
            start = T[i - 1].start
            new = (_cap(art) if T[i - 1].text[0].isupper() else art) + " " + form
    ctx.subs.setdefault(t.lem, cl)
    return Edit(start, t.end, new, family, f"{family}:{t.lem}->{cl}", reason,
                f"'{t.text}' (zipf {zipf(t.text):.1f}) -> '{form}' (zipf {zipf(form):.1f})",
                frozenset({t.lem, canon(t.text), t.text.lower(), "a", "an"}), frozenset({cl, canon(form), form.lower(), "a", "an"}))


def _substitutable(t, ctx):
    # nouns are excluded from open-ended swaps: in research prose they are usually terms of art
    return (t.text.isalpha() and t.tag in ("VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "JJ", "RB")
            and t.lem not in AUX_LEMMAS and t.lem not in CLAIM_WORDS and t.text.lower() not in CLAIM_WORDS
            and t.lem not in STAT_TERMS and t.lem not in ctx.profile.glossary_words
            and t.text.lower() not in ctx.profile.glossary_words)


def fam_plain(s, ctx):
    """Curated plain-language swaps (utilize -> use); inflection re-generated from the dictionary."""
    out = []
    skip = _planned_drop(s, ctx)                     # don't turn a planned "Additionally," drop into "Also,"
    for i, t in enumerate(s.toks):
        if i < skip:
            continue
        key = (t.lem, t.tag[:1]) if t.tag[:1] in "VJRN" else None
        if not key or key not in PLAIN or t.lem in ctx.profile.glossary_words:
            continue
        if key in PLAIN_MODES and ctx.mode not in PLAIN_MODES[key]:
            continue                                   # term of art in this mode ('terminate' a job)
        if i > 0 and t.text[:1].isupper() and t.tag.startswith("NNP"):
            continue
        tgt = PLAIN[key]
        nx = s.toks[i + 1] if i + 1 < len(s.toks) else None
        stag = surface_tag(t.lem, t.text, t.tag) if t.tag[:1] in "VN" else t.tag
        if not stag:
            continue
        if nx is not None and (tgt, nx.text.lower()) in PLAIN_PREP and t.tag.startswith("VB"):
            form = inflect(tgt, stag)                  # "assist in editing" -> "help with editing"
            if form:
                form = _cap(form) if t.text[:1].isupper() else form
                prep = PLAIN_PREP[(tgt, nx.text.lower())]
                out.append(Edit(t.start, nx.end, f"{form} {prep}", "plain", f"plain:{t.lem}+{nx.text.lower()}->{tgt}+{prep}",
                                f"'{t.lem} {nx.text.lower()}' is an inflated form of '{tgt} {prep}'",
                                f"'{t.text} {nx.text}' -> '{form} {prep}'",
                                frozenset({t.lem, canon(t.text), t.text.lower(), nx.text.lower()}),
                                frozenset({tgt, canon(form), form.lower(), prep})))
            continue
        if t.lem == "approximately":                   # "about" cannot modify an adjective
            nx = s.toks[i + 1] if i + 1 < len(s.toks) else None
            if not nx or not (PH.fullmatch(nx.text) or nx.tag == "CD" or nx.text.lower() in ("the", "half", "a", "one")):
                continue
        form = inflect(tgt, stag) if t.tag[:1] in "VN" else tgt
        if not form:
            continue
        e = _substitute(s, i, ctx, "plain", f"'{t.lem}' is an inflated form of '{tgt}'", 0, fixed=(tgt, form))
        if e:
            out.append(e)
    return out


def fam_lexical(s, ctx):
    """Open-ended swaps: PPDB proposes, WordNet must confirm dominant-sense synonymy both ways."""
    out = []
    for i, t in enumerate(s.toks):
        if i > 0 and t.text[:1].isupper():
            continue
        if (t.lem, t.tag[:1]) in PLAIN or not _substitutable(t, ctx) or len(t.text) < 6 or zipf(t.text) >= 4.0:
            continue
        e = _substitute(s, i, ctx, "lexical", "a more familiar word sharing the dominant WordNet sense", 1.0)
        if e:
            out.append(e)
    return out


def fam_repetition(s, ctx):
    out = []
    for i, t in enumerate(s.toks):
        if not _substitutable(t, ctx) or not t.tag[:2] in ("JJ", "RB", "VB"):
            continue
        ctx.rep_seen[t.lem] += 1
        if ctx.rep_seen[t.lem] >= 3:
            e = _substitute(s, i, ctx, "repetition", f"'{t.lem}' already appears twice in this paragraph", 0, True)
            if e:
                out.append(e)
    return out


_SUBJ_TO_OBJ = {"we": "us", "i": "me", "he": "him", "she": "her", "they": "them"}
_OBJ_TO_SUBJ = {v: k for k, v in _SUBJ_TO_OBJ.items()}


def fam_passive(s, ctx):
    if not s.parse:
        return []
    tree, deps = s.parse
    T, txt = s.toks, s.block.masked
    out = []
    kids = defaultdict(list)
    for rel, h, d in deps:
        kids[h].append((rel, d))
    for rel, V, subj in deps:
        if rel != "nsubjpass" or T[V - 1].tag != "VBN":
            continue
        rels = kids[V]
        if any(r in ("neg", "nsubj", "dobj", "ccomp", "xcomp", "csubjpass") for r, _ in rels):
            continue
        auxp = [d for r, d in rels if r == "auxpass"]
        aux = [d for r, d in rels if r == "aux"]
        if len(auxp) != 1 or len(aux) > 1:
            continue
        be = T[auxp[0] - 1].text.lower()
        if aux:
            if T[aux[0] - 1].tag != "MD" or be != "be":
                continue
        elif be not in ("is", "are", "was", "were"):
            continue
        agents = [d for r, d in rels if r.startswith("nmod") and any(r2 == "case" and T[d2 - 1].text.lower() == "by" for r2, d2 in kids[d])]
        if len(agents) != 1:
            continue
        A = sorted(subtree_span(deps, agents[0]))
        S = sorted(subtree_span(deps, subj))
        if A != list(range(A[0], A[-1] + 1)) or S != list(range(S[0], S[-1] + 1)):
            continue
        if T[A[0] - 1].text.lower() != "by" or not (1 <= len(A) - 1 <= 6) or len(S) > 10:
            continue
        # agent and patient must be simple noun phrases: no prepositional or clausal modifiers,
        # because a misattached "in 2020" would otherwise move with the agent and change meaning
        bad = {"case", "acl", "acl:relcl", "advcl", "nmod", "ccomp", "parataxis"}
        if any(r in bad and d != A[0] for x in A[1:] + S for r, d in kids[x]):
            continue
        if any(r in ("conj", "cc") for x in S for r, _ in kids[x]):
            continue                                   # coordinated subjects are often misparsed
        if any(PH.fullmatch(T[x - 1].text) for x in A[1:] + S):
            continue                                   # never move citations or formulas
        if S[0] > 1 and T[S[0] - 2].tag.startswith("VB"):
            continue                                   # a 'subject' right after a verb is a misparse
        chain = ([aux[0]] if aux else []) + [auxp[0], V]
        if S[-1] + 1 != chain[0] or any(b != a + 1 for a, b in zip(chain, chain[1:])) or V + 1 != A[0]:
            continue
        agent_head = T[agents[0] - 1]
        a_text = txt[T[A[1] - 1].start:T[A[-1] - 1].end]
        s_text = txt[T[S[0] - 1].start:T[S[-1] - 1].end]
        if len(A) == 2 and a_text.lower() in _OBJ_TO_SUBJ:
            a_text = _OBJ_TO_SUBJ[a_text.lower()]
            if a_text == "i":
                a_text = "I"
        if len(S) == 1 and s_text.lower() in _SUBJ_TO_OBJ:
            s_text = _SUBJ_TO_OBJ[s_text.lower()]
        plural = agent_head.tag in ("NNS", "NNPS") or any(T[x - 1].text.lower() == "and" for x in A) or a_text.lower() in ("we", "they", "you", "i")
        vlem = lemma(T[V - 1].text, "VBN")
        tag = "VB" if aux else ("VBD" if be in ("was", "were") else ("VBP" if plural else "VBZ"))
        form = inflect(vlem, tag)
        if not form:
            continue
        form = author_spelling(form, ctx.profile)
        prefix = txt[s.start:T[S[0] - 1].start]
        suffix = txt[T[A[-1] - 1].end:s.end]
        if A[-1] < len(T) and T[A[-1]].text.lower() == "of":
            continue                                   # an 'of' phrase belongs to the agent noun: moving it changes meaning
        if not prefix.strip():
            s_text = _decap(s_text, T[S[0] - 1])
            a_text = _cap(a_text)
        modal = (T[aux[0] - 1].text + " ") if aux else ""
        new = f"{prefix}{a_text} {modal}{form} {s_text}{suffix}"
        a_head_txt, s_head_txt = agent_head.text, T[subj - 1].text
        if len(A) == 2 and agent_head.text.lower() in _OBJ_TO_SUBJ:
            a_head_txt = _OBJ_TO_SUBJ[agent_head.text.lower()]
        if len(S) == 1 and s_head_txt.lower() in _SUBJ_TO_OBJ:
            s_head_txt = _SUBJ_TO_OBJ[s_head_txt.lower()]

        def verify(parsed, form=form, a_head=a_head_txt.lower(), s_head=s_head_txt.lower()):
            if not parsed.parse:
                return False, "re-parse failed"
            P, dps = parsed.toks, parsed.parse[1]
            ok_subj = any(r == "nsubj" and P[h - 1].text == form and P[d - 1].text.lower() == a_head for r, h, d in dps)
            ok_obj = any(r == "dobj" and P[h - 1].text == form and P[d - 1].text.lower() == s_head for r, h, d in dps)
            return (ok_subj and ok_obj), ("" if ok_subj and ok_obj else "re-parse does not show agent as subject and patient as object")

        out.append(Edit(s.start, s.end, new, "passive", "passive:by-agent-to-active",
                        "a passive with an explicit 'by' agent reads more directly in the active voice",
                        f"nsubjpass + auxpass '{be}' + by-agent '{a_text}'",
                        frozenset({"be", be, "by", "is", "are", "was", "were"} | {s_text.lower(), a_text.lower()}),
                        frozenset({form.lower(), vlem} | {s_text.lower(), a_text.lower()}),
                        structural=True, verify=verify))
    return out


# Findings from the labelled dev/test sets (section 15):
#  * semicolon splits were judged wrong 5/5: academic authors use ';' deliberately, so ';' splits are
#    limited to general/conversational writing;
#  * 'and' joins two closely related claims: split only genuinely overlong sentences (the same
#    45-word line at which the engine already advises a manual split);
#  * the new second sentence must open with its own subject NP, otherwise "but on X rather than Y"
#    becomes the fragment "However, on X rather than Y".
SPLIT_AND_MIN = 45
SPLIT_SEMI_MODES = {"general", "conversational"}


def fam_split(s, ctx, min_len):
    if not s.parse or min_len is None:
        return []
    tree = s.parse[0]
    if len(tree) != 1 or tree[0].label() != "S":
        return []
    root = tree[0]
    labels = [k.label() if isinstance(k, Tree) else k for k in root]
    T, txt = s.toks, s.block.masked
    n_words = sum(1 for t in T if t.text[:1].isalnum() or PH.fullmatch(t.text))
    if ctx.hplan is not None and s.block.bid in ctx.hplan.low_variation:
        min_len = min(min_len, ctx.hplan.split_min_low_var)     # uniform paragraph: allow 'but' splits sooner
    if n_words < min_len:
        return []
    spans, pos = [], 0
    for k in root:
        n = len(k.leaves()) if isinstance(k, Tree) else 1
        spans.append((pos, pos + n)); pos += n
    pat = None
    if labels[:5] == ["S", ",", "CC", "S", "."] and len(labels) == 5:
        cc = T[spans[2][0]].text.lower()
        if cc == "but" or (cc == "and" and n_words >= SPLIT_AND_MIN):
            pat = ("cc", cc)
    elif (labels[:4] == ["S", ":", "S", "."] and len(labels) == 4 and T[spans[1][0]].text == ";"
          and ctx.mode in SPLIT_SEMI_MODES):
        pat = ("semi", ";")
    if not pat:
        return []
    left, right = root[0], root[3] if pat[0] == "cc" else root[2]
    for clause in (left, right):
        cl = [c.label() for c in clause if isinstance(c, Tree)]
        if "NP" not in cl or "VP" not in cl:
            return []
    if not isinstance(right[0], Tree) or right[0].label() != "NP":
        return []                                    # second sentence would not start with its subject
    lspan = spans[0]
    rspan = spans[3] if pat[0] == "cc" else spans[2]
    lw = sum(1 for t in T[lspan[0]:lspan[1]] if t.text[:1].isalnum())
    rw = sum(1 for t in T[rspan[0]:rspan[1]] if t.text[:1].isalnum())
    if lw < 8 or rw < 8:
        return []
    ltxt = txt[T[lspan[0]].start:T[lspan[1] - 1].end]
    rfirst = T[rspan[0]]
    rtxt = txt[rfirst.start:s.end]
    if pat == ("cc", "but"):
        new = f"{ltxt}. However, {_decap(rtxt, rfirst)}"
        rem, add = {"but"}, {"however"}
    else:
        new = f"{ltxt}. {_cap(rtxt)}"
        rem, add = ({"and"} if pat[1] == "and" else set()), set()
    return [Edit(s.start, s.end, new, "split", f"split:{pat[1]}",
                 f"a {n_words}-word sentence joining two full clauses reads better as two sentences",
                 f"parse: S {pat[1]} S, clauses of {lw} and {rw} words", frozenset(rem), frozenset(add), structural=True)]


def advice_for(sents, profile):
    """Signals that have no safe automatic fix: reported, never applied."""
    adv = []
    for s in sents:
        n = sum(1 for t in s.toks if t.text[:1].isalnum())
        if n > 45:
            adv.append((s, "long-sentence", f"{n} words; consider splitting by hand"))
    for bid, i, w in profile.opener_runs:
        if sents and sents[0].block.bid == bid and i < len(sents):
            adv.append((sents[i], "opener-run", f"three consecutive sentences open with '{w}'"))
    for a, b in zip(sents, sents[1:]):
        na = sum(1 for t in a.toks if t.text[:1].isalnum()); nb = sum(1 for t in b.toks if t.text[:1].isalnum())
        if na < 8 and nb < 8 and a.toks[0].lem == b.toks[0].lem:
            adv.append((a, "choppy", "two short sentences with the same subject could be merged"))
    return adv

## 11 · Humanizer families: reconstruction, not substitution
These families change *how* an idea is expressed (its phrasing, clause order, sentence boundaries and connectives) without changing *what* is said. Every edit declares its footprint and passes the same meaning gate (section 12); the syntactic ones are also re-parsed to verify.

| Family | Example | Unlocked by | Default |
|---|---|---|---|
| cliche | *leverages* → *uses*; *plays a crucial role in* → *is crucial for*; *delves into* → *examines* | always | all intensities |
| hedge_stack | *may potentially* → *may* (one hedge kept: the claim stays hedged) | always | all |
| register | academic/technical: *don't* → *do not*; conversational: *do not* → *don't* | the mode | all |
| discourse | *Moreover, the model…* → *The model…*; *In conclusion,* inside a Conclusion section | additive-opener density | balanced+ |
| opener | *The model was evaluated after X converged.* → *After X converged, the model was evaluated.* | opener repetition | balanced+ |
| merge | *The model is small. It trains fast.* → *The model is small, and it trains fast.* | uniform sentence lengths | balanced+ (not technical) |
| dash | *X — a Y — is* → *X (a Y) is* | dash density | balanced+ |

**Never:** moving material past a negation or a focus word (*only, even, also*); fronting a phrase out of an embedded or second clause; moving a citation, formula or command; merging a sentence that contains a negation (it usually corrects rather than adds); contrast, cause and condition connectives are never dropped.

In [11]:
#@title 11 · Humanizer families: deterministic reconstruction of vocabulary, syntax and discourse
# Same contract as section 10: each family proposes an Edit with a precondition, an action and a
# declared footprint; the meaning gate (section 12) rejects anything outside it. The style plan
# (section 9) decides WHERE the discourse and syntax families may act.
#
#   cliche      stock phrasing -> plain equivalent       "delve into X" -> "examine X"
#   hedge_stack doubled hedges -> one hedge              "may potentially" -> "may"
#   register    contractions to suit the mode            academic: "don't" -> "do not"; conversational: reverse
#   discourse   thin additive openers (plan-driven)      "Moreover, the model ..." -> "The model ..."
#   opener      vary repeated openings (parse-verified)  "We stopped when X." -> "When X, we stopped."
#   merge       join two short uniform sentences         "A. It B." -> "A, and it B."   (cross-sentence)
#   dash        parenthetical dash pairs -> parentheses  "X — a Y — is" -> "X (a Y) is"
#   split       (section 10) also unlocked earlier in uniform paragraphs for 'but' clauses

PRIORS.update({"cliche": 0.88, "hedge_stack": 0.93, "register": 0.92, "discourse": 0.86,
               "opener": 0.76, "merge": 0.75, "dash": 0.80})


def _norm(w):
    return w.lower().replace("’", "'")


def _norm_hits(s, phrase):
    p = phrase.split()
    low = [_norm(t.text) for t in s.toks]
    for i in range(len(low) - len(p) + 1):
        if low[i:i + len(p)] == p:
            yield i, i + len(p) - 1


def _is_term(t, ctx):
    return t.lem in ctx.profile.glossary_words or t.text.lower() in ctx.profile.glossary_words


# ---- cliche: stock vocabulary and phrasing with a meaning-preserving plain equivalent ----------
STOCK_PHRASE_SWAPS = [
    ("a myriad of", "many"), ("a plethora of", "many"), ("in the realm of", "in"), ("within the realm of", "within"),
    ("of paramount importance", "essential"), ("of utmost importance", "essential"), ("of the utmost importance", "essential"),
    ("it is essential to note that", ""), ("it is crucial to note that", ""), ("it is worth mentioning that", ""),
    ("in today's fast-paced world ,", ""), ("in today's world ,", ""), ("in today's digital age ,", ""),
    ("in the ever-evolving landscape of", "in"), ("in the rapidly evolving landscape of", "in"),
    ("in the ever-changing landscape of", "in"),
]
# verb lemma + fixed continuation -> verb lemma (inflected to the original verb's tag)
STOCK_VERB_PHRASES = {("delve", "into"): "examine", ("embark", "on"): "begin", ("embark", "upon"): "begin",
                      ("harness", "the", "power", "of"): "use", ("shed", "light", "on"): "clarify",
                      ("pave", "the", "way", "for"): "enable"}
STOCK_VERBS = {"leverage": "use", "showcase": "show", "underscore": "stress", "foster": "encourage",
               "bolster": "strengthen", "streamline": "simplify", "spearhead": "lead",
               "revolutionize": "transform", "revolutionise": "transform", "harness": "use"}
STOCK_ADJ = {"pivotal": "central", "seamless": "smooth", "intricate": "complex", "multifaceted": "complex",
             "meticulous": "careful", "noteworthy": "notable", "cutting-edge": "advanced",
             "ever-evolving": "changing", "ever-changing": "changing", "myriad": "many"}
STOCK_ADV = {"seamlessly": "smoothly", "meticulously": "carefully"}
# "plays a crucial role in X" -> "is crucial for X" (agreement from the verb's tag; past tense is skipped
# because "was"/"were" needs the subject's number, which a tag does not give)
ROLE_ADJ = {"crucial": "crucial for", "key": "key to", "vital": "vital for", "critical": "critical for",
            "central": "central to", "pivotal": "central to", "important": "important for", "essential": "essential for"}
_BE_FOR_TAG = {"VBZ": "is", "VBP": "are", "VB": "be"}
_STOCK_VERB_LEMMAS = set(STOCK_VERBS) | {k[0] for k in STOCK_VERB_PHRASES} | {"play", "boast"}


def _verb_tag(T, i):
    """Contextual re-tag, Brill style, for stock verbs only: the perceptron tags 'The framework
    leverages pretrained features' as NN + NNS. A word with a verb reading between a singular
    subject and an object-like word is the verb (VBZ)."""
    t = T[i]
    if t.tag.startswith("VB"):
        return t.tag, t.lem
    vlem = lemma(t.text, "VBZ")                      # the noun reading may not even be in the dictionary
    if (t.tag == "NNS" and vlem in _STOCK_VERB_LEMMAS and "VBZ" in getAllInflections(vlem, upos="VERB")
            and i > 0 and T[i - 1].tag in ("NN", "NNP", "PRP") and i + 1 < len(T)
            and T[i + 1].tag[:2] in ("DT", "JJ", "VB", "NN", "PR", "CD", "IN")):
        return "VBZ", vlem
    return t.tag, t.lem


def fam_cliche(s, ctx):
    out, T = [], s.toks
    for pat, repl in STOCK_PHRASE_SWAPS:
        for i, j in _norm_hits(s, pat):
            e = _replace_phrase(s, i, j, repl, "cliche", f"cliche:{pat}",
                                f"'{pat}' is stock phrasing" + (f"; '{repl}' says the same" if repl else " that adds no content"))
            if e:
                out.append(e)
    for i, t in enumerate(T):
        if _is_term(t, ctx):
            continue
        vtag, vlem = _verb_tag(T, i)
        if vtag.startswith("VB"):
            vtag = surface_tag(vlem, t.text, vtag) or "?"            # conjugate to what the spelling allows
        if vtag.startswith("VB") and vlem == "play" and vtag in _BE_FOR_TAG and i + 4 < len(T):
            w = [_norm(x.text) for x in T[i + 1:i + 5]]
            if w[0] in ("a", "an") and w[1] in ROLE_ADJ and w[2] == "role" and w[3] == "in":
                be = _BE_FOR_TAG[vtag]
                e = _replace_phrase(s, i, i + 4, f"{be} {ROLE_ADJ[w[1]]}", "cliche", f"cliche:play-a-{w[1]}-role",
                                    f"'plays a {w[1]} role in' is stock phrasing for '{be} {ROLE_ADJ[w[1]]}'")
                if e:
                    out.append(e)
                continue
        if vtag.startswith("VB"):
            for (vl, *rest), tgt in STOCK_VERB_PHRASES.items():
                if vlem != vl or [_norm(x.text) for x in T[i + 1:i + 1 + len(rest)]] != rest:
                    continue
                form = inflect(tgt, vtag)
                if form:
                    e = _replace_phrase(s, i, i + len(rest), author_spelling(form, ctx.profile), "cliche",
                                        f"cliche:{vl} {' '.join(rest)}", f"'{vl} {' '.join(rest)}' is stock phrasing for '{tgt}'")
                    if e:
                        out.append(e)
                break
            else:
                if vlem in STOCK_VERBS and not (vlem == "harness" and i + 1 < len(T) and _norm(T[i + 1].text) == "the"):
                    tgt = STOCK_VERBS[vlem]
                    form = inflect(tgt, vtag)
                    if form:
                        e = _substitute(s, i, ctx, "cliche", f"'{vlem}' is stock vocabulary for '{tgt}'", 0,
                                        fixed=(tgt, author_spelling(form, ctx.profile)))
                        if e:
                            out.append(e)
                elif vlem == "boast" and i + 1 < len(T) and T[i + 1].tag[:2] in ("DT", "CD", "JJ", "NN", "PR"):
                    form = inflect("have", vtag)
                    if form:
                        e = _substitute(s, i, ctx, "cliche", "'boast' is stock vocabulary for 'have'", 0, fixed=("have", form))
                        if e:
                            out.append(e)
        elif t.tag.startswith("JJ") and _norm(t.text) in STOCK_ADJ:
            tgt = STOCK_ADJ[_norm(t.text)]
            if _norm(t.text) == "myriad" and not (i + 1 < len(T) and T[i + 1].tag == "NNS"):
                continue
            e = _substitute(s, i, ctx, "cliche", f"'{t.text.lower()}' is stock vocabulary for '{tgt}'", 0, fixed=(tgt, tgt))
            if e:
                out.append(e)
        elif t.tag.startswith("RB") and _norm(t.text) in STOCK_ADV:
            tgt = STOCK_ADV[_norm(t.text)]
            e = _substitute(s, i, ctx, "cliche", f"'{t.text.lower()}' is stock vocabulary for '{tgt}'", 0, fixed=(tgt, tgt))
            if e:
                out.append(e)
    return out


# ---- hedge_stack: two hedges doing one job ---------------------------------------------------
def fam_hedge_stack(s, ctx):
    out, T = [], s.toks
    for i in range(len(T) - 1):
        a, b = _norm(T[i].text), _norm(T[i + 1].text)
        if a in _HEDGE_PAIR and b in _HEDGE_ADV:
            out.append(Edit(T[i].end, T[i + 1].end, "", "hedge_stack", f"hedge_stack:{a}+{b}",
                            f"'{a} {b}' hedges twice; '{a}' alone keeps the same uncertainty",
                            f"modal '{a}' + hedging adverb '{b}'", frozenset({b}), frozenset()))
    return out


# ---- register: contractions follow the writing mode ------------------------------------------
EXPANSIONS = {"don't": "do not", "doesn't": "does not", "didn't": "did not", "isn't": "is not", "aren't": "are not",
              "wasn't": "was not", "weren't": "were not", "can't": "cannot", "couldn't": "could not",
              "wouldn't": "would not", "shouldn't": "should not", "won't": "will not", "haven't": "have not",
              "hasn't": "has not", "hadn't": "had not", "mustn't": "must not", "needn't": "need not",
              "let's": "let us", "we're": "we are", "they're": "they are", "you're": "you are", "i'm": "I am",
              "we've": "we have", "they've": "they have", "you've": "you have", "i've": "I have",
              "we'll": "we will", "they'll": "they will", "you'll": "you will", "i'll": "I will", "it'll": "it will"}
_IS_OR_HAS = {"it's": "it", "that's": "that", "there's": "there", "what's": "what", "here's": "here"}
CONTRACTIONS = {tuple(v.lower().split()): k for k, v in EXPANSIONS.items() if k not in ("can't",)}
CONTRACTIONS.update({("cannot",): "can't", ("it", "is"): "it's", ("that", "is"): "that's", ("there", "is"): "there's"})
REGISTER_EXPAND_MODES, REGISTER_CONTRACT_MODES = {"academic", "technical"}, {"conversational"}


def fam_register(s, ctx):
    out, T = [], s.toks
    curly = "’" in s.text
    if ctx.mode in REGISTER_EXPAND_MODES:
        for i, t in enumerate(T):
            w = _norm(t.text)
            if w in _IS_OR_HAS:
                nxt = T[i + 1] if i + 1 < len(T) else None
                new = f"{_IS_OR_HAS[w]} {'has' if nxt is not None and nxt.tag == 'VBN' else 'is'}"
            elif w in EXPANSIONS:
                new = EXPANSIONS[w]
            else:
                continue
            new = _cap(new) if t.text[:1].isupper() else new
            neg = {"not"} if w.endswith("n't") else set()
            out.append(Edit(t.start, t.end, new, "register", f"register:expand:{w}",
                            f"{ctx.mode} writing spells out contractions", f"contraction '{t.text}'",
                            frozenset({w, t.text.lower(), canon(t.text)} | neg), frozenset(_lems(new) | neg)))
    elif ctx.mode in REGISTER_CONTRACT_MODES:
        i = 0
        while i < len(T):
            for n in (2, 1):
                key = tuple(_norm(x.text) for x in T[i:i + n])
                if len(key) == n and key in CONTRACTIONS:
                    nxt = T[i + n] if i + n < len(T) else None
                    if key[-1] == "is" and (nxt is None or nxt.text in {",", ".", ";", ":", "?", "!"}):
                        continue                         # "that is," means "i.e."; sentence-final "it is." stays
                    new = CONTRACTIONS[key].replace("'", "’") if curly else CONTRACTIONS[key]
                    new = _cap(new) if T[i].text[:1].isupper() else new
                    old = s.block.masked[T[i].start:T[i + n - 1].end]
                    neg = {"not"} if new.lower().endswith(("n't", "n’t")) else set()
                    out.append(Edit(T[i].start, T[i + n - 1].end, new, "register", f"register:contract:{' '.join(key)}",
                                    "conversational writing contracts common verb pairs", f"'{old}'",
                                    frozenset(_lems(old) | neg), frozenset({new.lower(), _norm(new), canon(new)} | neg)))
                    i += n - 1
                    break
            i += 1
    return out


# ---- discourse: thin additive openers where the plan found them over-dense -------------------
def fam_discourse(s, ctx):
    hp = ctx.hplan
    if hp is None:
        return []
    key = (s.block.bid, s.idx)
    if key in hp.thin:
        n, why = _starts_with(s, ADDITIVE_OPENERS), "additive openers are over-dense here; adjacent sentences already read as 'and'"
    elif key in hp.summary:
        n, why = _starts_with(s, SUMMARY_OPENERS), "the section heading already says this is the conclusion"
    else:
        return []
    T = s.toks
    if not n or len(T) <= n + 2:
        return []
    nxt = T[n + 1]
    removed = {t.text.lower() for t in T[:n]} | {t.lem for t in T[:n]}
    return [Edit(T[0].start, nxt.end, _cap(nxt.text), "discourse", f"discourse:drop:{' '.join(t.text.lower() for t in T[:n])}",
                 why, f"plan: sentence {s.idx} of block {s.block.bid}", frozenset(removed), frozenset({nxt.lem, nxt.text.lower()}))]


# ---- opener: front a final adverbial clause/phrase to vary a repeated opening ----------------
FRONT_SUB = {"when", "after", "before", "because", "although", "though", "while", "if", "once", "since",
             "unless", "until", "whereas"}
FRONT_PREP = {"during", "after", "before", "throughout", "despite", "following", "since", "until"}
_SCOPE = {"only", "even", "just", "also", "too", "either"}      # focus words whose scope depends on position


def _constituents(tree):
    """(label, ancestor_labels, first_leaf, end_leaf) for every phrase; leaves == tokens.
    ancestor_labels runs from the root down to the phrase's parent."""
    out = []

    def walk(t, pos, anc):
        if not isinstance(t, Tree):
            return pos + 1
        start = pos
        for k in t:
            pos = walk(k, pos, anc + (t.label(),))
        out.append((t.label(), anc, start, pos))
        return pos
    walk(tree, 0, ())
    return out


_DISCOURSE_LINKS = set().union(*CONNECTIVE_CLASS.values()) | {w for p in ADDITIVE_OPENERS for w in p.split()} | {
    "therefore", "thus", "hence", "accordingly", "instead", "meanwhile", "nonetheless", "similarly", "likewise"}


def _root_and_subject(deps, toks):
    root = next((d for r, h, d in deps if r == "root"), None)
    subj = next((d for r, h, d in deps if h == root and r in ("nsubj", "nsubjpass")), None) if root else None
    return (toks[root - 1].text if root else None), (toks[subj - 1].text.lower() if subj else None)


def fam_opener(s, ctx):
    hp = ctx.hplan
    if hp is None or (s.block.bid, s.idx) not in hp.opener or not s.parse:
        return []
    tree, deps = s.parse
    # one simple clause only: (ROOT (S NP VP .)). "X; confirm Y before Z" has two clauses, and a
    # phrase fronted out of the second one would silently re-attach to the first.
    if (len(tree) != 1 or tree[0].label() != "S"
            or [k.label() if isinstance(k, Tree) else k for k in tree[0]] != ["NP", "VP", "."]):
        return []
    T, txt, n = s.toks, s.block.masked, len(s.toks)
    low = [_norm(t.text) for t in T]
    if T[-1].text != "." or any(w in NEGATIONS or w.endswith("n't") or w in _SCOPE for w in low):
        return []                                    # moving material past a negation or focus word shifts scope
    root_v, subj = _root_and_subject(deps, T)
    if not root_v or not subj:
        return []
    for lab, anc, a, b in sorted(_constituents(tree), key=lambda c: c[2]):
        if b != n - 1 or lab not in ("SBAR", "PP"):
            continue
        # path ROOT > S > VP (> VP ...) only: the phrase modifies the MAIN verb, never a verb
        # inside an embedded clause ("we know that X happened after Y")
        if anc[:2] != ("ROOT", "S") or len(anc) < 3 or any(x != "VP" for x in anc[2:]):
            continue
        head = low[a]
        if (lab == "SBAR" and head not in FRONT_SUB) or (lab == "PP" and head not in FRONT_PREP):
            continue
        main_end = a - 1 if T[a - 1].text == "," else a
        if any(w in _DISCOURSE_LINKS for w in low[:main_end]):
            continue                                 # "therefore", "however"... tie the main clause to its neighbour
        adv_w =sum(1 for t in T[a:b] if t.text[:1].isalnum() or PH.fullmatch(t.text))
        main_w = sum(1 for t in T[:main_end] if t.text[:1].isalnum() or PH.fullmatch(t.text))
        if not (2 <= adv_w <= 15) or main_w < 4:
            continue
        if any(PH.fullmatch(t.text) and not re.fullmatch(r"[\d.,%–\-\s]+", s.block.spans.get(t.text, "")) for t in T[a:b]):
            continue                                 # citations, formulas and commands never move
        adv = txt[T[a].start:T[b - 1].end]
        main = txt[T[0].start:T[main_end - 1].end]
        new = f"{_cap(adv)}, {_decap(main, T[0])}{txt[T[b - 1].end:s.end]}"

        def verify(parsed, rv=root_v, sj=subj, head=head):
            if parsed is None or not parsed.parse:
                return False, "re-parse failed"
            r2, s2 = _root_and_subject(parsed.parse[1], parsed.toks)
            ok = r2 == rv and s2 == sj and _norm(parsed.toks[0].text) == head
            return ok, "" if ok else f"re-parse changed the main clause (root {r2!r}/{rv!r}, subject {s2!r}/{sj!r})"

        return [Edit(s.start, s.end, new, "opener", f"opener:front-{lab.lower()}-{head}",
                     "this sentence opens like the previous one; fronting its final adverbial varies the opening",
                     f"parse: final {lab} '{head} ...' modifying the verb phrase", frozenset(), frozenset(),
                     structural=True, verify=verify)]
    return []


# ---- dash: parenthetical dash pairs -> parentheses; a dash before a short final list -> colon ----
def fam_dash(s, ctx):
    if ctx.hplan is None or not ctx.hplan.dash:
        return []
    out, txt = [], s.text
    ms = list(DASHES.finditer(txt))
    flank = lambda m: (txt[max(0, m.start() - 4):m.start()].rstrip()[-1:] or " ", txt[m.end():m.end() + 4].lstrip()[:1] or " ")
    if any(a in "⟧0123456789" or b in "⟦0123456789" for m in ms for a, b in [flank(m)]):
        return []                                    # '3 -- 5' is a range, not a dash
    used = set()
    for k in range(len(ms) - 1):
        m1, m2 = ms[k], ms[k + 1]
        if k in used:
            continue
        inner = txt[m1.end():m2.start()].strip()
        words = re.findall(r"[A-Za-z⟦]+", inner)
        if 1 <= len(words) <= 15 and not re.search(r"[()\[\]]", inner):
            out.append(Edit(s.start + m1.start(), s.start + m2.end(), f" ({inner}) ", "dash", "dash:pair->parens",
                            "a dash pair marks a parenthesis; parentheses say the same without the dash rhythm",
                            f"dashes around '{inner[:40]}'"))
            used |= {k, k + 1}
    for k, m in enumerate(ms):
        if k in used:
            continue
        tail = txt[m.end():].strip()
        if not tail.endswith("."):
            continue
        tt = tag_tokens(tokenize(tail[:-1]))
        if 1 <= len([t for t in tt if t.text[:1].isalnum()]) <= 8 and not any(t.tag.startswith(("VB", "MD")) for t in tt):
            out.append(Edit(s.start + m.start(), s.start + m.end(), ": ", "dash", "dash:final->colon",
                            "a dash before a short closing list or phrase reads as a colon", f"dash before '{tail[:40]}'"))
    return out


# ---- merge (cross-sentence): planned pairs in uniform paragraphs --------------------------------
def merge_candidates(bid, ss, ctx):
    out = []
    for i, j in (ctx.hplan.merge_pairs.get(bid, []) if ctx.hplan else []):
        a, b = ss[i], ss[j]
        first = b.toks[0]
        new = ", and " + _decap(first.text, first)
        e = Edit(a.toks[-1].start, first.end, new, "merge", "merge:and",
                 "two short sentences in a paragraph of uniform sentence lengths read better joined",
                 f"sentences {i}+{j}: {_nwords(a)} + {_nwords(b)} words", frozenset({first.text.lower(), first.lem}),
                 frozenset({"and", first.text.lower(), first.lem}))
        out.append((e, a, b))
    return out


HUMANIZER_FAMILIES = (("cliche", fam_cliche), ("hedge_stack", fam_hedge_stack), ("register", fam_register),
                      ("discourse", fam_discourse), ("opener", fam_opener), ("dash", fam_dash))


## 12 · Meaning gate and confidence
Hard invariants (any failure = edit rejected): protected content identical · negations · hedges, boosters and quantifiers · logical connectives · content words (only the edit's declared footprint may change) · glossary-term counts · length ratio. Structural edits must also survive a re-parse.

In [12]:
#@title 12 · Meaning gate and confidence
# An edit is applied only if every hard invariant holds AND its confidence clears the threshold.
THRESHOLDS = {"conservative": 0.85, "balanced": 0.72, "extensive": 0.60}
# Editing families (section 10) + humanizer families (section 11). Conservative only fixes and
# de-clichés; balanced adds plan-driven reconstruction of discourse and syntax; extensive adds
# open-ended lexical variation and pushes the style targets further (section 9).
_HUMAN_SAFE = {"cliche", "hedge_stack", "register"}
_HUMAN_RECON = {"discourse", "opener", "merge", "dash"}
FAMILIES_BY_INTENSITY = {
    "conservative": {"wordy", "redundancy", "grammar", "light_verb", "transition"} | _HUMAN_SAFE,
    "balanced": {"wordy", "redundancy", "grammar", "light_verb", "transition", "nominalization",
                 "passive", "split", "plain"} | _HUMAN_SAFE | _HUMAN_RECON,
    "extensive": {"wordy", "redundancy", "grammar", "light_verb", "transition", "nominalization",
                  "passive", "split", "plain", "lexical", "repetition"} | _HUMAN_SAFE | _HUMAN_RECON,
}
MODES = ("academic", "technical", "general", "conversational")
MODE_OFF = {"academic": set(), "general": set(), "conversational": set(),
            "technical": {"lexical", "repetition", "passive", "plain", "merge"}}
SPLIT_MIN = {"conservative": None, "balanced": 34, "extensive": 26}
FAMILY_MIN_PRECISION = 0.95
PRECISION_FILE = OUT / "family_precision.json"


def measured_precision():
    return json.loads(PRECISION_FILE.read_text()) if PRECISION_FILE.exists() else {}


def enabled_families(mode, intensity):
    assert mode in MODES, f"MODE must be one of {MODES}"
    fams = FAMILIES_BY_INTENSITY[intensity] - MODE_OFF[mode]
    mp = measured_precision()
    return {f for f in fams if f not in mp or mp[f]["precision"] >= FAMILY_MIN_PRECISION}


def confidence(e, s):
    mp = measured_precision()
    if e.family in mp:
        base, src = mp[e.family]["precision"], "measured"
    else:
        base, src = PRIORS[e.family], "prior"
    n = len(s.toks)
    pen = 1.0
    if n > 40:
        pen *= 0.95
    if e.structural and n > 35:
        pen *= 0.93
    near = [t for t in s.toks if PH.fullmatch(t.text) and (abs(t.start - e.start) < 12 or abs(t.end - e.end) < 12)]
    if near and not e.structural:
        pen *= 0.98
    return round(base * pen, 3), src


def _classes(toks, glossary):
    content, neg, claim, conn = Counter(), Counter(), Counter(), Counter()
    allconn = set().union(*CONNECTIVE_CLASS.values())
    for t in toks:
        w = t.text.lower()
        if w.endswith("n't") or w in NEGATIONS:
            neg["not" if w.endswith("n't") else w] += 1
        if w in CLAIM_WORDS:
            claim[w] += 1
        if w in allconn:
            conn[w] += 1
        if t.text[:1].isalpha() and t.tag[:2] in ("NN", "VB", "JJ", "RB") and canon(t.text) not in AUX_LEMMAS and w not in NEGATIONS and not w.endswith("n't"):
            content[canon(t.text)] += 1
    lem_seq = " " + " ".join(lemma(t.text, "NN") for t in toks) + " "
    gl = Counter({g: lem_seq.count(f" {g} ") for g in glossary if f" {g} " in lem_seq})
    return content, neg, claim, conn, gl


def gate(s, new_text, edits, profile, fmt):
    """Hard invariants. Returns (ok, reasons)."""
    reasons = []
    old_ph, new_ph = PH.findall(s.text), PH.findall(new_text)
    if Counter(old_ph) != Counter(new_ph):
        reasons.append("protected content added, lost or duplicated")
    if fmt == "docx":
        el = lambda seq: [p for p in seq if PUA <= ord(s.block.spans[p][0]) < PUA + 4096]
        if el(old_ph) != el(new_ph):
            reasons.append("a Word field, link or equation would move")
    removed_ok = set().union(*(e.removed_ok for e in edits))
    added_ok = set().union(*(e.added_ok for e in edits))
    nt = tag_tokens(tokenize(new_text))
    a = _classes(s.toks, profile.glossary)
    b = _classes(nt, profile.glossary)
    names = ["content word", "negation", "hedge/booster/quantifier", "logical connective"]
    for name, ca, cb in zip(names, a[:4], b[:4]):
        lost = [w for w in (ca - cb) if w not in removed_ok]
        gained = [w for w in (cb - ca) if w not in added_ok]
        if lost:
            reasons.append(f"{name} lost: {sorted(lost)}")
        if gained:
            reasons.append(f"{name} added: {sorted(gained)}")
    if a[4] != b[4]:
        reasons.append("glossary term count changed")
    ratio = len(new_text) / max(1, len(s.text))
    lo, hi = (0.9, 1.25) if any(e.family == "split" for e in edits) else (0.4, 1.2)
    small_cut = 0 <= len(s.text) - len(new_text) <= 25      # "Moreover, it may potentially help." -> "It may help."
    if not (lo <= ratio <= hi or (ratio < lo and small_cut)):
        reasons.append(f"length ratio {ratio:.2f} outside [{lo}, {hi}]")
    return (not reasons), reasons


def apply_edits(text, base, edits):
    """Apply non-overlapping edits (offsets relative to the block) to a sentence text starting at base."""
    for e in sorted(edits, key=lambda e: -e.start):
        text = text[:e.start - base] + e.new + text[e.end - base:]
    return text

## 13 · Engine
analyse → **style plan** → propose (editing + humanizer families) → gate → select → cross-sentence merges (gated on both sentences together) → render → verify integrity → re-measure the style markers on the output.

In [13]:
#@title 13 · Orchestrator: analyse -> plan -> propose -> gate -> record -> render -> verify
import time, tracemalloc, threading, psutil, csv, html as _html


@dataclass
class Record:
    rid: str
    doc: str
    file: str
    block: int
    section: str
    family: str
    rule: str
    before: str
    after: str
    sentence_before: str
    sentence_after: str
    reason: str
    signal: str
    confidence: float
    conf_source: str
    status: str            # applied | rejected-gate | below-threshold | overlap | advice | refused-docx
    gate_reasons: str
    start: int
    end: int
    new: str


class PeakRSS:
    """Samples resident memory every 50 ms in a background thread."""
    def __enter__(self):
        self.proc, self.peak, self._run = psutil.Process(), 0, True
        self.t = threading.Thread(target=self._loop, daemon=True); self.t.start(); return self

    def _loop(self):
        while self._run:
            self.peak = max(self.peak, self.proc.memory_info().rss); time.sleep(0.05)

    def __exit__(self, *a):
        self._run = False; self.t.join()


def _rid(doc, bl, e):
    return hashlib.sha1(f"{doc.path.name}|{bl.file}|{bl.bid}|{e.start}|{e.end}|{e.rule}|{e.new}".encode()).hexdigest()[:12]


def _passive_trigger(s):
    T = s.toks
    return any(T[i].lem == "be" and i + 1 < len(T) and T[i + 1].tag == "VBN" for i in range(len(T) - 1)) and \
        any(t.text.lower() == "by" for t in T)


def _split_trigger(s, min_len):
    if min_len is None:
        return False
    n = sum(1 for t in s.toks if t.text[:1].isalnum() or PH.fullmatch(t.text))
    return n >= min_len and (";" in [t.text for t in s.toks] or re.search(r", (and|but) ", s.text))


def freeze_terms(prof, terms):
    """Words/phrases the user froze: never substituted by any family (glossary words, every inflection)
    AND counted by the meaning gate (glossary terms), so an edit that changes them is rejected."""
    for term in terms or ():
        ws = re.findall(r"[A-Za-z0-9][A-Za-z0-9'’-]*", str(term).lower())
        if not ws:
            continue
        for w in ws:
            forms = {w, canon(w), lemma(w, "NN")}
            for up in ("NOUN", "VERB", "ADJ", "ADV"):
                for base in {w, canon(w)}:
                    forms |= {f.lower() for fs in getAllInflections(base, upos=up).values() for f in fs}
            prof.glossary_words |= forms
            if len(ws) == 1:
                prof.glossary |= {lemma(f, "NN") for f in forms}
        if len(ws) > 1:
            prof.glossary.add(" ".join(lemma(w, "NN") for w in ws))


def process(path, mode=MODE, intensity=INTENSITY, write=True, timings=None, freeze=()):
    """Run the whole pipeline on one document. Returns (doc, records, output_path, report).
    `freeze`: words or phrases that must come out exactly as they went in."""
    tm = timings if timings is not None else {}
    t0 = time.perf_counter()
    doc = load_document(path); tm["ingest"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    sents = {bl.bid: split_sentences(bl) for bl in doc.blocks}
    prof = build_profile(doc, sents); freeze_terms(prof, freeze); tm["analysis"] = time.perf_counter() - t0

    records, accepted_by_block = [], defaultdict(list)
    plan = {}
    for bl in doc.blocks:
        if bl.section in SKIP_SECTIONS or not is_prose_block(bl):
            continue
        eff = "conservative" if bl.section in CONSERVATIVE_SECTIONS else intensity
        plan[bl.bid] = (eff, enabled_families(mode, eff), SPLIT_MIN[eff] if mode != "technical" or SPLIT_MIN[eff] is None else SPLIT_MIN[eff] + 10)

    # humanizer analysis: measure the style markers of the editable text, then plan where to act
    t0 = time.perf_counter()
    style_before = style_markers({bid: sents[bid] for bid in plan})
    hplan = plan_humanizer(sents, plan, style_before, mode, intensity)
    tm["analysis"] += time.perf_counter() - t0

    t0 = time.perf_counter()
    to_parse = [s for bid, (eff, fams, smin) in plan.items() for s in sents[bid]
                if ("passive" in fams and _passive_trigger(s)) or ("split" in fams and _split_trigger(s, smin))
                or ("opener" in fams and (bid, s.idx) in hplan.opener)]
    parse_batch(to_parse); tm["parse"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    lt_by_block = {bid: lt_matches(doc.blocks[bid]) for bid, (eff, fams, _) in plan.items() if "grammar" in fams}
    tm["languagetool"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    pending_struct = []
    cand_by_sent = {}
    for bid, (eff, fams, smin) in plan.items():
        bl = doc.blocks[bid]
        ctx = Ctx(prof, mode, eff, doc.fmt, block_sents=sents[bid], lt=lt_by_block.get(bid), hplan=hplan)
        for s in sents[bid]:
            cands = []
            for fam, fn in (("wordy", fam_wordy), ("redundancy", fam_redundancy), ("light_verb", fam_light_verb),
                            ("transition", fam_transition), ("grammar", fam_grammar), ("plain", fam_plain), ("lexical", fam_lexical),
                            ("repetition", fam_repetition), ("passive", fam_passive)) + HUMANIZER_FAMILIES:
                if fam in fams or (fam == "light_verb" and "nominalization" in fams):
                    cands += [e for e in fn(s, ctx) if e.family in fams]
            if "split" in fams:
                cands += fam_split(s, ctx, smin)
            for e in cands:
                e.sentence = s
                e.conf, e.conf_source = confidence(e, s)
                if e.structural:
                    pending_struct.append(e)
            cand_by_sent[(bid, s.idx)] = (s, cands, eff)
        for s, kind, msg in advice_for(sents[bid], prof):
            records.append(Record(hashlib.sha1(f"{doc.path.name}|{bid}|{s.idx}|{kind}".encode()).hexdigest()[:12],
                                  doc.path.name, bl.file, bid, bl.section, "advice", kind, "", "", display(unmask(s.text, bl.spans), bl),
                                  "", msg, kind, 0.0, "-", "advice", "", s.start, s.end, ""))

    # structural edits must survive a re-parse of the rewritten sentence
    checks = []
    for e in pending_struct:
        if e.verify is None:
            continue
        nb = Block(-1, "", 0, 0, "", e.new, {})
        ns = split_sentences(nb)
        if len(ns) == 1:
            checks.append((e, ns[0]))
        else:
            e.verify = lambda parsed: (False, "rewritten sentence does not segment as one sentence")
            checks.append((e, None))
    parse_batch([ps for _, ps in checks if ps is not None])
    for e, ps in checks:
        ok, why = e.verify(ps) if ps is not None else e.verify(None)
        e.verified = (ok, why)
    tm["rules_and_gate"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    thr = THRESHOLDS
    for (bid, _), (s, cands, eff) in cand_by_sent.items():
        bl = doc.blocks[bid]
        chosen = []
        status = {}
        for e in sorted(cands, key=lambda e: (-e.conf, e.start, e.rule)):
            if e.conf < thr[eff]:
                status[id(e)] = ("below-threshold", ""); continue
            vr = getattr(e, "verified", (True, ""))
            if not vr[0]:
                status[id(e)] = ("rejected-gate", vr[1]); continue
            if any(not (e.end <= c.start or c.end <= e.start) for c in chosen):
                status[id(e)] = ("overlap", ""); continue
            ok, why = gate(s, apply_edits(s.text, s.start, [e]), [e], prof, doc.fmt)
            if not ok:
                status[id(e)] = ("rejected-gate", "; ".join(why)); continue
            chosen.append(e); status[id(e)] = ("applied", "")
        while chosen:                          # the combination must pass too
            ok, why = gate(s, apply_edits(s.text, s.start, chosen), chosen, prof, doc.fmt)
            if ok:
                break
            worst = min(chosen, key=lambda e: e.conf)
            chosen.remove(worst); status[id(worst)] = ("rejected-gate", "combined: " + "; ".join(why))
        accepted_by_block[bid] += chosen
        new_sent = apply_edits(s.text, s.start, chosen)
        for e in cands:
            st, why = status[id(e)]
            records.append(Record(_rid(doc, bl, e), doc.path.name, bl.file, bid, bl.section, e.family, e.rule,
                                  display(unmask(bl.masked[e.start:e.end], bl.spans), bl), display(unmask(e.new, bl.spans), bl),
                                  display(unmask(s.text, bl.spans), bl), display(unmask(apply_edits(s.text, s.start, [e]), bl.spans), bl),
                                  e.reason, e.signal, e.conf, e.conf_source, st, why, e.start, e.end, e.new))

    # cross-sentence reconstruction (merge): gated on both sentences together, after every
    # within-sentence edit is settled, so it never overlaps or contradicts one of them
    for bid, (eff, fams, smin) in plan.items():
        if "merge" not in fams or bid not in hplan.merge_pairs:
            continue
        bl = doc.blocks[bid]
        ctx = Ctx(prof, mode, eff, doc.fmt, block_sents=sents[bid], hplan=hplan)
        for e, a, b in merge_candidates(bid, sents[bid], ctx):
            pseudo = Sentence(bl, a.idx, a.start, b.end, bl.masked[a.start:b.end], a.toks + b.toks)
            e.sentence = pseudo
            e.conf, e.conf_source = confidence(e, a)
            inside = [x for x in accepted_by_block[bid] if a.start <= x.start and x.end <= b.end]
            new_pair = apply_edits(pseudo.text, pseudo.start, inside + [e])
            if e.conf < thr[eff]:
                st, why = "below-threshold", ""
            elif any(not (e.end <= x.start or x.end <= e.start) for x in accepted_by_block[bid]):
                st, why = "overlap", ""
            elif len(split_sentences(Block(-1, "", 0, 0, "", new_pair, {}))) != 1:
                st, why = "rejected-gate", "merged text does not segment as one sentence"
            else:
                ok, reasons = gate(pseudo, new_pair, inside + [e], prof, doc.fmt)
                st, why = ("applied", "") if ok else ("rejected-gate", "; ".join(reasons))
            if st == "applied":
                accepted_by_block[bid].append(e)
            records.append(Record(_rid(doc, bl, e), doc.path.name, bl.file, bid, bl.section, e.family, e.rule,
                                  display(unmask(bl.masked[e.start:e.end], bl.spans), bl), display(unmask(e.new, bl.spans), bl),
                                  display(unmask(pseudo.text, bl.spans), bl),
                                  display(unmask(apply_edits(pseudo.text, pseudo.start, [e]), bl.spans), bl),
                                  e.reason, e.signal, e.conf, e.conf_source, st, why, e.start, e.end, e.new))
    tm["select"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    new_masked = {bid: apply_edits(doc.blocks[bid].masked, 0, edits) for bid, edits in accepted_by_block.items() if edits}
    rendered, refused = render_document(doc, new_masked)
    for r in records:
        if r.block in refused and r.status == "applied":
            r.status = "refused-docx"
    out_path = write_document(doc, rendered) if write else None
    tm["render"] = time.perf_counter() - t0
    integrity = verify_integrity(doc, rendered, new_masked, refused)
    after_sents = {bid: split_sentences(Block(bid, "", 0, 0, doc.blocks[bid].section, new_masked.get(bid, doc.blocks[bid].masked), {}))
                   for bid in plan}
    style_after = style_markers(after_sents)
    return doc, records, out_path, {"profile": prof, "integrity": integrity, "timings": tm, "new_masked": new_masked,
                                    "style_before": style_before, "style_after": style_after,
                                    "style_check": {"before": markers_vs_target(style_before, mode, intensity),
                                                    "after": markers_vs_target(style_after, mode, intensity)},
                                    "plan": hplan, "mode": mode, "intensity": intensity}


def verify_integrity(doc, rendered, new_masked, refused):
    """Independent check on the OUTPUT: bytes outside edited blocks are identical, and every block's
    protected content is present exactly as before (same multiset of protected strings)."""
    problems = []
    if doc.fmt == "docx":
        fresh = read_docx(doc.path)
        out_tmp = OUT / ".integrity_tmp.docx"; rendered.save(out_tmp)
        after = read_docx(out_tmp); out_tmp.unlink()
        body_xml = lambda d: [el.tag for el in d.element.body.iter()]
        if len(after.blocks) != len(fresh.blocks):
            problems.append("paragraph count changed")
        tags_a = Counter(t for t in body_xml(fresh.docx) if not t.endswith(("}t", "}r", "}rPr")))
        tags_b = Counter(t for t in body_xml(after.docx) if not t.endswith(("}t", "}r", "}rPr")))
        if tags_a != tags_b:
            problems.append("non-text Word structure changed")
        for a, b in zip(fresh.blocks, after.blocks):
            pa = Counter(a.spans.values()); pb = Counter(b.spans.values())
            if Counter(v for v in pa.elements() if not (PUA <= ord(v[0]) < PUA + 4096)) != \
               Counter(v for v in pb.elements() if not (PUA <= ord(v[0]) < PUA + 4096)):
                problems.append(f"block {a.bid}: protected text changed")
        return {"ok": not problems, "problems": problems}
    for key, txt in doc.sources.items():
        new = rendered[key]
        blocks = sorted((b for b in doc.blocks if b.file == key), key=lambda b: b.start)
        pos_o, pos_n = 0, 0
        for b in blocks:
            nb = unmask(new_masked.get(b.bid, b.masked), b.spans)
            if txt[pos_o:b.start] != new[pos_n:pos_n + (b.start - pos_o)]:
                problems.append(f"{key}: text outside block {b.bid} changed"); break
            pos_n += b.start - pos_o
            if new[pos_n:pos_n + len(nb)] != nb:
                problems.append(f"{key}: block {b.bid} not where expected"); break
            seq_o = PH.findall(b.masked); seq_n = PH.findall(new_masked.get(b.bid, b.masked))
            if Counter(seq_o) != Counter(seq_n):
                problems.append(f"{key}: block {b.bid} protected content changed")
            pos_n += len(nb); pos_o = b.end
        if not problems and txt[pos_o:] != new[pos_n:]:
            problems.append(f"{key}: tail changed")
    return {"ok": not problems, "problems": problems}


def records_frame(records):
    import pandas as pd
    return pd.DataFrame([r.__dict__ for r in records])

## 14 · Review and reversibility

In [14]:
#@title 14 · Review: style profile, side-by-side diff, accept/reject by id, re-render (reversibility)
def _diff_html(a, b):
    sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
    out = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            out.append(_html.escape(a[i1:i2]))
        if tag in ("delete", "replace"):
            out.append(f"<del>{_html.escape(a[i1:i2])}</del>")
        if tag in ("insert", "replace"):
            out.append(f"<ins>{_html.escape(b[j1:j2])}</ins>")
    return "".join(out)


def _style_html(report):
    """Style markers before -> after against the mode's targets, plus the plan the engine followed."""
    if not report or "style_check" not in report:
        return ""
    b, a = report["style_check"]["before"], report["style_check"]["after"]
    rows = "".join(
        f"<tr><td>{_html.escape(k)}</td><td>{b[k][0]}</td><td>{a[k][0] if k in a else ''}</td><td>{_html.escape(b[k][1])}</td>"
        f"<td>{'within' if (a.get(k, b[k])[2]) else '<b>outside</b>'}</td></tr>" for k in b)
    notes = "".join(f"<li>{_html.escape(n)}</li>" for n in report["plan"].notes) or "<li>every marker already within target: no reconstruction planned</li>"
    return (f"<h2>Style profile — mode <code>{report['mode']}</code>, intensity <code>{report['intensity']}</code></h2>"
            f"<table><tr><th>marker</th><th>before</th><th>after</th><th>target</th><th>after vs target</th></tr>{rows}</table>"
            f"<p>Plan derived from the markers:</p><ul>{notes}</ul>"
            f"<p><small>Markers describe writing style (variation, stock phrasing, connective density). "
            f"They are not an authorship or AI-detection score.</small></p>")


def review_files(doc, records, tag="dnwe", report=None):
    """Writes <name>.edits.csv (one row per proposed edit, with a 'decision' column you can edit)
    and <name>.review.html (style profile, colour diff, rule, reason, confidence)."""
    stem = doc.path.stem
    csv_path, html_path = OUT / f"{stem}.{tag}.edits.csv", OUT / f"{stem}.{tag}.review.html"
    rows = [r for r in records if r.status != "advice"]
    with open(csv_path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["rid", "decision", "status", "family", "rule", "section", "confidence", "conf_source",
                    "before", "after", "reason", "gate_reasons", "sentence_before", "sentence_after"])
        for r in rows:
            w.writerow([r.rid, "accept" if r.status == "applied" else "", r.status, r.family, r.rule, r.section,
                        r.confidence, r.conf_source, r.before, r.after, r.reason, r.gate_reasons,
                        r.sentence_before, r.sentence_after])
    applied = [r for r in rows if r.status == "applied"]
    other = [r for r in rows if r.status != "applied"]
    adv = [r for r in records if r.status == "advice"]

    def table(rs, title):
        h = [f"<h2>{_html.escape(title)} ({len(rs)})</h2><table><tr><th>id</th><th>family</th><th>conf</th><th>edit</th><th>why</th></tr>"]
        for r in rs:
            why = _html.escape(r.reason + (f" — gate: {r.gate_reasons}" if r.gate_reasons else ""))
            h.append(f"<tr><td><code>{r.rid}</code></td><td>{r.family}<br><small>{r.section}</small></td>"
                     f"<td>{r.confidence:.2f}<br><small>{r.conf_source}</small></td>"
                     f"<td>{_diff_html(r.sentence_before, r.sentence_after)}</td><td><small>{why}</small></td></tr>")
        return "".join(h) + "</table>"

    page = f"""<!doctype html><meta charset="utf-8"><title>{_html.escape(stem)} review</title>
<style>body{{font:14px/1.5 system-ui,sans-serif;margin:24px;max-width:1200px}}table{{border-collapse:collapse;width:100%}}
td,th{{border-bottom:1px solid #ddd;padding:6px;vertical-align:top;text-align:left}}del{{background:#fdd;text-decoration:line-through}}
ins{{background:#dfd;text-decoration:none}}code{{font-size:12px}}</style>
<h1>DNWE review — {_html.escape(doc.path.name)}</h1>
<p>{len(applied)} edits applied, {len(other)} proposed but not applied, {len(adv)} advisory notes.
To reject an applied edit, set its <code>decision</code> to <code>reject</code> in <code>{csv_path.name}</code> and run
<code>apply_decisions()</code>.</p>
{_style_html(report)}
{table(applied, "Applied")}{table(other, "Not applied")}
<h2>Advice ({len(adv)})</h2><ul>{"".join(f"<li>{_html.escape(r.signal)}: {_html.escape(r.sentence_before[:200])}</li>" for r in adv)}</ul>"""
    html_path.write_text(page, encoding="utf-8")
    return csv_path, html_path


def apply_decisions(path, records, csv_path, tag="dnwe.reviewed"):
    """Re-render the ORIGINAL document with only the edits marked 'accept' in the CSV.
    Edits never depend on each other, so any subset can be applied."""
    import pandas as pd
    dec = pd.read_csv(csv_path, dtype=str).fillna("")
    keep = set(dec.loc[dec["decision"].str.lower() == "accept", "rid"])
    doc = load_document(path)
    by_block = defaultdict(list)
    for r in records:
        if r.rid in keep:
            by_block[r.block].append(Edit(r.start, r.end, r.new, r.family, r.rule, "", ""))
    new_masked = {bid: apply_edits(doc.blocks[bid].masked, 0, es) for bid, es in by_block.items()}
    rendered, refused = render_document(doc, new_masked)
    integ = verify_integrity(doc, rendered, new_masked, refused)
    assert integ["ok"], integ
    return write_document(doc, rendered, tag=tag)

## 15 · Dev set and measured precision
Labels follow the *edit* (document, family, sentence before and after), not its character offsets, so they survive engine changes. Every label you have ever given is kept in `out/labels_archive.csv`.

In [15]:
#@title 15 · Dev set: export candidate edits for YOUR labels, then measure each family's precision
# Workflow: run the corpus (section 16) -> export_dev_set() -> open out/dev_set.csv and fill the
# 'label' column with: correct | wrong | meaning  -> run load_labels() -> re-run the corpus.
# Until labels exist, confidences are PRIORS (marked "prior" everywhere) — not measurements.
#
# Labels follow the EDIT, not its position: an edit is identified by (document, family, sentence
# before, sentence after), so a label survives engine changes that shift offsets. Every label ever
# given is kept in out/labels_archive.csv; nothing you labelled is ever dropped.
import random
DEV_CSV = OUT / "dev_set.csv"
TEST_CSV = OUT / "test_set.csv"
LABEL_ARCHIVE = OUT / "labels_archive.csv"
_LABEL_COLS = ["rid", "family", "doc", "section", "sentence_before", "sentence_after", "label"]


def _ckey(doc, family, before, after):
    norm = lambda x: re.sub(r"\s+", " ", str(x)).strip()
    return f"{doc}|{family}|{norm(before)}|{norm(after)}"


def _split_of(key):
    """dev or test from a hash of the edit itself: stable however the candidate pool changes."""
    return "dev" if int(hashlib.sha1(key.encode()).hexdigest(), 16) % 2 == 0 else "test"


def archive_labels():
    """Fold every labelled row of dev_set.csv / test_set.csv into the archive (latest label wins)."""
    import pandas as pd
    frames = [pd.read_csv(p, dtype=str).fillna("") for p in (LABEL_ARCHIVE, DEV_CSV, TEST_CSV) if p.exists()]
    if not frames:
        return pd.DataFrame(columns=_LABEL_COLS + ["key"])
    allr = pd.concat([f.reindex(columns=_LABEL_COLS).fillna("") for f in frames], ignore_index=True)
    allr["label"] = allr["label"].str.strip().str.lower()
    allr = allr[allr.label != ""].copy()
    allr["key"] = [_ckey(*v) for v in allr[["doc", "family", "sentence_before", "sentence_after"]].values]
    allr = allr.drop_duplicates("key", keep="last")
    allr.drop(columns="key").to_csv(LABEL_ARCHIVE, index=False)
    return allr


def export_dev_set(records, per_family=40, seed=13):
    """Stratified sample of gated candidates (applied or blocked only by confidence), split by a
    content hash into a dev set (sets thresholds) and a held-out test set (checks them).
    Earlier labels are carried over by rid or by edit content."""
    import pandas as pd
    arch = archive_labels()
    by_rid = dict(zip(arch["rid"], arch["label"])) if len(arch) else {}
    by_key = dict(zip(arch["key"], arch["label"])) if len(arch) else {}
    rng = random.Random(seed)
    pool = defaultdict(list)
    for r in records:
        if r.status in ("applied", "below-threshold", "overlap") and r.family != "advice":
            pool[r.family].append(r)
    rows, seen = [], set()
    for fam in sorted(pool):
        rs = sorted(pool[fam], key=lambda r: r.rid)
        rng.shuffle(rs)
        for r in rs[:per_family]:
            key = _ckey(r.doc, fam, r.sentence_before, r.sentence_after)
            if key in seen:
                continue
            seen.add(key)
            rows.append({"rid": r.rid, "split": _split_of(key), "family": fam, "doc": r.doc,
                         "section": r.section, "sentence_before": r.sentence_before,
                         "sentence_after": r.sentence_after, "label": by_rid.get(r.rid) or by_key.get(key, "")})
    df = pd.DataFrame(rows, columns=["rid", "split", "family", "doc", "section", "sentence_before", "sentence_after", "label"])
    df[df.split == "dev"].to_csv(DEV_CSV, index=False)
    df[df.split == "test"].to_csv(TEST_CSV, index=False)
    carried = int((df.label != "").sum())
    print(f"Wrote {sum(df.split == 'dev')} dev and {sum(df.split == 'test')} test items ({df.family.nunique()} families) "
          f"to {DEV_CSV.name} / {TEST_CSV.name}; {carried} already labelled, {len(df) - carried} need a label. "
          f"Archive: {len(arch)} labels in {LABEL_ARCHIVE.name}.")
    return df


def load_labels():
    """Precision per family from YOUR dev labels (Laplace-smoothed), checked on the test labels.
    Writes out/family_precision.json, which the engine uses instead of the priors. A family needs
    at least MIN_LABELS dev labels before its measurement replaces the prior."""
    import pandas as pd
    archive_labels()
    res = {}
    dev = pd.read_csv(DEV_CSV, dtype=str).fillna("")
    test = pd.read_csv(TEST_CSV, dtype=str).fillna("") if TEST_CSV.exists() else dev.iloc[0:0].copy()
    dev["label"] = dev["label"].str.strip().str.lower()
    test["label"] = test["label"].str.strip().str.lower()
    bad = set(dev.label) - {"", "correct", "wrong", "meaning"}
    bad |= set(test.label) - {"", "correct", "wrong", "meaning"}
    assert not bad, f"unknown labels {bad}: use correct | wrong | meaning"
    for fam, g in dev[dev.label != ""].groupby("family"):
        c, n = (g.label == "correct").sum(), len(g)
        t = test[(test.family == fam) & (test.label != "")]
        tc, tn = (t.label == "correct").sum(), len(t)
        res[fam] = {"precision": round((c + 1) / (n + 2), 3), "dev_n": int(n), "dev_correct": int(c),
                    "test_n": int(tn), "test_precision": round(tc / tn, 3) if tn else None,
                    "meaning_changes": int((g.label == "meaning").sum())}
    MIN_LABELS = 15
    usable = {f: v for f, v in res.items() if v["dev_n"] >= MIN_LABELS}
    PRECISION_FILE.write_text(json.dumps(usable, indent=2))
    for fam, v in sorted(res.items()):
        flag = ("too few labels: keeps prior" if v["dev_n"] < MIN_LABELS else
                "ENABLED" if v["precision"] >= FAMILY_MIN_PRECISION else "disabled (below target)")
        print(f"{fam:15} dev {v['dev_correct']}/{v['dev_n']} -> {v['precision']:.3f}  test {v['test_precision']}  {flag}")
    unl = pd.concat([dev, test])
    unl = unl[unl.label == ""].groupby("family").size()
    if len(unl):
        print("still unlabelled:", ", ".join(f"{f} {n}" for f, n in unl.items()))
    return res


## 16 · Run on the corpus
Processes every document in `corpus/` and writes outputs plus review pages to `out/`. An assertion fails the cell if any output violates integrity. Each document also reports how many style markers were outside their target before and after. Every document is also written as `.txt`, `.md` and `.pdf` (`<name>.dnwe.<ext>`).

In [16]:
#@title 16 · Run the engine on every document in corpus/ (real files only)
def discover_corpus(root=CORPUS):
    """DOCX/MD/TXT/PDF files in corpus/, plus the main .tex of every LaTeX project folder."""
    files = sorted(p for p in root.iterdir() if p.is_file() and p.suffix.lower() in (".docx", ".md", ".txt", ".pdf"))
    for d in sorted(p for p in root.iterdir() if p.is_dir()):
        mains = sorted(t for t in d.rglob("*.tex") if "\\begin{document}" in t.read_text(encoding="utf-8", errors="replace"))
        if mains:
            files.append(mains[0])
    return files


CORPUS_FILES = discover_corpus()
assert CORPUS_FILES, f"Put your papers in {CORPUS} (DOCX, LaTeX folders, MD, TXT, PDF) and re-run."
RUNS = {}
for f in CORPUS_FILES:
    tm = {}
    tracemalloc.start()
    with PeakRSS() as mem:
        t0 = time.perf_counter()
        doc, recs, out_path, rep = process(f, MODE, INTENSITY, timings=tm)
        total = time.perf_counter() - t0
    py_peak = tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    csv_path, html_path = review_files(doc, recs, report=rep)
    for ext, data in export_all(doc, render_document(doc, rep["new_masked"])[0]).items():
        if ext in ("txt", "md", "pdf") and not (out_path and Path(out_path).suffix == f".{ext}"):
            (OUT / f"{doc.path.stem}.dnwe.{ext}").write_bytes(data)    # every document also as .txt / .md / .pdf
    RUNS[f] = {"doc": doc, "records": recs, "out": out_path, "report": rep, "seconds": total,
               "peak_rss_mb": mem.peak / 2**20, "py_peak_mb": py_peak / 2**20, "timings": tm,
               "csv": csv_path, "html": html_path}
    st = Counter(r.status for r in recs)
    assert rep["integrity"]["ok"], (f, rep["integrity"])
    print(f"{f.name:40} {rep['profile'].words:6} words  {total:6.1f}s  applied {st['applied']:3}  "
          f"blocked {st['rejected-gate']:3}  low-conf {st['below-threshold']:3}  advice {st['advice']:3}  integrity OK")
    out_b = [k for k, v in rep["style_check"]["before"].items() if not v[2]]
    out_a = [k for k, v in rep["style_check"]["after"].items() if not v[2]]
    print(f"{'':40} style markers outside target: {len(out_b)} before -> {len(out_a)} after"
          + (f"  (still outside: {', '.join(out_a)})" if out_a else ""))

ALL_RECORDS = [r for v in RUNS.values() for r in v["records"]]
print(f"\n{len(CORPUS_FILES)} documents, {sum(r.status == 'applied' for r in ALL_RECORDS)} edits applied. "
      f"Outputs and review pages are in {OUT}")

2026-11-23 21:08:51,569 : INFO : language_tool_python initializing with port: 8619


2026-11-23 21:09:01,498 : INFO : maxp pruned


2026-11-23 21:09:01,499 : INFO : cmap pruned


2026-11-23 21:09:01,500 : INFO : kern dropped


2026-11-23 21:09:01,501 : INFO : post pruned


2026-11-23 21:09:01,502 : INFO : FFTM dropped


... (2,079 outputs left out)

2026-11-23 21:09:52,506 : INFO : OS/2 CodePage ranges pruned: [0]


2026-11-23 21:09:52,507 : INFO : glyf pruned


2026-11-23 21:09:52,507 : INFO : name pruned


main_morphena.tex                          3080 words     7.5s  applied  12  blocked   2  low-conf   0  advice   3  integrity OK
                                         style markers outside target: 0 before -> 0 after

8 documents, 31 edits applied. Outputs and review pages are in /home/saifarfi/Volumetric/Sapienaizerv2/out


## 17 · Tests

In [17]:
#@title 17 · Tests: protection, segmentation, gate refusals, every family (positive + refusal), guards, end-to-end
import pandas as pd, tempfile as _tf, pdfplumber


def _sent(text, section="body"):
    mk = Masker()
    masked, spans = mk.mask_atoms([(text, False)])
    bl = Block(0, "t", 0, len(text), section, masked, spans)
    return bl, split_sentences(bl)


def _profile_for(text):
    bl, ss = _sent(text)
    d = Doc(Path("t.txt"), "txt", {"t": text}, [bl])
    return build_profile(d, {0: ss}), bl, ss


def _run_family(fn, text, mode="academic", hplan=None, **kw):
    """Proposals of one family on `text`, each with the gate verdict (and the re-parse check
    for structural edits that carry one)."""
    prof, bl, ss = _profile_for(text)
    ctx = Ctx(prof, mode, "balanced", "txt", block_sents=ss, lt=lt_matches(bl) if fn is fam_grammar else None, hplan=hplan)
    if fn in (fam_passive, fam_split, fam_opener):
        parse_batch(ss)
    out = []
    for s in ss:
        for e in fn(s, ctx, **kw):
            new = apply_edits(s.text, s.start, [e])
            ok, why = gate(s, new, [e], prof, "txt")
            if ok and e.verify is not None:
                ns = split_sentences(Block(-1, "", 0, 0, "", e.new, {}))
                parse_batch(ns)
                ok, w2 = e.verify(ns[0]) if len(ns) == 1 else (False, "not one sentence")
                why = why + ([w2] if w2 else [])
            out.append((unmask(new, bl.spans), ok, why))
    return out


def _plan(**kw):
    hp = HumanPlan("academic", "balanced")
    for k, v in kw.items():
        setattr(hp, k, v)
    return hp


RESULTS = []


def check(name, cond, detail=""):
    RESULTS.append({"test": name, "pass": bool(cond), "detail": str(detail)[:200]})


# 1. protection: rendering with no edits reproduces every corpus file byte for byte
for f in CORPUS_FILES:
    d = load_document(f)
    rendered, _ = render_document(d, {})
    if d.fmt == "docx":
        with _tf.TemporaryDirectory() as td:
            p = Path(td) / "x.docx"; rendered.save(p)
            same = [b.masked for b in read_docx(p).blocks] == [b.masked for b in d.blocks]
    else:
        same = all(rendered[k] == d.sources[k] for k in d.sources)
    check(f"round-trip unchanged: {f.name}", same)
    check(f"unmask restores every block: {f.name}",
          all(PH.findall(unmask(b.masked, b.spans)) == [] or d.fmt == "docx" for b in d.blocks))

# 2. segmentation: protected numbers, codes and URLs never swallow the full stop
for text, n in [("We report U0 on QM9. The module is small.", 2), ("It ran in 2020. Then it stopped.", 2),
                ("See https://example.org/a. Done.", 2), ("Accuracy rose to 95%. It held.", 2),
                ("Version v2.0 is stable. It ships today.", 2), ("We used 3.5 g of salt.", 1)]:
    got = [s.text for s in _sent(text)[1]]
    check(f"segmentation: {text!r} -> {n} sentence(s)", len(got) == n, got)

# 3. non-prose blocks (command listings) are never edited
check("non-prose: a command listing is not prose", not is_prose_block(_sent("squeue\nscontrol show job 12\nsacct")[0]))
check("non-prose: a wrapped paragraph is prose", is_prose_block(_sent(
    "We evaluate the method on three datasets and report the mean error\nacross five seeds, together with the standard deviation of each\nestimate.")[0]))

# 4. the gate refuses meaning changes
prof, bl, ss = _profile_for("The treatment did not reduce mortality in 12 of 40 patients, which may suggest a weak effect (Lee et al., 2021).")
s = ss[0]
for label, new in [("negation removed", s.text.replace("did not reduce", "reduced")),
                   ("hedge strengthened", s.text.replace("may suggest", "proves")),
                   ("number dropped", PH.sub("", s.text, count=1)),
                   ("citation dropped", s.text.replace(PH.findall(s.text)[-1], "")),
                   ("content word swapped", s.text.replace("mortality", "survival"))]:
    ok, why = gate(s, new, [Edit(0, 0, "", "test", "test", "", "")], prof, "txt")
    check(f"gate blocks {label}", not ok, "; ".join(why))
ok, _ = gate(s, s.text, [], prof, "txt")
check("gate passes an unchanged sentence", ok)

# 5. one positive case per editing family (section 10)
cases = [
    ("wordy", fam_wordy, "Due to the fact that the samples were small, we repeated the assay.", "Because the samples", {}),
    ("redundancy", fam_redundancy, "The end result of the procedure was a stable estimate of the error.", "The result of", {}),
    ("light_verb", fam_light_verb, "We conducted an analysis of the residuals before fitting the final model.", "We analyzed the residuals", {}),
    ("plain", fam_plain, "We utilize a hierarchical model to pool information across all sites.", "We use a hierarchical", {}),
    ("plain+prep", fam_plain, "The agents assist in editing long recordings for the archive.", "help with editing", {}),
    ("transition", fam_transition, "Moreover, the variance was small. Furthermore, the bias was negligible.", "The bias was negligible", {}),
    ("passive", fam_passive, "The final dataset was curated by two independent annotators.", "Two independent annotators curated the final dataset.", {}),
    ("split", fam_split, "The estimator converges quickly on synthetic graphs with known structure, but the bootstrap intervals remain poorly calibrated for the smallest sample sizes that we tested in the experiments.", ". However, the bootstrap", {"min_len": 26}),
    ("grammar", fam_grammar, "This is an useful result for the field of network epidemiology.", "a useful", {}),
]
for fam, fn, text, expect, kw in cases:
    got = _run_family(fn, text, **kw)
    check(f"family {fam} proposes and passes the gate", [g for g in got if expect in g[0] and g[1]], got[0][0] if got else "no proposal")

# 6. refusals that protect meaning (regressions from the labelled dev/test sets included)
for name, fn, text, kw in [
    ("passive with misattached 'of' agent", fam_passive, "The comparisons are given by the limits of the sequence.", {}),
    ("light verb 'make decisions of' not rewritten", fam_light_verb, "Systems make the decisions of the committee correctly.", {}),
    ("no swap inside a direct quotation", fam_plain, "They wrote that \"we utilize the full sample\" in the protocol.", {}),
    ("'prior' as a noun is not 'before'", fam_wordy, "These results show the value of a frozen feature prior to guide diffusion models toward stable restoration.", {}),
    ("'terminate' stays a term of art in academic mode", fam_plain, "The job is terminated when the allocation expires on the cluster.", {}),
    ("no semicolon split in academic mode", fam_split, "For a later comparison, the partitions and metric definitions should remain fixed across all runs; hardware latency would then be an additional reported outcome rather than a confound.", {"min_len": 26}),
    ("no 'and' split below 45 words", fam_split, "A quantum-inspired network has separately been evaluated directly on the benchmark, and a genuine hybrid framework with a real quantum circuit has been applied to the same property prediction task.", {"min_len": 26}),
]:
    got = _run_family(fn, text, **kw)
    check(f"refuses: {name}", not [g for g in got if g[1]], got[0][0] if got else "")
_, gbl, gss = _profile_for("squeue is a command.")
check("refuses: spelling fix that changes the first letter", not [m for m in fam_grammar(gss[0], Ctx(_profile_for('squeue is a command.')[0], 'academic', 'balanced', 'txt', lt=[{"start": 0, "end": 6, "rule": "MORFOLOGIK_RULE_EN_US", "category": "TYPOS", "message": "", "replacements": ["Queue"]}]))])

# 7. humanizer families (section 11): positive cases
hcases = [
    ("cliche: stock verb", fam_cliche, "The framework leverages pretrained features to detect labelling errors.", "uses pretrained", {}),
    ("cliche: role phrase", fam_cliche, "Data quality plays a crucial role in machine learning pipelines.", "is crucial for machine", {}),
    ("cliche: verb phrase", fam_cliche, "The system delves into the relationships between labels and errors.", "examines the relationships", {}),
    ("cliche: stock phrase", fam_cliche, "The method draws on a myriad of signals from the training data.", "on many signals", {}),
    ("hedge_stack", fam_hedge_stack, "The method may potentially reduce the cost of annotation.", "may reduce", {}),
    ("register: academic expands", fam_register, "We don't rely on any external service during training.", "do not rely", {}),
    ("register: conversational contracts", fam_register, "We do not rely on any external service during training.", "don't rely", {"mode": "conversational"}),
    ("discourse: planned drop", fam_discourse, "The model is small. Moreover, it trains quickly on one GPU.", "It trains quickly", {"hplan": _plan(thin={(0, 1)})}),
    ("opener: fronted adverbial", fam_opener, "The model achieves strong results. The model was evaluated after the training had converged.",
     "After the training had converged, the model was evaluated.", {"hplan": _plan(opener={(0, 1)})}),
    ("dash: pair to parentheses", fam_dash, "Our approach — a lightweight filter — runs before training starts.", "(a lightweight filter) runs", {"hplan": _plan(dash=True)}),
]
for name, fn, text, expect, kw in hcases:
    got = _run_family(fn, text, **kw)
    check(f"humanizer {name} proposes and passes the gate", [g for g in got if expect in g[0] and g[1]], got)

# 8. humanizer refusals
for name, fn, text, kw in [
    ("opener: not past a negation", fam_opener, "The model works well. The model did not stop when the loss plateaued.", {"hplan": _plan(opener={(0, 1)})}),
    ("opener: not out of an embedded clause", fam_opener, "The model works well. The model shows that the error fell after the update.", {"hplan": _plan(opener={(0, 1)})}),
    ("opener: not out of a second clause", fam_opener, "The layout works. The layout uses nine points; confirm that the venue permits it before submission.", {"hplan": _plan(opener={(0, 1)})}),
    ("opener: only where the plan asks", fam_opener, "The model works well. The model was evaluated after the training had converged.", {"hplan": _plan()}),
    ("dash: a numeric range is not a dash", fam_dash, "The values 3 – 5 were used in every run of the study.", {"hplan": _plan(dash=True)}),
    ("cliche: a name is never swapped", fam_cliche, "We built Seamless as a sync tool for the lab. The tool offers a seamless protocol for all users.", {}),
]:
    got = _run_family(fn, text, **kw)
    check(f"humanizer refuses: {name}", not [g for g in got if g[1]], got)
_, _, mss = _profile_for("SLURM does not stop because Ceph is part of it. They are separate layers.")
check("humanizer refuses: merge across a negation", not merge_eligible(mss[0], mss[1]))
_, _, mss = _profile_for("The framework uses many signals. The framework is designed to be robust.")
check("humanizer refuses: merge that repeats the subject", not merge_eligible(mss[0], mss[1]))
_, _, mss = _profile_for("The model is small. It trains in an hour.")
check("humanizer merge: two short sentences are eligible", merge_eligible(mss[0], mss[1]))

# 9. guards
try:
    importlib.import_module("torch")
    check("AI-free guard blocks torch", False)
except ImportError as e:
    check("AI-free guard blocks torch", "AI-free guard" in str(e), e)
try:
    _lemminflect.getLemma("running", upos="VERB")
    check("AI-free guard blocks LemmInflect's learned OOV model", False)
except RuntimeError as e:
    check("AI-free guard blocks LemmInflect's learned OOV model", "AI-free guard" in str(e))

# 10. end to end: a machine-style passage through the whole engine, twice
SAMPLE = """Introduction

In today's fast-paced world, data quality plays a crucial role in machine learning. The framework leverages a myriad of signals to detect errors. The framework is designed for large corpora. Moreover, it showcases a seamless integration with existing pipelines. Furthermore, it can potentially reduce annotation costs. Additionally, the system delves into the intricate relationships between labels.

The model achieves strong results. It uses a small encoder. The model runs on a single GPU. The model was evaluated after the training had converged. Our approach — a lightweight filter — runs before training. We don't rely on any external service.
"""
with _tf.TemporaryDirectory() as td:
    p = Path(td) / "sample.txt"; p.write_text(SAMPLE, encoding="utf-8")
    runs = [process(p, "academic", "balanced", write=False) for _ in range(2)]
_, recs_e2e, _, rep_e2e = runs[0]
check("end-to-end: integrity holds", rep_e2e["integrity"]["ok"], rep_e2e["integrity"])
check("end-to-end: identical output on a second run", runs[0][3]["new_masked"] == runs[1][3]["new_masked"])
fams_applied = Counter(r.family for r in recs_e2e if r.status == "applied")
for fam in ("cliche", "hedge_stack", "register", "discourse", "opener", "dash", "merge"):
    check(f"end-to-end: {fam} applied at least once", fams_applied[fam] >= 1, dict(fams_applied))
sa = rep_e2e["style_after"]
check("end-to-end: stock phrasing removed", sa["stock_per_1k"] == 0, sa)
check("end-to-end: additive openers reduced", sa["additive_per_100s"] < rep_e2e["style_before"]["additive_per_100s"], sa)
check("end-to-end: reconstruction does not increase opener repetition",
      sa["opener_repeat_rate"] <= rep_e2e["style_before"]["opener_repeat_rate"], sa)

# 11. PDF reader: every word comes from the PDF (no clipped or invented words), no page furniture
_dehyph = lambda w, raw: any(w[:i] + "-" + w[i:] in raw for i in range(2, len(w) - 1))
for f in [p for p in CORPUS_FILES if p.suffix.lower() == ".pdf"]:
    with _tf.TemporaryDirectory() as td:
        md = pdf_to_markdown(f, Path(td) / (f.stem + ".md")).read_text(encoding="utf-8")
    with pdfplumber.open(f) as _pp:
        _raw = re.sub(r"\s+", "", "".join(c["text"] for pg in _pp.pages for c in pg.chars).translate(_PDF_CHARS))
    unknown = sorted(w for w in set(re.findall(r"[A-Za-z]{4,}", md)) if w not in _raw and not _dehyph(w, _raw))
    check(f"pdf reader: every word comes from the PDF ({f.name})", not unknown, unknown[:6])
    check(f"pdf reader: no page numbers or running headers ({f.name})", not re.search(r"^(Page \d+ of \d+|\d{1,3})$", md, re.M))
    if f.name == "llmqqm9.pdf":                                  # IEEE two-column paper with a full-width title
        check("pdf reader: two-line title becomes one heading", md.startswith("# A Quantum-Inspired Classical Bottleneck for LLM-Based Molecular Property Prediction on QM9"))
        check("pdf reader: small-caps headings stay whole", "## V. RESULTS AND DISCUSSION" in md and "## II. RELATED WORK" in md)
        check("pdf reader: italic subsection headings found", "### E. Statistical robustness of the ablation" in md)
        check("pdf reader: left column is read before the right column",
              0 <= md.find("otherwise identical classical baseline.") < md.find("Our contributions are"))
        check("pdf reader: prose that starts with 'Figure 1' stays editable", re.search(r"^Figure 1 summarizes", md, re.M))

# 12. exports: round trip (export a PDF, read it back) and determinism
_els = [("title", "Round Trip Test Document"), ("h2", "Methods and Materials"),
        ("p", "The method was evaluated on three datasets and the mean error was reported across five seeds for every configuration."),
        ("li", "First item of the list with enough words to wrap nicely."), ("li", "Second item of the list."),
        ("table", [["Tier", "CPU", "RAM"], ["Minimum", "2 cores", "4 GB"], ["Full", "8+ cores", "16 GB"]]),
        ("p", "A closing paragraph follows the table and ends the document cleanly.")]
_pdf1, _pdf2 = elements_to_pdf(_els, "t"), elements_to_pdf(_els, "t")
check("export: the same document gives byte-identical PDFs", _pdf1 == _pdf2)
with _tf.TemporaryDirectory() as td:
    (Path(td) / "rt.pdf").write_bytes(_pdf1)
    _rt = pdf_to_markdown(Path(td) / "rt.pdf", Path(td) / "rt.md").read_text(encoding="utf-8")
for name, want in [("title", "# Round Trip Test Document"), ("heading", "## Methods and Materials"),
                   ("bullets", "- First item of the list with enough words to wrap nicely."),
                   ("table", "| Minimum | 2 cores | 4 GB |"),
                   ("paragraph", "the mean error was reported across five seeds for every configuration.")]:
    check(f"export round trip keeps the {name}", want in _rt, _rt[:300])
_txt = elements_to_text(_els); _mdx = elements_to_markdown(_els)
check("export: text has no Markdown or placeholder residue", not re.search(r"[`⟦|]|^#", _txt, re.M), _txt[:120])
check("export: markdown keeps the table", "| Full | 8+ cores | 16 GB |" in _mdx)

# 13. freeze words and acronym-safe case changes
with _tf.TemporaryDirectory() as td:
    p = Path(td) / "fz.txt"; p.write_text("The framework leverages a myriad of signals to detect errors in the data.\n", encoding="utf-8")
    d0, _, _, r0 = process(p, "academic", "balanced", write=False)
    _free = render_document(d0, r0["new_masked"])[0]["main"]
    d1, _, _, r1 = process(p, "academic", "balanced", write=False, freeze=["leverage"])
    _frz = render_document(d1, r1["new_masked"])[0]["main"]
check("freeze: without freezing, the stock verb is rewritten", "uses" in _free and "leverages" not in _free, _free)
check("freeze: a frozen word comes out unchanged", "leverages" in _frz and "many signals" in _frz, _frz)
check("case: an acronym moved mid-sentence keeps its capitals", _decap("LLM-based results", Tok("LLM-based", 0, 9, "JJ", "llm-based")) == "LLM-based results")
check("case: an ordinary word moved mid-sentence is lower-cased", _decap("The model", Tok("The", 0, 3, "DT", "the")) == "the model")

# 14. tense is kept when the tagger mislabels a verb; short sentences can lose a connective and a hedge
with _tf.TemporaryDirectory() as td:
    p = Path(td) / "t.txt"
    p.write_text("Moreover, existing tools showcase a seamless integration with annotation platforms.\n", encoding="utf-8")
    d0, _, _, r0 = process(p, "academic", "balanced", write=False)
    _o1 = render_document(d0, r0["new_masked"])[0]["main"]
    p.write_text("The system uses many signals to find errors. Moreover, it may potentially help.\n", encoding="utf-8")
    d0, _, _, r0 = process(p, "academic", "balanced", write=False)
    _o2 = render_document(d0, r0["new_masked"])[0]["main"]
check("tense: a stock verb keeps its present tense ('showcase' -> 'show', never 'showed')", "tools show a smooth" in _o1, _o1)
check("short sentence: connective and doubled hedge both go", "It may help." in _o2 and "Moreover" not in _o2, _o2)
check("surface_tag: spelling decides the tense", surface_tag("showcase", "showcase", "VBD") == "VBP" and surface_tag("show", "showed", "VBD") == "VBD")

TESTS = pd.DataFrame(RESULTS)
print(f"{TESTS['pass'].sum()}/{len(TESTS)} tests passed")
assert TESTS["pass"].all(), TESTS[~TESTS["pass"]].to_string()
TESTS


2026-11-23 21:09:57,394 : INFO : maxp pruned


2026-11-23 21:09:57,395 : INFO : cmap pruned


2026-11-23 21:09:57,396 : INFO : kern dropped


2026-11-23 21:09:57,396 : INFO : post pruned


2026-11-23 21:09:57,397 : INFO : FFTM dropped


2026-11-23 21:09:57,397 : INFO : MATH dropped


... (512 outputs left out)

2026-11-23 21:09:57,798 : INFO : glyf pruned


2026-11-23 21:09:57,801 : INFO : name pruned


104/104 tests passed


,test,pass,detail
0,round-trip unchanged: CANTOR.pdf,True,
1,unmask restores every block: CANTOR.pdf,True,
2,round-trip unchanged: CONVERSION.md,True,
3,unmask restores every block: CONVERSION.md,True,
4,round-trip unchanged: HPC.docx,True,
...,...,...,...
99,case: an acronym moved mid-sentence keeps its ...,True,
100,case: an ordinary word moved mid-sentence is l...,True,
101,tense: a stock verb keeps its present tense ('...,True,Existing tools show a smooth integration with ...
102,short sentence: connective and doubled hedge b...,True,The system uses many signals to find errors. I...


## 18 · Benchmark
Quality before/after (LanguageTool grammar matches, proselint alerts, Flesch–Kincaid, sentence-length mean and spread, nominalisation and passive density, MTLD/HD-D). Style markers before/after (section 9). Preservation (integrity, chrF to the input). Speed per stage, memory (peak RSS and Python peak). Determinism: 3 identical runs in one process, plus fresh processes with different `PYTHONHASHSEED`s, which catches any hidden dependence on hash order. The parse cache makes repeat runs faster; first-run timings are in section 16's output.

In [18]:
#@title 18 · Benchmark: quality, style, preservation, speed, memory, determinism (all computed here)
import sacrebleu
import pandas as pd
from lexicalrichness import LexicalRichness


def _metrics(masked_blocks):
    text = readable_text("\n\n".join(masked_blocks))
    words = re.findall(r"[A-Za-z]+(?:[-'][A-Za-z]+)*", text)
    n = max(1, len(words))
    sents = [s for b in masked_blocks for s in split_sentences(Block(0, "", 0, 0, "", b, {}))]
    lens = [sum(1 for t in s.toks if t.text[:1].isalnum() or PH.fullmatch(t.text)) for s in sents]
    toks = [t for s in sents for t in s.toks]
    nominal = sum(1 for t in toks if t.tag.startswith("NN") and re.search(r"(tion|sion|ment|ance|ence)s?$", t.text.lower())
                  and _derived_verb(t.lem))
    passive = sum(1 for s in sents for i in range(len(s.toks) - 1) if s.toks[i].lem == "be" and s.toks[i + 1].tag == "VBN")
    # grammar-type matches only: spelling (technical vocabulary) and typography/whitespace rules
    # mostly flag markup and terms of art, which would swamp the signal
    lt_n = sum(1 for b in masked_blocks for m in lt().check(readable_text(b))
               if "MORFOLOGIK" not in m.rule_id and m.rule_id not in _GRAMMAR_SKIP
               and str(getattr(m, "category", "")).upper() not in ("TYPOGRAPHY", "CASING", "STYLE", "REDUNDANCY", "PLAIN_ENGLISH"))
    lr = LexicalRichness(" ".join(words)) if len(words) > 60 else None
    return {"words": len(words), "sentences": len(sents),
            "lt_per_1k": round(1000 * lt_n / n, 2), "proselint_per_1k": round(1000 * proselint_count(text) / n, 2),
            "fk_grade": textstat.flesch_kincaid_grade(text),
            "sent_len_mean": round(statistics.mean(lens), 2) if lens else 0,
            "sent_len_sd": round(statistics.pstdev(lens), 2) if lens else 0,
            "nominalisations_per_100w": round(100 * nominal / n, 2),
            "passives_per_100_sent": round(100 * passive / max(1, len(sents)), 2),
            "mtld": round(lr.mtld(), 2) if lr else None, "hdd": round(lr.hdd(draws=42), 4) if lr else None}


def benchmark(runs, determinism_docs=2, repeats=3):
    import pandas as pd
    rows = []
    for f, v in runs.items():
        doc, rep = v["doc"], v["report"]
        before = [b.masked for b in doc.blocks if b.section not in SKIP_SECTIONS]
        after = [rep["new_masked"].get(b.bid, b.masked) for b in doc.blocks if b.section not in SKIP_SECTIONS]
        mb, ma = _metrics(before), _metrics(after)
        chrf = sacrebleu.corpus_chrf([readable_text(a) for a in after], [[readable_text(b) for b in before]]).score
        st = Counter(r.status for r in v["records"])
        row = {"document": f.name, "format": doc.fmt, "words": mb["words"],
               "edits_applied": st["applied"], "edits_blocked_by_gate": st["rejected-gate"],
               "integrity": "OK" if rep["integrity"]["ok"] else "FAIL",
               "seconds": round(v["seconds"], 1), "sec_per_1k_words": round(1000 * v["seconds"] / max(1, mb["words"]), 2),
               "peak_rss_mb": round(v["peak_rss_mb"]), "python_peak_mb": round(v["py_peak_mb"]),
               "chrF_vs_input": round(chrf, 1)}
        for k in ("lt_per_1k", "proselint_per_1k", "fk_grade", "sent_len_mean", "sent_len_sd",
                  "nominalisations_per_100w", "passives_per_100_sent", "mtld", "hdd", "words"):
            row[f"{k}_before"], row[f"{k}_after"] = mb[k], ma[k]
        sb, sa = rep["style_before"], rep["style_after"]           # humanizer style markers (section 9)
        for k in ("sent_len_cv", "para_len_cv_median", "additive_per_100s", "opener_repeat_rate",
                  "stock_per_1k", "hedge_stacks_per_1k", "dash_per_1k", "contractions_per_1k"):
            row[f"style_{k}_before"], row[f"style_{k}_after"] = sb[k], sa[k]
        row["style_markers_outside_target_before"] = sum(not v[2] for v in rep["style_check"]["before"].values())
        row["style_markers_outside_target_after"] = sum(not v[2] for v in rep["style_check"]["after"].values())
        rows.append(row)
    df = pd.DataFrame(rows)
    # determinism: identical output bytes across repeated runs
    det = []
    for f in list(runs)[:determinism_docs]:
        hashes = set()
        for _ in range(repeats):
            _, _, outp, _ = process(f, MODE, INTENSITY)
            outp = Path(outp)
            if outp.suffix == ".docx":
                import zipfile
                with zipfile.ZipFile(outp) as z:
                    data = z.read("word/document.xml")
            elif outp.suffix == ".tex":
                data = b"".join(p.read_bytes() for p in sorted(outp.parent.rglob("*.tex")))
            else:
                data = outp.read_bytes()
            hashes.add(hashlib.sha256(data).hexdigest())
        det.append({"document": f.name, "check": "same process", "runs": repeats, "distinct_outputs": len(hashes)})
    return df, pd.DataFrame(det)


def determinism_across_hash_seeds(doc_path, seeds=(1, 2), notebook=BASE / "DNWE_2.ipynb"):
    """Python randomises string hashing in every new process, so code that iterates a set of strings
    could produce different output in a different process even though repeated runs in ONE process
    agree. This runs the engine cells of this notebook (as saved on disk) in fresh processes with
    different PYTHONHASHSEEDs and compares the edited text byte for byte."""
    import nbformat
    nb = nbformat.read(notebook, as_version=4)
    code = []
    for c in nb.cells:
        if c.cell_type == "code":
            code.append(c.source)
            if c.source.startswith("#@title 13 "):
                break
    probe = (f"\n_d, _r, _o, _rep = process(Path({str(doc_path)!r}), {MODE!r}, {INTENSITY!r}, write=False)\n"
             "import hashlib as _h, json as _j\n"
             "print('DNWE_HASH', _h.sha256(_j.dumps({str(k): v for k, v in sorted(_rep['new_masked'].items())}, sort_keys=True).encode()).hexdigest())\n"
             "if _LT is not None: _LT.close()\n")
    hashes = {}
    with tempfile.TemporaryDirectory() as td:
        script = Path(td) / "engine_probe.py"
        script.write_text("\n\n".join(code) + probe, encoding="utf-8")
        for seed in seeds:
            env = {**os.environ, "PYTHONHASHSEED": str(seed)}
            res = subprocess.run([sys.executable, str(script)], cwd=BASE, env=env, capture_output=True, text=True, timeout=1800)
            m = re.search(r"DNWE_HASH (\w+)", res.stdout)
            assert m, f"engine probe failed (seed {seed}):\n{res.stderr[-1500:]}"
            hashes[seed] = m.group(1)
    return {"document": Path(doc_path).name, "check": f"fresh processes, PYTHONHASHSEED {'/'.join(map(str, seeds))}",
            "runs": len(seeds), "distinct_outputs": len(set(hashes.values()))}


BENCH, DETERMINISM = benchmark(RUNS)
_smallest = min(RUNS, key=lambda f: RUNS[f]["report"]["profile"].words)
DETERMINISM = pd.concat([DETERMINISM, pd.DataFrame([determinism_across_hash_seeds(_smallest)])], ignore_index=True)
BENCH.to_csv(OUT / "benchmark.csv", index=False)
DETERMINISM.to_csv(OUT / "determinism.csv", index=False)
stage = pd.DataFrame([{**{"document": f.name}, **{k: round(t, 2) for k, t in v["timings"].items()}} for f, v in RUNS.items()])
stage.to_csv(OUT / "stage_timings.csv", index=False)
assert (BENCH.integrity == "OK").all(), "integrity failure"
assert (DETERMINISM.distinct_outputs == 1).all(), "non-deterministic output"
print("Benchmark written to out/benchmark.csv, out/stage_timings.csv, out/determinism.csv")
BENCH

Benchmark written to out/benchmark.csv, out/stage_timings.csv, out/determinism.csv


,document,format,words,edits_applied,edits_blocked_by_gate,integrity,seconds,sec_per_1k_words,peak_rss_mb,python_peak_mb,...,style_stock_per_1k_before,style_stock_per_1k_after,style_hedge_stacks_per_1k_before,style_hedge_stacks_per_1k_after,style_dash_per_1k_before,style_dash_per_1k_after,style_contractions_per_1k_before,style_contractions_per_1k_after,style_markers_outside_target_before,style_markers_outside_target_after
0,CANTOR.pdf,pdf-derived-markdown,819,2,0,OK,15.8,19.31,583,97,...,1.19,0.00,0.0,0.0,2.38,0.00,0.00,0.00,3,1
1,CONVERSION.md,markdown,307,1,0,OK,0.8,2.49,578,1,...,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00,1,1
2,HPC.docx,docx,2206,2,0,OK,10.9,4.95,579,5,...,0.00,0.00,0.0,0.0,0.00,0.00,6.04,6.04,0,0
3,llmqqm9.pdf,pdf-derived-markdown,3590,3,4,OK,14.0,3.91,609,52,...,0.28,0.28,0.0,0.0,5.29,5.29,2.23,2.23,1,1
4,sapienzer.txt,txt,835,6,0,OK,2.1,2.50,620,2,...,0.00,0.00,0.0,0.0,7.25,7.18,10.14,1.44,1,1
5,main_bhortijuddho.tex,latex,1925,1,0,OK,4.8,2.50,625,3,...,0.00,0.00,0.0,0.0,0.00,0.00,1.02,1.02,0,0
6,main_celeba.tex,latex,2428,4,0,OK,6.6,2.72,616,4,...,1.64,0.82,0.0,0.0,1.64,1.64,1.23,1.23,2,1
7,main_morphena.tex,latex,3222,12,2,OK,7.5,2.33,616,5,...,0.92,0.00,0.0,0.0,0.31,0.31,3.08,3.08,0,0


## 19 · Label the dev set (manual step)
Run this cell, then open `out/dev_set.csv` and `out/test_set.csv` and fill the `label` column for each row: **correct**, **wrong**, or **meaning** (the edit changed what you meant). Earlier labels are carried over automatically. About 20 per family is enough to start; a family needs 15 dev labels before its measured precision replaces the prior.

In [19]:
DEV = export_dev_set(ALL_RECORDS)
DEV.groupby(['family', 'split']).size().unstack(fill_value=0)

Wrote 20 dev and 13 test items (8 families) to dev_set.csv / test_set.csv; 15 already labelled, 18 need a label. Archive: 29 labels in labels_archive.csv.


split,dev,test
family,,
cliche,2,4
dash,1,0
light_verb,0,2
merge,0,1
plain,7,4
register,4,2
split,3,0
wordy,3,0


## 20 · Load your labels
Turns your labels into per-family precision. Re-run section 16 afterwards: confidences switch from *prior* to *measured*, and families below 0.95 are disabled.

In [20]:
import pandas as pd
if DEV_CSV.exists() and (pd.read_csv(DEV_CSV, dtype=str).fillna("")["label"] != "").any():
    PRECISION = load_labels()
else:
    print("No labels yet: fill the 'label' column in out/dev_set.csv and out/test_set.csv, then re-run this cell and section 14.")

plain           dev 5/6 -> 0.750  test 1.0  too few labels: keeps prior
split           dev 2/2 -> 0.750  test None  too few labels: keeps prior
wordy           dev 2/3 -> 0.600  test None  too few labels: keeps prior
still unlabelled: cliche 6, dash 1, merge 1, plain 3, register 6, split 1


## 21 · Edit one paper
Point `MY_PAPER` at a file (PDF, Word, Markdown, text, or the main `.tex` of a LaTeX folder), and pick `MODE` / `INTENSITY` in section 1. `FREEZE` lists words or phrases that must come out exactly as they went in. The humanized document is written in every format: its own plus `.txt`, `.md` and `.pdf`. To undo individual edits: set `decision` to `reject` in the CSV, then run the last line.

In [21]:
MY_PAPER = None          # e.g. CORPUS / "my_paper.pdf"  or  CORPUS / "my_paper.docx"  or  CORPUS / "my_latex_project" / "main.tex"
FREEZE = []              # words/phrases that must not change, e.g. ["leverage", "graph neural network"]
if MY_PAPER:
    doc, recs, out_path, rep = process(MY_PAPER, MODE, INTENSITY, freeze=FREEZE)
    csv_path, html_path = review_files(doc, recs, report=rep)
    for ext, data in export_all(doc, render_document(doc, rep["new_masked"])[0]).items():
        dst = OUT / f"{Path(MY_PAPER).stem}.dnwe.{ext}"; dst.write_bytes(data); print("humanized:", dst)
    print("review:", html_path); print("decisions:", csv_path)
    # after editing decisions in the CSV:
    # print(apply_decisions(MY_PAPER, recs, csv_path))

## 22 · Shut down LanguageTool

In [22]:
if _LT is not None:
    _LT.close(); _LT = None
print('done')

2026-11-23 21:11:16,440 : INFO : Terminating LanguageTool server on port 8619


done


## 23 · Local web app (optional)
The same engine also runs behind a small local web page, outside this notebook:

```bash
cd /home/saifarfi/Volumetric/Sapienaizerv2
.venv/bin/python server.py          # opens http://127.0.0.1:8765
```

`server.py` executes sections 1–13 of **this notebook** at start-up, so the notebook stays the only copy of the engine; restart the server after you change a cell. `web/index.html` lets you paste text or drop a document (PDF, Word, Markdown, text, LaTeX), pick the mode and intensity, freeze words that must not change, and see the rewritten text, a word-level diff, every edit with its reason, and the style profile. Every result downloads as `.pdf`, `.md` and `.txt` (plus `.docx` / `.tex` for those inputs). The server listens on 127.0.0.1 only, refuses other hosts and web origins, and runs with the offline and AI-free guards of section 3.